1. 共用參數設定


In [ ]:
'''
Measure the elapsed time of every code cell, including cells that raise errors.
記錄每個 Code Cell 的執行時間，即使發生錯誤也顯示已耗費的時間。
'''
from time import perf_counter as _cell_perf_counter
from IPython import get_ipython as _cell_get_ipython

_CELL_TIMER_SHELL = _cell_get_ipython()
if _CELL_TIMER_SHELL is None:
    raise RuntimeError('請在 Jupyter Notebook 中執行本教材。')
_CELL_TIMER_STATE = {'started': _cell_perf_counter()}

def _cell_timer_begin(_info):
    _CELL_TIMER_STATE['started'] = _cell_perf_counter()

def _cell_timer_end(result):
    started = _CELL_TIMER_STATE.pop('started', None)
    if started is None:
        return
    minutes, seconds = divmod(_cell_perf_counter() - started, 60)
    elapsed = f'{int(minutes)} 分 {seconds:05.2f} 秒' if minutes else f'{seconds:.2f} 秒'
    status = '執行失敗' if result.error_before_exec or result.error_in_exec else '執行完成'
    print(f'⏱ 本格{status}，耗時：{elapsed}')

'''
Replace existing timers when the first cell is executed again.
重新執行第一格時，先取消舊計時器，避免重複顯示耗時。
'''
for _event_name, _callback in globals().get('_CELL_TIMER_CALLBACKS', ()):
    try:
        _CELL_TIMER_SHELL.events.unregister(_event_name, _callback)
    except ValueError:
        pass
_CELL_TIMER_CALLBACKS = (
    ('pre_run_cell', _cell_timer_begin),
    ('post_run_cell', _cell_timer_end),
)
for _event_name, _callback in _CELL_TIMER_CALLBACKS:
    _CELL_TIMER_SHELL.events.register(_event_name, _callback)

'''
Import standard-library tools for editable paths and local service settings.
匯入設定路徑與本機服務參數所需的 Python 內建工具。
'''
import os
from pathlib import Path
ROOT = Path.home() / 'ai_models'

'''
Select the GPU IDs available to this notebook.
指定這份 Notebook 可使用的 GPU ID。
'''
GPU_IDS = [0]


'''
Select the GPUs for the generation model.
指定生成模型使用的 GPU ID。
'''
MODEL_GPU_IDS = [0]


'''
Select the GPU for text encoders, VAE, and codec models.
指定文字編碼器、VAE 與 Codec 等輔助模型使用的 GPU ID。
'''
AUX_GPU_ID = 0


'''
Derive the primary model GPU for existing loaders; no manual edit needed.
依上述設定取得 Loader 使用的主要生成 GPU，不需要手動修改。
'''
MODEL_GPU_ID = MODEL_GPU_IDS[0]


'''
Count the selected GPUs for the existing initialization flow.
計算指定的 GPU 數量，供原有初始化流程使用。
'''
GPU_COUNT = len(GPU_IDS)

'''
Reject duplicate, out-of-range, or inconsistent GPU assignments.
拒絕重複、超出設定範圍或互相衝突的 GPU 編號。
'''
assert GPU_IDS and len(set(GPU_IDS)) == len(GPU_IDS) and all(type(i) is int and i >= 0 for i in GPU_IDS)
assert MODEL_GPU_IDS and len(set(MODEL_GPU_IDS)) == len(MODEL_GPU_IDS)
assert set(MODEL_GPU_IDS).issubset(GPU_IDS) and AUX_GPU_ID in GPU_IDS
ROOT.mkdir(parents=True, exist_ok=True)

'''
Validate the selected IDs against the ROCm/CUDA-visible PyTorch devices.
比對 ROCm／CUDA 版 PyTorch 可見的 GPU 編號，避免錯誤分配。
'''
def validate_torch_gpus(sharding=False):
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('找不到 PyTorch GPU。請先在 Ubuntu 安裝符合 GPU 型號的 ROCm PyTorch。')
    available = torch.cuda.device_count()
    if max(GPU_IDS) >= available:
        raise RuntimeError(f'GPU_IDS={GPU_IDS}，但 PyTorch 只看得到 {available} 張 GPU。')
    if len(MODEL_GPU_IDS) > 1 and not sharding:
        raise RuntimeError('這套 Loader 尚未驗證跨 GPU 主模型切分；MODEL_GPU_IDS 請先只指定一張。')
    backend = 'AMD ROCm' if torch.version.hip else 'NVIDIA CUDA'
    print(f'{backend}：選擇 {GPU_IDS}；生成 {MODEL_GPU_IDS}；輔助 cuda:{AUX_GPU_ID}')
    for gpu_id in GPU_IDS:
        memory = torch.cuda.get_device_properties(gpu_id).total_memory / 1024**3
        print(f'GPU {gpu_id}: {torch.cuda.get_device_name(gpu_id)} ({memory:.1f} GiB)')


        '''
        Warn if the selected device has less memory than the 32GB profile expects.
        若選用的 GPU 容量低於 32GB 配置需求，提醒可能需要更積極的 Offload。
        '''
        if memory < 27:
            '''
            Explain the available-VRAM limitation without blocking execution.
            提醒可用 VRAM 較少，但不阻擋使用者繼續執行。
            '''
            print('提醒：此 Notebook 預設供約 32GB VRAM 的 GPU 使用；目前容量可能需要更積極的 Offload。')


COMFY = ROOT / 'ComfyUI-HyperFlowH3'

'''
Store large downloaded GGUF weights in scratch storage.
設定 GGUF 模型權重的暫存磁碟位置，減少工作空間占用。
'''
STORE = ROOT / 'hyperflow_h3_models'
PORT = 8194  # 本機可用 Port
COMFY_REF = 'v0.45.0'

'''
Set the Hugging Face cache directory before any model downloads.
在下載模型之前指定 Hugging Face 快取目錄，避免大型檔案占滿工作空間。
'''
HF_CACHE_DIR = ROOT / 'hyperflow_h3_hf'
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE_DIR / 'hub')
os.environ['HF_XET_CHUNK_CACHE_SIZE_BYTES'] = '0'

'''
Select the HyperFlow GGUF quantization. Q8 is memory-intensive on smaller VRAM cards.
選擇 HyperFlow 的 GGUF 精度；Q8 需要較多 VRAM。
'''
QUANT = 'Q8_0'  # 'Q3_K_M', 'Q4_K_S', 'Q4_K_M', 'Q5_K_M', 'Q6_K', 'Q8_0'
DEFAULT_QUANT = QUANT

'''
Set the primary model GPU memory budget; leave headroom for runtime tensors.
設定生成 GPU 的權重預算（GiB），保留中間張量與執行時的空間。
'''
GPU0_MODEL_GIB = 24.0  # 單張 32GB 卡的主要 VRAM 預算（GiB），其餘供暫存／Offload
GPU1_MODEL_GIB = 24.0  # 輔助模型監控參考值；同一張卡不會額外增加 24 GiB

'''
Select the text-encoder weight placement strategy.
設定文字編碼器權重管理：dynamic 分批調度較節省 GPU 記憶體；streaming 使用時整體載入；resident 盡量常駐 GPU 記憶體。
'''
TEXT_MODE = 'dynamic'  # 'dynamic', 'streaming', 'resident'
TEXT_GPU = f'cuda:{AUX_GPU_ID}'  # 輔助 GPU，可與生成 GPU 相同
NODE_CACHE_MODE = 'classic'  # 'classic', 'none'

'''
Choose the HyperFlow text encoder; select a format actually supported by the installed loader.
選擇 HyperFlow 文字編碼器；需搭配實際支援的權重與 Loader。
'''
ENCODER_VARIANT = '4B_FP8'  # 可選 4B_FP8、4B_BF16、8B_NVFP4、8B_W4A8、8B_INT8、8B_FP8、8B_BF16、32B_NVFP4、32B_INT8、32B_BF16
DEFAULT_ENCODER_VARIANT = ENCODER_VARIANT
WEIGHTS_IN = 'disk'  # 'disk', 'auto', 'ram'

'''
Select the local directory used by R2V image uploads.
指定 R2V 參考圖片的儲存目錄；圖片檔案由該 Task Cell 管理。
'''
R2V_REF_DIR = ROOT / 'hyperflow_h3_r2v_references'
REMOTE_PORT = 8765  # 例如 8765、8766，範圍 1024～65535
REMOTE_SERVICE_ID = '06_HyperFlow_H3_Flash_影片生成_Ubuntu'

'''
Place optional API job data inside the configured workspace.
將遠端工作與暫存資料存放在第一格設定的工作空間底下。
'''
REMOTE_ROOT = ROOT / 'remote_api_jobs' / REMOTE_SERVICE_ID
REMOTE_QUEUE_CAPACITY = 32  # 建議 12～32，依課堂人數調整
REMOTE_UPLOAD_CONCURRENCY = 2  # 建議 1～2

'''
Choose output retention and storage safety limits.
設定遠端成果保留期限、空間限制及主機磁碟最低剩餘量。
'''
REMOTE_RESULT_TTL_HOURS = 24
REMOTE_RESULT_LIMIT_GIB = 8
REMOTE_MIN_FREE_GIB = 2
REMOTE_STALL_HOURS = 2
GGUF_REPO = 'VideoRebirthAI/hyperflow-h3-flash'

'''
Validate user-editable local ports and remote queue limits.
確認教師設定的本機 Port 與遠端佇列限制有效，避免服務啟動後才失敗。
'''
assert 1024 <= REMOTE_PORT <= 65535, 'REMOTE_PORT 必須介於 1024～65535。'
assert 1 <= REMOTE_QUEUE_CAPACITY <= 1024, 'FIFO 佇列容量設定無效。'
assert 1 <= REMOTE_UPLOAD_CONCURRENCY <= 8, '上傳並發限制設定無效。'
assert REMOTE_RESULT_TTL_HOURS > 0 and REMOTE_RESULT_LIMIT_GIB > 0 and REMOTE_MIN_FREE_GIB >= 0
assert REMOTE_STALL_HOURS > 0
assert 1024 <= PORT <= 65535 and PORT != REMOTE_PORT
print('✓ 第 1 節完成：共用參數已設定，請繼續執行環境安裝。')


2. 安裝 HyperFlow H3 Flash 環境


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import os
import subprocess
import sys
from pathlib import Path
import torch
validate_torch_gpus()
for i in GPU_IDS:
    print(f'GPU {i}: {torch.cuda.get_device_name(i)} / {torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GiB')

'''
Install the official HyperFlow loader and the 8B/4B projection runtime.
安裝官方 HyperFlow Loader 與 8B／4B 投影節點。
'''
repositories = {
    COMFY: 'https://github.com/Comfy-Org/ComfyUI.git',
    COMFY / 'custom_nodes/ComfyUI-GGUF': 'https://github.com/city96/ComfyUI-GGUF.git',
    COMFY / 'custom_nodes/ComfyUI-HyperFlow-Flash': 'https://github.com/Video-Rebirth/ComfyUI-HyperFlow-Flash.git',
    COMFY / 'custom_nodes/ComfyUI-ClipProj': 'https://github.com/nicolab28/ComfyUI-ClipProj.git',
}
H3_RESTART_REQUIRED = False
for directory, url in repositories.items():
    assert not directory.exists() or (directory / '.git').is_dir(), f'不是 Git 倉庫：{directory}'
    if (directory / '.git').is_dir():
        dirty = subprocess.check_output(['git', '-C', str(directory), 'status', '--porcelain'], text=True).strip()
        if not dirty and directory != COMFY:
            before = subprocess.check_output(['git', '-C', str(directory), 'rev-parse', 'HEAD'], text=True).strip()
            subprocess.run(['git', '-C', str(directory), 'pull', '--ff-only', '-q'], check=True)
            H3_RESTART_REQUIRED |= before != subprocess.check_output(['git', '-C', str(directory), 'rev-parse', 'HEAD'], text=True).strip()

        elif dirty:
            '''
            Preserve local changes for manual inspection.
            保留本機修改供使用者自行檢查。
            '''
            print(f'本機修改未覆寫：{directory.name}')

    else:
        directory.parent.mkdir(parents=True, exist_ok=True)

        '''
        Pin only the core ComfyUI checkout to a release tag.
        只有 ComfyUI 核心固定於正式版標籤。
        '''
        options = ['--branch', COMFY_REF] if directory == COMFY else []
        subprocess.run(['git', 'clone', '--depth', '1', *options, url, str(directory)], check=True)
        H3_RESTART_REQUIRED = True

'''
Fetch the selected release on every run.
每次執行都確認指定的 ComfyUI 正式版。
'''
subprocess.run(['git', '-C', str(COMFY), 'fetch', '--depth', '1', 'origin', 'tag', COMFY_REF], check=True)
previous_core = subprocess.check_output(['git', '-C', str(COMFY), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run(['git', '-C', str(COMFY), 'checkout', '-q', '--detach', COMFY_REF], check=True)

'''
Restart if an existing process is using an older core checkout.
如果執行中的 ComfyUI 核心版本過舊，標記為需要重新啟動。
'''
H3_RESTART_REQUIRED |= previous_core != subprocess.check_output(['git', '-C', str(COMFY), 'rev-parse', 'HEAD'], text=True).strip()
constraint = ROOT / 'hyperflow_torch_constraints.txt'
constraint.write_text(f'torch=={torch.__version__}\n', encoding='utf-8')

'''
Install only the dependencies declared by ComfyUI and HyperFlow.
僅安裝 ComfyUI 與 HyperFlow 官方宣告的相依套件。
'''
requirements = [
    COMFY / 'requirements.txt',
    COMFY / 'custom_nodes/ComfyUI-GGUF/requirements.txt',
    COMFY / 'custom_nodes/ComfyUI-HyperFlow-Flash/requirements.txt',
]
for requirement in requirements:
    '''
    Skip an optional file that is not present in this release.
    如果某版沒有相依套件檔案，就安全略過。
    '''
    if requirement.is_file():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint), '-r', str(requirement)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint),
                'huggingface_hub', 'hf_xet', 'requests', 'websocket-client', 'tqdm', 'ipywidgets>=8,<9'], check=True)

'''
Display installed source revisions for reproducibility.
顯示各程式庫的實際版本，方便問題追蹤。
'''
for directory in repositories:
    revision = subprocess.check_output(['git', '-C', str(directory), 'rev-parse', '--short', 'HEAD'], text=True).strip()
    print(f'{directory.name}: {revision}')

print('✓ 第 2 節完成：HyperFlow H3 Flash / ComfyUI v0.45.0')


3. 下載 HyperFlow 模型與輔助權重


In [ ]:
'''
Check disk capacity before downloading gigabyte-scale files.
下載大型權重前先檢查可用磁碟空間。
'''
import shutil
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE_DIR / 'hub')

'''
Disable the optional extra Xet chunk cache.
停用不必要的 Xet 分塊快取以節省空間。
'''
os.environ['HF_XET_CHUNK_CACHE_SIZE_BYTES'] = '0'
from huggingface_hub import hf_hub_download

'''
Choose the generation task before model download; select the same task for generation.
在下載模型前先選擇任務；生成 Cell 會沿用此設定。 Options / 可選值: 't2v' 文字生成、'i2v' 起始圖片、'flf2v' 首尾圖片、'r2v' 多張參考圖片。
'''
TASK = globals().get('TASK', 't2v')  # 't2v', 'i2v', 'l2v', 'flf2v', 'r2v'
assert TASK in ('t2v', 'i2v', 'l2v', 'flf2v', 'r2v'), f'不支援的 TASK：{TASK}'

'''
Select the compressed FL2VA or the separate Ref2VA diffusion architecture.
一般影片使用 FL2VA，多圖參考 R2V 則使用獨立的 Ref2VA。
'''
MODEL_FAMILY = 'ref2va' if TASK == 'r2v' else 'fl2va'
task_models = globals().get('TASK_CONFIGS', {}).get(TASK, {})
ENCODER_SIZE = ENCODER_VARIANT.split('_', 1)[0]

'''
Map published GGUF sizes for the two independent diffusion architectures (decimal GB).
記錄 FL2VA 與 Ref2VA 兩套獨立模型的公開權重容量（十進位 GB）。
'''
MODEL_SIZES_GB = {
    'fl2va': {'Q3_K_M': 7.22, 'Q4_K_S': 8.01, 'Q4_K_M': 9.05,
              'Q5_K_M': 10.3, 'Q6_K': 11.6, 'Q8_0': 14.9},
    'ref2va': {'Q3_K_M': 7.4, 'Q4_K_S': 8.14, 'Q4_K_M': 9.22,
               'Q5_K_M': 10.45, 'Q6_K': 11.76, 'Q8_0': 15.16},
}
QUANT_SIZES_GB = MODEL_SIZES_GB[MODEL_FAMILY]
assert QUANT in QUANT_SIZES_GB, f'GPU 0 不支援 {QUANT}。'

'''
Obtain the specific compressed HyperFlow checkpoint, not the old 31B GGUF.
下載真正的 HyperFlow 壓縮模型，而不是先前 31B 的 GGUF。
'''
DIT_NAME = f'minimax_h3_{MODEL_FAMILY}_hyperflow_flash_14b-{QUANT}.gguf'
ENCODERS = {
    '32B_NVFP4': ('Comfy-Org/MiniMax-H3', 'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', 15.7),
    '32B_INT8': ('Comfy-Org/MiniMax-H3', 'qwen3vl_32b_minimax_h3_int8_convrot.safetensors', 27.1),
    '32B_BF16': ('Comfy-Org/MiniMax-H3', 'qwen3vl_32b_minimax_h3_bf16.safetensors', 51.5),
    '8B_NVFP4': ('Comfy-Org/Qwen3-VL', 'qwen3vl_8b_nvfp4.safetensors', 6.31),
    '8B_W4A8': ('Comfy-Org/Qwen3-VL', 'qwen3vl_8b_w4a8.safetensors', 6.31),
    '8B_INT8': ('Comfy-Org/Qwen3-VL', 'qwen3vl_8b_int8_convrot.safetensors', 9.35),
    '8B_FP8': ('Comfy-Org/Qwen3-VL', 'qwen3vl_8b_fp8_scaled.safetensors', 10.6),
    '8B_BF16': ('Comfy-Org/Qwen3-VL', 'qwen3vl_8b_bf16.safetensors', 17.5),
    '4B_FP8': ('Comfy-Org/Qwen3-VL', 'qwen3vl_4b_fp8_scaled.safetensors', 5.24),
    '4B_BF16': ('Comfy-Org/Qwen3-VL', 'qwen3vl_4b_bf16.safetensors', 8.88),
}

'''
Stop before downloading an unsupported GPU 1 model variant.
若 GPU 1 模型選項不存在，下載前就明確停止。
'''
assert ENCODER_VARIANT in ENCODERS, f'輔助 GPU 不支援 {ENCODER_VARIANT}；可選：{list(ENCODERS)}'
ENCODER_REPO, encoder_name, ENCODER_GB = ENCODERS[ENCODER_VARIANT]
ENCODER_FILE = f'text_encoders/{encoder_name}'

'''
Select the matching v3.1 projection only for smaller encoders.
只有小型 Encoder 需要對應的 v3.1 ClipProj 投影矩陣。
'''
PROJECTION_FILE = f'mmh3-{ENCODER_SIZE.lower()}-ClipProj-v3.1-mlp.safetensors' if ENCODER_SIZE != '32B' else None
TEXT_LOADER_NODE = 'ClipProjDeviceLoader' if ENCODER_SIZE == '32B' else 'ClipProjLoader'
print(f'GPU {MODEL_GPU_ID}：HyperFlow {QUANT}（{QUANT_SIZES_GB[QUANT]} GB）；GPU {AUX_GPU_ID}：{ENCODER_VARIANT}（{ENCODER_GB} GB）；工作目錄可用空間 {shutil.disk_usage(ROOT).free / 1024**3:.1f} GiB')

'''
Download exactly one task-specific GGUF plus its encoder and two VAEs.
只下載所選任務的 FL2VA 或 Ref2VA GGUF，以及 Encoder 與兩個 VAE。
'''
models = [
    (GGUF_REPO, f'{MODEL_FAMILY}_gguf/{DIT_NAME}', 'diffusion_models'),
    (ENCODER_REPO, ENCODER_FILE, 'text_encoders'),
    ('Comfy-Org/MiniMax-H3', 'vae/minimax_h3_video_vae_fp16.safetensors', 'vae'),
    ('Comfy-Org/MiniMax-H3', 'vae/minimax_h3_audio_vae_fp32.safetensors', 'vae'),
]
if PROJECTION_FILE:
    models.append(('NicoLab28/ClipProj-MiniMax-H3', PROJECTION_FILE, 'clip_projections'))

STORE.mkdir(parents=True, exist_ok=True)

'''
Estimate the additional space needed for all selected model weights (decimal GB).
估算本次所選全部模型權重還需要多少暫存空間（十進位 GB）。
'''
model_sizes = [QUANT_SIZES_GB[QUANT], ENCODER_GB, 5.21, 0.605]
if PROJECTION_FILE:
    model_sizes.append(0.577 if ENCODER_SIZE == '8B' else 0.481)

missing_gb = sum(size for (_, filename, folder), size in zip(models, model_sizes)
                 if not (STORE / folder / filename).is_file())

'''
Read free space on the actual filesystem containing the model directory.
取得真正儲存模型權重的檔案系統剩餘空間。
'''
free_gb = shutil.disk_usage(STORE).free / 1e9
assert free_gb >= missing_gb * 1.05, (
    f'模型儲存空間不足：尚需約 {missing_gb:.1f} GB，剩餘 {free_gb:.1f} GB；'
    '請清除暫存模型，或改用較小的 生成／輔助 GPU 權重。'
)
print(f'下載容量檢查：尚需約 {missing_gb:.1f} GB；剩餘 {free_gb:.1f} GB。')

'''
Track newly linked files so a running service can rescan dropdowns.
追蹤新增的模型連結，確保已啟動的服務更新選單。
'''
for repo, filename, folder in models:
    print('模型：', Path(filename).name, flush=True)
    source = Path(hf_hub_download(repo_id=repo, filename=filename, local_dir=STORE / folder))

    '''
    Make the model visible to ComfyUI without copying it.
    利用符號連結讓 ComfyUI 辨識模型，避免複製權重。
    '''
    target = COMFY / 'models' / folder / source.name
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_symlink() and (not target.exists() or not target.samefile(source)):
        target.unlink()
    if not target.exists():
        target.symlink_to(source.resolve())
        H3_RESTART_REQUIRED = True

    '''
    Reject unexpected same-name files to avoid silent model substitution.
    遇到同名但內容不同的權重檔案時，避免默默載入錯誤檔案。
    '''
    assert target.samefile(source), f'模型檔案衝突：{target}'

H3_MODEL_CHOICE = (MODEL_FAMILY, QUANT, ENCODER_VARIANT)
print(f'✓ 第 3 節完成：{TASK.upper()} / {MODEL_FAMILY} / GPU {MODEL_GPU_ID} {QUANT}；GPU {AUX_GPU_ID} {ENCODER_VARIANT}（{TEXT_LOADER_NODE}）；Video/Audio VAE 已就緒。')


4. 可設定 GPU 與 CPU Offload 設定


In [ ]:
'''
Require the teacher settings cell before applying ComfyUI GPU configuration.
套用 ComfyUI GPU 設定前，確認已執行教師共用設定 Cell。
'''
assert all(name in globals() for name in ('GPU0_MODEL_GIB', 'GPU1_MODEL_GIB', 'TEXT_MODE', 'TEXT_GPU', 'NODE_CACHE_MODE')), '請先執行「教師共用設定」Cell。'
gpu0_capacity = torch.cuda.get_device_properties(MODEL_GPU_ID).total_memory / 1024**3
gpu1_capacity = torch.cuda.get_device_properties(AUX_GPU_ID).total_memory / 1024**3

'''
Reject physically impossible budgets and unsupported load modes.
避免設定不可能的 GPU 預算或不支援的載入模式。
'''
assert 0 < GPU0_MODEL_GIB < gpu0_capacity and 0 < GPU1_MODEL_GIB < gpu1_capacity
assert WEIGHTS_IN in ('auto', 'disk', 'ram')
assert TEXT_MODE in ('dynamic', 'streaming', 'resident')

'''
Reserve GPU 0 VRAM for activation tensors and decoder workloads.
保留 GPU 0 的部分 VRAM 給中間 Tensor 與解碼運算。
'''
GPU0_RESERVE_GIB = round(gpu0_capacity - GPU0_MODEL_GIB, 3)

assert NODE_CACHE_MODE in ('classic', 'none')
H3_DESIRED_CONFIG = (MODEL_GPU_ID, AUX_GPU_ID, GPU0_RESERVE_GIB, WEIGHTS_IN, TEXT_GPU, TEXT_MODE, NODE_CACHE_MODE)
print(f'✓ 第 4 節完成：GPU {MODEL_GPU_ID} 目標 {GPU0_MODEL_GIB:.1f} GiB、保留 {GPU0_RESERVE_GIB:.1f} GiB；GPU {AUX_GPU_ID} 使用 {TEXT_MODE}，參考值 {GPU1_MODEL_GIB:.1f} GiB（非硬上限）；offload={WEIGHTS_IN}')


5. 啟動 ComfyUI 與節點相容性檢查


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import requests
import time
LOG = ROOT / 'hyperflow_h3_comfyui.log'
BASE = f'http://127.0.0.1:{PORT}'

'''
Test the ComfyUI API without exposing any public tunnel.
不開放公開網址，直接測試本機 ComfyUI API。
'''
def ready():
    '''
    Handle normal startup connection errors without crashing.
    安全處理服務啟動期間尚未開放連線的情況。
    '''
    try:
        return requests.get(f'{BASE}/system_stats', timeout=3).ok

    except requests.RequestException:
        return False

'''
Read a node's enumerated dropdown options from the live API.
從服務目前註冊的節點規格讀取下拉式選單。
'''
def choices(node_name, input_name):
    '''
    Inspect the requested node's input declaration.
    讀取指定節點的輸入規格。
    '''
    groups = NODE_INFO[node_name].get('input', {})
    for kind in ('required', 'optional'):
        spec = groups.get(kind, {}).get(input_name)

        '''
        Return the dropdown list only when its schema is enumerated.
        只有明確的選單才回傳可選值。
        '''
        if isinstance(spec, (list, tuple)) and spec and isinstance(spec[0], list):
            return spec[0]
        if spec is not None:
            return None
    raise RuntimeError(f'{node_name}.{input_name} 欄位不存在。')

need_restart = globals().get('H3_RESTART_REQUIRED', False) or globals().get('H3_ACTIVE_CONFIG') != H3_DESIRED_CONFIG
if ready() and not need_restart:
    need_restart = globals().get('H3_ACTIVE_MODEL') != H3_MODEL_CHOICE

'''
Shut down only a child process owned by this notebook kernel.
只允許關閉目前 Notebook Kernel 自己管理的服務行程。
'''
if ready() and need_restart:
    '''
    Avoid killing a model service that belongs to a different session.
    避免終止其他 Notebook 或 Session 的模型服務。
    '''
    assert 'H3_PROC' in globals() and H3_PROC.poll() is None, f'埠 {PORT} 已被其他行程使用；請重新啟動 Notebook Session。'
    H3_PROC.terminate()
    try:
        H3_PROC.wait(timeout=20)

    except subprocess.TimeoutExpired:
        H3_PROC.kill()

        '''
        Wait for the kernel to release process resources.
        等待系統完全回收該行程資源。
        '''
        H3_PROC.wait(timeout=10)

if not ready():
    command = [sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', str(PORT), '--cuda-device', str(MODEL_GPU_ID),
               '--enable-dynamic-vram', '--fast-disk', '--reserve-vram', str(GPU0_RESERVE_GIB),
               '--cache-classic' if NODE_CACHE_MODE == 'classic' else '--cache-none',
               '--disable-pinned-memory']

    '''
    Store complete startup warnings in a diagnostic file.
    將完整的啟動警告保存於診斷檔案。
    '''
    with LOG.open('w', encoding='utf-8') as stream:
        H3_PROC = subprocess.Popen(command, cwd=COMFY, stdout=stream, stderr=subprocess.STDOUT)
    for attempt in range(180):
        if H3_PROC.poll() is not None:
            raise RuntimeError(LOG.read_text(errors='replace')[-6500:])
        if ready():
            break
        time.sleep(2)
    assert ready(), LOG.read_text(errors='replace')[-6500:]

H3_ACTIVE_CONFIG = H3_DESIRED_CONFIG

'''
Retrieve registered node types and parameter schemas.
取得已註冊的節點類型與參數規格。
'''
NODE_INFO = requests.get(f'{BASE}/object_info', timeout=90).json()
REQUIRED_NODES = (TEXT_LOADER_NODE, 'HyperFlowLoader', 'HyperFlowSigmas',
                  'MiniMaxH3ImageToVideo', 'MiniMaxH3ReferenceToVideo', 'VAELoader', 'BasicGuider', 'KSamplerSelect',
                  'RandomNoise', 'SamplerCustomAdvanced', 'VAEDecode', 'VAEDecodeAudio',
                  'CreateVideo', 'SaveVideo', 'LoadImage')

missing = [name for name in REQUIRED_NODES if name not in NODE_INFO]

'''
Show actionable errors for incompatible or incomplete plugin installs.
外掛版本不相容時顯示可追蹤的具體缺失。
'''
assert not missing, f'ComfyUI 缺少節點 {missing}。請檢查 {LOG}。'
assert DIT_NAME in (choices('HyperFlowLoader', 'unet_name') or []), f'HyperFlow Loader 未辨識模型：{DIT_NAME}'
assert Path(ENCODER_FILE).name in (choices(TEXT_LOADER_NODE, 'clip_name') or []), '找不到 Qwen3-VL Encoder。'

'''
Validate the projection only when selecting a smaller encoder.
只有選擇小型 Encoder 時才確認 ClipProj 投影矩陣。
'''
if PROJECTION_FILE:
    assert PROJECTION_FILE in (choices(TEXT_LOADER_NODE, 'projection') or []), '找不到 ClipProj v3.1 投影矩陣。'

assert TEXT_GPU in (choices(TEXT_LOADER_NODE, 'device') or []), f'{TEXT_LOADER_NODE} 不支援 {TEXT_GPU}。'
assert TEXT_MODE in (choices(TEXT_LOADER_NODE, 'mode') or []), 'Encoder Loader 不支援指定的 Offload 模式。'

'''
Require the native MiniMax text-encoder architecture when using 32B.
選擇原生 32B 時確認節點有 MiniMax 的文字編碼器型別。
'''
if ENCODER_SIZE == '32B':
    '''
    Avoid loading the 32B weights under an unrelated encoder type.
    避免將 32B 權重當成其他架構載入。
    '''
    assert 'minimax' in (choices(TEXT_LOADER_NODE, 'type') or []), '缺少 MiniMax 原生 Encoder 型別。'

for name in ('minimax_h3_video_vae_fp16.safetensors', 'minimax_h3_audio_vae_fp32.safetensors'):
    assert name in (choices('VAELoader', 'vae_name') or []), f'找不到 VAE：{name}'

assert 'steps' in NODE_INFO['HyperFlowSigmas']['input'].get('optional', {}), 'HyperFlow Sigmas 缺少 steps 欄位。'

'''
Verify the canonical Euler sampler remains available.
確認官方 HyperFlow 所需的 Euler Sampler 已存在。
'''
assert 'euler' in (choices('KSamplerSelect', 'sampler_name') or []), 'Sampler 不支援 euler。'
H3_ACTIVE_MODEL = H3_MODEL_CHOICE
H3_RESTART_REQUIRED = False

'''
Confirm the server is ready for generation.
確認 HyperFlow 服務已可接受生成工作。
'''
print(f'✓ 第 5 節完成：GPU {MODEL_GPU_ID} {QUANT} / GPU {AUX_GPU_ID} {ENCODER_VARIANT} / Disk Offload。')


6. T2V：Prompt 與生成參數


In [ ]:
'''
Describe the text-to-video scene and synchronized sound.
設定文生影片的場景與同步音效描述。
'''
T2V_PROMPT = '''
A cinematic close-up of a red panda in a misty forest. The red panda slowly turns its head.
Natural fur details, gentle camera movement, soft morning light. Sound: birds chirping and light wind.
'''.strip()
T2V_WIDTH, T2V_HEIGHT = 384, 224
T2V_SECONDS = 10

'''
Control video FPS for both the notebook and its remote API.
同時控制 Notebook 與遠端 API 的影片影格率。
'''
T2V_FPS = 24

'''
Select the official HyperFlow 4-, 6-, or 8-step schedule.
選擇官方 HyperFlow 的 4、6 或 8 steps 排程。
'''
T2V_STEPS = 6
T2V_SEED = 42
T2V_OUTPUT = 'hyperflow_h3_flash_t2v.mp4'

'''
Preserve configurations for every task independently.
分開保存各項任務的設定，避免不同 Prompt 互相覆寫。
'''
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})
TASK_CONFIGS['t2v'] = dict(prompt=T2V_PROMPT, width=T2V_WIDTH, height=T2V_HEIGHT,
                           seconds=T2V_SECONDS, fps=T2V_FPS, steps=T2V_STEPS, seed=T2V_SEED,
                           encoder_variant=ENCODER_VARIANT, quant=QUANT,
                           output_name=T2V_OUTPUT, first_image='', last_image='')

'''
Select the current generation task and its model settings.
選擇目前生成模式及此模式的模型設定；尚未執行生成。
'''
TASK = 't2v'

print(f'✓ 第 6 節完成：T2V | {T2V_WIDTH}×{T2V_HEIGHT} | {T2V_SECONDS} 秒 | {T2V_STEPS} steps | {QUANT} | GPU {AUX_GPU_ID} {ENCODER_VARIANT} | {T2V_OUTPUT}')


7. I2V：Prompt、圖片與生成參數


In [ ]:
'''
Describe how the uploaded first frame should animate.
描述上傳的第一張圖片要如何轉化為動態影片。
'''
I2V_PROMPT = '''
Animate the subject in the reference image with subtle natural motion and smooth cinematic camera movement.
Keep the person's or object's identity, clothing and environment consistent. Include realistic ambient sound.
'''.strip()
I2V_FIRST_IMAGE = str(ROOT / 'first_frame.png')
I2V_WIDTH, I2V_HEIGHT = 384, 224

'''
Configure output duration for image-to-video.
設定圖生影片的目標秒數。
'''
I2V_SECONDS = 10

'''
Control video FPS for both the notebook and its remote API.
同時控制 Notebook 與遠端 API 的影片影格率。
'''
I2V_FPS = 24
I2V_STEPS = 6
I2V_SEED = 42

'''
Set an independent image-to-video output filename.
指定圖生影片自己的輸出檔名。
'''
I2V_OUTPUT = 'hyperflow_h3_flash_i2v.mp4'
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})
TASK_CONFIGS['i2v'] = dict(prompt=I2V_PROMPT, width=I2V_WIDTH, height=I2V_HEIGHT,
                           seconds=I2V_SECONDS, fps=I2V_FPS, steps=I2V_STEPS, seed=I2V_SEED,
                           encoder_variant=ENCODER_VARIANT, quant=QUANT,
                           output_name=I2V_OUTPUT, first_image=I2V_FIRST_IMAGE, last_image='')

'''
Confirm the configuration cell has run without attempting to open the image yet.
顯示設定完成；實際圖片檔案會在生成時再驗證。
'''
TASK = 'i2v'

print(f'✓ 第 7 節完成：I2V | {I2V_WIDTH}×{I2V_HEIGHT} | {I2V_SECONDS} 秒 | {QUANT} | GPU {AUX_GPU_ID} {ENCODER_VARIANT} | 圖片：{I2V_FIRST_IMAGE}')


8. L2VA：最後影格生成影片


In [ ]:
'''
Describe how the uploaded last frame should animate.
描述上傳的最後一張圖片要如何轉化為動態影片。
'''
L2V_PROMPT = '''
Animate the subject in the reference image with subtle natural motion and smooth cinematic camera movement.
Keep the person's or object's identity, clothing and environment consistent. Include realistic ambient sound.
'''.strip()
L2V_LAST_IMAGE = str(ROOT / 'last_frame.png')
L2V_WIDTH, L2V_HEIGHT = 384, 224

'''
Configure output duration for image-to-video.
設定最後影格生成影片的目標秒數。
'''
L2V_SECONDS = 10

'''
Control video FPS for both the notebook and its remote API.
同時控制 Notebook 與遠端 API 的影片影格率。
'''
L2V_FPS = 24
L2V_STEPS = 6
L2V_SEED = 42

'''
Set an independent image-to-video output filename.
指定圖生影片自己的輸出檔名。
'''
L2V_OUTPUT = 'hyperflow_h3_flash_l2v.mp4'
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})
TASK_CONFIGS['l2v'] = dict(prompt=L2V_PROMPT, width=L2V_WIDTH, height=L2V_HEIGHT,
                           seconds=L2V_SECONDS, fps=L2V_FPS, steps=L2V_STEPS, seed=L2V_SEED,
                           encoder_variant=ENCODER_VARIANT, quant=QUANT,
                           output_name=L2V_OUTPUT, first_image='', last_image=L2V_LAST_IMAGE)

'''
Confirm the configuration cell has run without attempting to open the image yet.
顯示設定完成；實際圖片檔案會在生成時再驗證。
'''
TASK = 'l2v'

print(f'✓ 第 8 節完成：L2V | {L2V_WIDTH}×{L2V_HEIGHT} | {L2V_SECONDS} 秒 | {QUANT} | GPU {AUX_GPU_ID} {ENCODER_VARIANT} | 圖片：{L2V_LAST_IMAGE}')


9. FLF2V：Prompt、首尾圖片與生成參數


In [ ]:
'''
Describe a video that connects its first and last reference frames.
描述從第一張圖片過渡到最後一張圖片的影片內容。
'''
FLF2V_PROMPT = '''
Create a smooth cinematic transition from the first reference frame to the last reference frame.
Maintain consistent subject details and lighting, with natural intermediate motion and synchronized ambient sound.
'''.strip()
FLF2V_FIRST_IMAGE = str(ROOT / 'first_frame.png')
FLF2V_LAST_IMAGE = str(ROOT / 'last_frame.png')

'''
Set output dimensions for first-and-last-frame generation.
設定首尾影格模式的輸出解析度。
'''
FLF2V_WIDTH, FLF2V_HEIGHT = 384, 224
FLF2V_SECONDS = 10

'''
Control video FPS for both the notebook and its remote API.
同時控制 Notebook 與遠端 API 的影片影格率。
'''
FLF2V_FPS = 24
FLF2V_STEPS = 6

'''
Set this task’s seed; use -1 to randomize each generation.
設定此任務的種子；-1 代表每次生成都重新抽取。
'''
FLF2V_SEED = 42
FLF2V_OUTPUT = 'hyperflow_h3_flash_flf2v.mp4'
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})

'''
Register both conditioning images and all parameters.
登記兩張參考圖片與全部生成參數。
'''
TASK_CONFIGS['flf2v'] = dict(prompt=FLF2V_PROMPT, width=FLF2V_WIDTH, height=FLF2V_HEIGHT,
                             seconds=FLF2V_SECONDS, fps=FLF2V_FPS, steps=FLF2V_STEPS, seed=FLF2V_SEED,
                             encoder_variant=ENCODER_VARIANT, quant=QUANT,
                             output_name=FLF2V_OUTPUT, first_image=FLF2V_FIRST_IMAGE,
                             last_image=FLF2V_LAST_IMAGE)

'''
Select the current generation task and its model settings.
選擇目前生成模式及此模式的模型設定；尚未執行生成。
'''
TASK = 'flf2v'

print(f'✓ 第 9 節完成：FLF2V | {FLF2V_WIDTH}×{FLF2V_HEIGHT} | {FLF2V_SECONDS} 秒 | {QUANT} | GPU {AUX_GPU_ID} {ENCODER_VARIANT} | 參考圖 2 張')


10. R2V：Prompt、參考圖片與生成參數


In [ ]:
'''
Describe the primary picture; the prompt stays valid when selecting only one image.
描述第一張參考圖；即使只選一張圖片，Prompt 仍能正常使用。
'''
R2V_PROMPT = '''
<Picture 1> defines the main subject's appearance.
Create a cinematic scene with natural motion and ambient sound.
Keep the main subject consistent, and use additional pictures as visual references when supplied.
'''.strip()
import ipywidgets as widgets
from IPython.display import display
from PIL import Image
import io
import tempfile
import shutil
from html import escape
R2V_FORMATS = {'.png': 'PNG', '.jpg': 'JPEG', '.jpeg': 'JPEG', '.webp': 'WEBP'}

'''
Preserve the most recently confirmed batch across cell reruns in this Kaggle session.
在同一個 Notebook Session 中重跑 Cell 時，沿用最後一次成功選取的圖片組合。
'''
R2V_REF_DIR.mkdir(parents=True, exist_ok=True)
R2V_ACTIVE_FILES = sorted(p for p in R2V_REF_DIR.iterdir()
                          if p.is_file() and p.suffix.lower() in R2V_FORMATS
                          and len(p.stem) == 3 and p.stem.isdigit())

assert len(R2V_ACTIVE_FILES) <= 9 and all(
    path.stem == f'{index:03d}' for index, path in enumerate(R2V_ACTIVE_FILES, 1)
), 'R2V 圖片資料夾編號不連續；請重新選取整組圖片。'

'''
Keep the public reference list synchronized with the files actually on disk.
讓公開的參考圖片清單與磁碟上的實際檔案保持一致。
'''
R2V_REF_IMAGES = [str(path) for path in R2V_ACTIVE_FILES]
R2V_REF_VIDEOS = []  # Example: ['/home/your_user/ai_models/clip.mp4']
R2V_REF_AUDIOS = []  # Example: ['/home/your_user/ai_models/voice.wav']

'''
Use soundtrack of a reference video if the uploaded video has audio.
若參考影片含音軌，選擇是否同時使用該音軌作為聲音條件。
'''
R2V_VIDEO_AUDIO = False  # True, False
R2V_REF_IMAGE_SIZE = 'match'
R2V_WIDTH, R2V_HEIGHT = 384, 224

'''
Set the approximate video duration in seconds.
設定目標影片秒數，實際影格數會依 H3 規則對齊。
'''
R2V_SECONDS = 10

'''
Control video FPS for both the notebook and its remote API.
同時控制 Notebook 與遠端 API 的影片影格率。
'''
R2V_FPS = 24
R2V_STEPS = 6

'''
Encoder sizes and precisions; B means one billion parameters. Prefer native 32B for picture references; it needs GPU 1 offload on smaller VRAM cards.
Encoder 大小與精度；B 代表十億個參數，32B=320 億、8B=80 億、4B=40 億。 Setting / 設定值  | Model / 模型                 | Precision / 權重精度 32B_NVFP4       | 原生 Qwen3-VL-32B            | NVFP4（4-bit 浮點） 32B_INT8        | 原生 Qwen3-VL-32B            | INT8 ConvRot（8-bit 整數） 32B_BF16        | 原生 Qwen3-VL-32B            | BF16（16-bit 浮點） 8B_NVFP4        | Qwen3-VL-8B + ClipProj      | NVFP4（4-bit 浮點） 8B_W4A8         | Qwen3-VL-8B + ClipProj      | W4A8（4-bit 權重／8-bit activation） 8B_INT8         | Qwen3-VL-8B + ClipProj      | INT8 ConvRot（8-bit 整數） 8B_FP8          | Qwen3-VL-8B + ClipProj      | FP8（8-bit 浮點） 8B_BF16         | Qwen3-VL-8B + ClipProj      | BF16（16-bit 浮點） 4B_FP8          | Qwen3-VL-4B + ClipProj      | FP8（8-bit 浮點） 4B_BF16         | Qwen3-VL-4B + ClipProj      | BF16（16-bit 浮點） 多圖參考預設使用原生 32B；在 VRAM 較小的 GPU 上須透過 GPU 1 卸載管理。 8B／4B + ClipProj 可節省資源，但多圖參考的影像條件品質尚需實測。
'''
R2V_SEED = 42
R2V_OUTPUT = 'hyperflow_h3_flash_r2v.mp4'

'''
Retain the other three task configurations in the same notebook session.
保留同一個 Notebook Session 內其他三種生成模式的參數。
'''
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})
TASK_CONFIGS['r2v'] = dict(prompt=R2V_PROMPT, width=R2V_WIDTH, height=R2V_HEIGHT,
                           seconds=R2V_SECONDS, fps=R2V_FPS, steps=R2V_STEPS, seed=R2V_SEED,
                           encoder_variant=ENCODER_VARIANT, quant=QUANT,
                           output_name=R2V_OUTPUT, first_image='', last_image='',
                           ref_images=R2V_REF_IMAGES, ref_image_size=R2V_REF_IMAGE_SIZE,
                           ref_videos=R2V_REF_VIDEOS, ref_audios=R2V_REF_AUDIOS,
                           video_audio=R2V_VIDEO_AUDIO)

R2V_UPLOAD_STATUS = widgets.HTML()

'''
Keep the active file picker replaceable so the next selection cannot append old uploads.
讓選檔元件可重新建立，確保下次多選不會累積上一批檔案。
'''
R2V_UPLOAD_PANEL = widgets.VBox()
R2V_UPLOAD_HELP = widgets.HTML(value=(
    '<b>R2V 參考圖片（1～9 張）</b>：一次選取所有圖片；每次重新選取會完整取代上一批，'
    '從 001 重新編號，並刪除不再使用的舊圖片。'
))

'''
Apply one complete browser selection as a new image set, never merging it with an earlier batch.
將瀏覽器本次選取視為完整的新圖片組合，絕不與上一批合併。
'''
def on_r2v_upload(change):
    '''
    Update the global image list when a valid new batch is committed.
    新圖片批次確認有效後，更新全域參考圖片清單。
    '''
    global R2V_REF_IMAGES
    uploaded = change['new']
    if not uploaded:
        return
    staging = None
    message = ''

    '''
    Try to stage and atomically replace the entire selected batch.
    嘗試建立暫存圖片組合，再以整批替換方式更新正式資料夾。
    '''
    try:
        assert 1 <= len(uploaded) <= 9, '請一次選取 1～9 張圖片。'
        staging = Path(tempfile.mkdtemp(prefix='.hyperflow_r2v_new_', dir=ROOT))
        mapping = []

        '''
        Keep the browser's selected file order for <Picture 1> onward.
        按照瀏覽器回傳的選取順序，對應 <Picture 1> 等圖片標籤。
        '''
        for index, item in enumerate(uploaded, 1):
            original = item['name'].replace('\\', '/').split('/')[-1]
            extension = Path(original).suffix.lower()
            assert extension in R2V_FORMATS, f'不支援的圖片格式：{original}'
            content = bytes(item['content'])

            '''
            Reject an empty upload before opening the image decoder.
            在開啟圖片解碼器前先拒絕空白檔案。
            '''
            assert content, f'圖片內容為空：{original}'
            with Image.open(io.BytesIO(content)) as picture:
                assert picture.format == R2V_FORMATS[extension], f'圖片內容格式不符：{original}'
                picture.verify()
            filename = f'{index:03d}{extension}'
            (staging / filename).write_bytes(content)
            mapping.append(f'&lt;Picture {index}&gt; = {escape(filename)} ← {escape(original)}')
        backup = ROOT / f'.hyperflow_r2v_old_{__import__("uuid").uuid4().hex}'
        R2V_REF_DIR.rename(backup)
        try:
            '''
            Publish the verified batch as the sole active image directory.
            將已驗證的新批次整批設為唯一有效的圖片目錄。
            '''
            staging.rename(R2V_REF_DIR)

        except Exception:
            backup.rename(R2V_REF_DIR)
            raise
        shutil.rmtree(backup, ignore_errors=True)

        '''
        Read only the currently numbered files in exact Picture order.
        只讀取現在有效的圖片，並依 Picture 順序排列。
        '''
        R2V_REF_IMAGES = [str(path) for path in sorted(R2V_REF_DIR.iterdir()) if path.is_file()]
        TASK_CONFIGS['r2v']['ref_images'] = list(R2V_REF_IMAGES)
        message = '<b>已套用最新 ' + str(len(uploaded)) + ' 張圖片：</b><br>' + '<br>'.join(mapping)

    except Exception as error:
        message = '<b>未更新參考圖片：</b>' + escape(str(error))

    finally:
        '''
        Always clean up any unused staging directory and reset the browser picker.
        無論成功或失敗，均清除暫存目錄並重新建立瀏覽器選檔元件。
        '''
        if staging is not None and staging.exists():
            shutil.rmtree(staging, ignore_errors=True)
        R2V_UPLOAD_STATUS.value = message
        R2V_UPLOAD_PANEL.children = (new_r2v_uploader(), R2V_UPLOAD_STATUS)

'''
Create a new browser file picker and subscribe to its batch-upload event.
建立新的瀏覽器選檔元件，並監聽每批圖片上傳完成事件。
'''
def new_r2v_uploader():
    '''
    Offer one browser picker for multiple image files.
    提供可一次選取多張圖片的瀏覽器元件。
    '''
    uploader = widgets.FileUpload(accept='.png,.jpg,.jpeg,.webp', multiple=True,
                                   description='多選參考圖片')
    uploader.observe(on_r2v_upload, names='value')
    return uploader

R2V_UPLOAD_STATUS.value = ('目前圖片：' + (', '.join(Path(path).name for path in R2V_REF_IMAGES)
                                          if R2V_REF_IMAGES else '尚未選擇'))

R2V_UPLOAD_PANEL.children = (new_r2v_uploader(), R2V_UPLOAD_STATUS)

'''
Render the upload widget directly in this R2V settings cell.
直接在 R2V 參數 Cell 顯示圖片上傳元件。
'''
display(widgets.VBox([R2V_UPLOAD_HELP, R2V_UPLOAD_PANEL]))
'''
Select the current generation task and its model settings.
選擇目前生成模式及此模式的模型設定；尚未執行生成。
'''
TASK = 'r2v'

print(f'✓ 第 10 節完成：R2V | {R2V_WIDTH}×{R2V_HEIGHT} | {len(R2V_REF_IMAGES)} 張參考圖 | {QUANT} | GPU {AUX_GPU_ID} {ENCODER_VARIANT}。素材設定完成後請執行第 11 節 Generate。')


11. 生成影片（T2V／I2V／L2VA／FLF2V／R2V）


In [ ]:
'''
Serialize notebook generation with optional remote FIFO GPU jobs.
讓 Notebook 手動生成與選用的遠端 FIFO 共用 GPU 鎖，避免同時推論。
不必先啟動 ngrok 或任何 HTTP API。
'''
import threading
GENERATION_LOCK = globals().setdefault('GENERATION_LOCK', threading.Lock())
with GENERATION_LOCK:
    import json
    import shutil
    import uuid
    import math
    import websocket
    from tqdm.auto import tqdm
    TASK = globals().get('TASK', 't2v')
    assert TASK in globals().get('TASK_CONFIGS', {}), f'請先執行 {TASK.upper()} 的參數設定 Cell。'
    config = TASK_CONFIGS[TASK]

    '''
    Share sampling, decoding and video output logic across four task modes.
    四種生成模式共用採樣、解碼及影片輸出流程。
    '''
    MODE = TASK
    PROMPT = config['prompt']
    WIDTH, HEIGHT = config['width'], config['height']
    SECONDS, STEPS = config['seconds'], config['steps']

    '''
    Apply the task-specific output frame rate to generation and encoding.
    將各模式的輸出影格率同步用於生成影格數與影片編碼。
    '''
    FPS = config['fps']
    SEED, OUTPUT_NAME = config['seed'], config['output_name']

    '''
    Resolve task-specific reference images without mixing modes.
    依據所選任務使用對應參考圖片，避免不同模式混用。
    '''
    FIRST_IMAGE, LAST_IMAGE = config['first_image'], config['last_image']
    REF_IMAGES = list(config.get('ref_images', []))
    REF_VIDEOS = list(config.get('ref_videos', []))
    REF_AUDIOS = list(config.get('ref_audios', []))
    VIDEO_AUDIO = bool(config.get('video_audio', False))
    REF_IMAGE_SIZE = config.get('ref_image_size', 'match')
    ENCODER_VARIANT = config['encoder_variant']

    '''
    Derive the encoder architecture needed by the selected loader.
    取得 Loader 所需的文字編碼器架構大小。
    '''
    ENCODER_SIZE = ENCODER_VARIANT.split('_', 1)[0]
    QUANT = config['quant']
    assert QUANT in QUANT_SIZES_GB and ENCODER_VARIANT in ENCODERS, f'不支援的 GPU 模型組合：{QUANT}/{ENCODER_VARIANT}'
    assert STEPS in (4, 6, 8), 'HyperFlow STEPS 僅支援 4、6、8。'

    '''
    Ensure edited GPU 0 memory budgets have been applied to the running service.
    確認調整過的 GPU 0 記憶體預算已套用到目前服務。
    '''
    assert globals().get('H3_ACTIVE_CONFIG') == H3_DESIRED_CONFIG, 'GPU 0 預算已變更，請重新執行第 4 節啟動 ComfyUI。'
    assert globals().get('H3_MODEL_CHOICE') == (MODEL_FAMILY, QUANT, ENCODER_VARIANT) and globals().get('H3_ACTIVE_MODEL') == (MODEL_FAMILY, QUANT, ENCODER_VARIANT), (
        f'目前選擇 GPU {MODEL_GPU_ID} {QUANT}／GPU {AUX_GPU_ID} {ENCODER_VARIANT}，但尚未完成下載或服務預檢。請重跑第 2 節與第 4 節。'
    )
    assert MODEL_FAMILY == ('ref2va' if TASK == 'r2v' else 'fl2va'), '目前 GPU 0 模型架構與任務不一致，請重跑第 2、4 節。'
    assert DIT_NAME == f'minimax_h3_{MODEL_FAMILY}_hyperflow_flash_14b-{QUANT}.gguf', 'GPU 0 模型檔名不一致，請重跑第 2 節。'

    '''
    Reject a stale GPU 1 encoder that belongs to an earlier selection.
    避免切換 GPU 1 模型後仍使用前一次載入的 Encoder。
    '''
    assert Path(ENCODER_FILE).name == ENCODERS[ENCODER_VARIANT][1], '輔助 GPU 的模型檔名不一致，請重跑第 2 節。'
    LAST_OUTPUT = None
    assert ready(), 'ComfyUI 已中斷，請先重新執行第 3 節。'
    assert MODE in ('t2v', 'i2v', 'l2v', 'flf2v', 'r2v') and PROMPT.strip() and STEPS > 0 and math.isfinite(SECONDS) and SECONDS > 0, (
        '請確認生成模式、Prompt、STEPS 與 SECONDS 設定正確；SECONDS 必須為正數。'
    )

    '''
    Require positive dimensions aligned to 32, without an artificial resolution limit.
    寬高需為正數且符合 32 的倍數，不另外限制解析度或長寬比。
    '''
    assert WIDTH > 0 and HEIGHT > 0 and WIDTH % 32 == 0 and HEIGHT % 32 == 0, (
        'WIDTH 與 HEIGHT 必須為正數且是 32 的倍數。'
    )
    assert isinstance(FPS, int) and not isinstance(FPS, bool) and 1 <= FPS <= 60, 'FPS 必須是 1～60 的整數。'
    FRAMES = max(22, 17 * math.ceil((SECONDS * FPS - 5) / 17) + 5)
    assert FRAMES <= 3600, f'影格數 {FRAMES} 超出官方 H3 節點上限 3600，請縮短影片秒數。'
    import secrets
    RUN_SEED = secrets.randbelow(2**32) if SEED == -1 else int(SEED)

    '''
    Require a supported seed before building the inference workflow.
    在建立推論工作流程前，先檢查種子範圍。
    '''
    assert 0 <= RUN_SEED < 2**32, 'SEED 必須介於 0 至 4294967295，或設為 -1。'
    print(f'本次生成 seed：{RUN_SEED}')
    output_stem = Path(OUTPUT_NAME).stem
    run_name = f'{output_stem}_{QUANT}_{uuid.uuid4().hex[:8]}'
    text_inputs = {'clip_name': Path(ENCODER_FILE).name,
                   'type': 'minimax' if ENCODER_SIZE == '32B' else 'auto',
                   'device': TEXT_GPU, 'mode': TEXT_MODE}

    '''
    Add the learned projection only when a 4B/8B encoder is selected.
    只有使用 4B／8B Encoder 時才接入學習式投影矩陣。
    '''
    if PROJECTION_FILE:
        text_inputs['projection'] = PROJECTION_FILE
    graph = {
        '1': {'class_type': TEXT_LOADER_NODE, 'inputs': text_inputs},
        '2': {'class_type': 'HyperFlowLoader', 'inputs': {'unet_name': DIT_NAME, 'weights_in': WEIGHTS_IN}},
        '4': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'minimax_h3_video_vae_fp16.safetensors'}},
        '5': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'minimax_h3_audio_vae_fp32.safetensors'}},
        '6': {'class_type': ('MiniMaxH3ReferenceToVideo' if MODE == 'r2v' else 'MiniMaxH3ImageToVideo'), 'inputs': {
            'clip': ['1', 0], 'vae': ['4', 0], 'prompt': PROMPT,
            'width': WIDTH, 'height': HEIGHT, 'length': FRAMES}},
        '7': {'class_type': 'BasicGuider', 'inputs': {
            'model': ['2', 0], 'conditioning': ['6', 0]}},
        '8': {'class_type': 'KSamplerSelect', 'inputs': {'sampler_name': 'euler'}},
        '9': {'class_type': 'HyperFlowSigmas', 'inputs': {'model': ['2', 0], 'steps': STEPS}},
        '10': {'class_type': 'RandomNoise', 'inputs': {'noise_seed': RUN_SEED}},
        '11': {'class_type': 'SamplerCustomAdvanced', 'inputs': {
            'noise': ['10', 0], 'guider': ['7', 0],
            'sampler': ['8', 0], 'sigmas': ['9', 0], 'latent_image': ['6', 1]}},
        '12': {'class_type': 'VAEDecode', 'inputs': {'samples': ['11', 0], 'vae': ['4', 0]}},
        '13': {'class_type': 'VAEDecodeAudio', 'inputs': {'samples': ['11', 0], 'vae': ['5', 0]}},
        '14': {'class_type': 'CreateVideo', 'inputs': {
            'images': ['12', 0], 'audio': ['13', 0], 'fps': FPS}},
        '15': {'class_type': 'SaveVideo', 'inputs': {
            'video': ['14', 0], 'filename_prefix': run_name, 'format': 'mp4'}},
    }
    if MODE == 'r2v':
        assert isinstance(REF_IMAGES, (list, tuple)) and 0 <= len(REF_IMAGES) <= 9, 'R2V_REF_IMAGES 最多 9 張；也可以改用影片或音訊。'
        assert all(Path(path).is_file() for path in REF_IMAGES), 'R2V 有不存在的圖片；請重新選擇。'

        '''
        Restrict the reference resize mode to the two official choices.
        參考圖尺寸選項只能填寫官方支援的 match 或 max。
        '''
        assert REF_IMAGE_SIZE in ('match', 'max'), 'R2V_REF_IMAGE_SIZE 只能是 match 或 max。'
        import re
        mentioned = {int(number) for number in re.findall(r'<Picture\s+(\d+)>', PROMPT, re.IGNORECASE)}
        assert mentioned <= set(range(1, len(REF_IMAGES) + 1)), 'Prompt 的 <Picture N> 必須對應已提供圖片。'
        graph['6']['inputs']['audio_vae'] = ['5', 0]

        '''
        Apply the documented image-sizing method for every reference.
        對所有參考圖套用官方定義的縮放模式。
        '''
        graph['6']['inputs']['ref_image_size'] = REF_IMAGE_SIZE
        for index, image_path in enumerate(REF_IMAGES):
            source = Path(image_path)
            assert source.is_file() and source.stat().st_size > 0 and source.suffix.lower() in ('.png', '.jpg', '.jpeg', '.webp'), f'無效的 R2V 參考圖片：{source}'
            node_id = str(20 + index)
            destination = COMFY / 'input' / f'r2v_{index + 1}_{uuid.uuid4().hex[:8]}{source.suffix.lower()}'

            '''
            Create the input directory if it does not exist.
            視需要建立 ComfyUI input 資料夾。
            '''
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
            graph[node_id] = {'class_type': 'LoadImage', 'inputs': {'image': destination.name}}
            graph['6']['inputs'][f'ref_images.ref_image_{index}'] = [node_id, 0]
    if MODE == 'r2v':
        assert len(REF_IMAGES) <= 9 and len(REF_VIDEOS) <= 3 and len(REF_AUDIOS) <= 3, 'R2V 最多 9 張圖片、3 段影片、3 段獨立音訊。'
        assert REF_IMAGES or REF_VIDEOS, 'R2V 至少需要一張參考圖片或一段參考影片；音訊可作為附加條件。'
        assert len(REF_IMAGES) + len(REF_VIDEOS) + len(REF_AUDIOS) <= 12, 'R2V 參考素材合計最多 12 個檔案。'
        tags = {'Picture': len(REF_IMAGES), 'Video': len(REF_VIDEOS),
                'Audio': len(REF_AUDIOS) + (len(REF_VIDEOS) if VIDEO_AUDIO else 0)}
        for label, count in tags.items():
            numbers = [int(n) for n in re.findall(r'<%s\s+(\d+)>' % label, PROMPT, re.I)]
            assert all(1 <= n <= count for n in numbers), f'<{label} N> 編號超出實際上傳數量：{numbers}。'
        for index, filename in enumerate(REF_VIDEOS):
            original = Path(filename)
            assert original.is_file() and original.suffix.lower() in ('.mp4', '.mov', '.webm'), f'參考影片不存在或格式不正確：{original}'
            destination = COMFY / 'input' / f'h3_ref_video_{uuid.uuid4().hex[:12]}{original.suffix.lower()}'
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(original, destination)
            node_id = str(120 + index * 2)
            parts_id = str(121 + index * 2)
            graph[node_id] = {'class_type': 'LoadVideo', 'inputs': {'file': destination.name}}
            graph[parts_id] = {'class_type': 'GetVideoComponents', 'inputs': {'video': [node_id, 0]}}
            graph['6']['inputs'][f'ref_videos.ref_video_{index}'] = [parts_id, 0]
            if VIDEO_AUDIO:
                graph['6']['inputs'][f'ref_video_audios.ref_video_audio_{index}'] = [parts_id, 1]
        for index, filename in enumerate(REF_AUDIOS):
            original = Path(filename)
            assert original.is_file() and original.suffix.lower() in ('.wav', '.mp3', '.flac', '.ogg', '.m4a'), f'參考音訊不存在或格式不正確：{original}'
            destination = COMFY / 'input' / f'h3_ref_audio_{uuid.uuid4().hex[:12]}{original.suffix.lower()}'
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(original, destination)
            node_id = str(140 + index)
            graph[node_id] = {'class_type': 'LoadAudio', 'inputs': {'audio': destination.name}}
            graph['6']['inputs'][f'ref_audios.ref_audio_{index}'] = [node_id, 0]

    '''
    Attach a verified local image to an H3 image-conditioning socket.
    將確認存在的本機圖片接到 H3 圖片條件輸入。
    '''
    for mode, image_path, node_id, input_name in [
        ('i2v', FIRST_IMAGE, '16', 'first_frame'),
        ('flf2v', FIRST_IMAGE, '16', 'first_frame'),
        ('flf2v', LAST_IMAGE, '17', 'last_frame'),
        ('l2v', LAST_IMAGE, '18', 'last_frame'),
    ]:

        '''
        Only attach images required by the active mode.
        只接入目前模式需要的圖片。
        '''
        if MODE == mode:
            source = Path(image_path)

            '''
            Stop before generation if a reference file is missing.
            若圖片不存在，在生成前立即停止。
            '''
            assert source.is_file(), f'找不到參考圖片：{source}'
            destination = COMFY / 'input' / f'{node_id}_{uuid.uuid4().hex[:8]}{source.suffix.lower()}'
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
            graph[node_id] = {'class_type': 'LoadImage', 'inputs': {'image': destination.name}}

            '''
            Connect the loaded image to H3 conditioning.
            將已載入圖片連到 H3 條件輸入。
            '''
            graph['6']['inputs'][input_name] = [node_id, 0]
    NODE_INFO = requests.get(f'{BASE}/object_info', timeout=90).json()
    assert TEXT_GPU in (choices(TEXT_LOADER_NODE, 'device') or []), 'Encoder Loader 不支援 GPU 1。'
    assert TEXT_MODE in (choices(TEXT_LOADER_NODE, 'mode') or []), 'Encoder Loader 不支援指定 Offload 模式。'

    '''
    Verify that the official HyperFlow Euler sampler is available.
    確認官方 HyperFlow Euler Sampler 可用。
    '''
    assert 'euler' in (choices('KSamplerSelect', 'sampler_name') or []), 'HyperFlow 必須使用 Euler Sampler。'
    assert 'HyperFlowSigmas' in NODE_INFO, '缺少 HyperFlow Sigma 節點。'
    if MODE in ('i2v', 'l2v', 'flf2v'):
        frame_inputs = NODE_INFO['MiniMaxH3ImageToVideo'].get('input', {}).get('optional', {})
        expected = ('first_frame', 'last_frame') if MODE == 'flf2v' else (('last_frame',) if MODE == 'l2v' else ('first_frame',))
        assert all(field in frame_inputs for field in expected), f'MiniMaxH3ImageToVideo 缺少 {expected} 輸入。'
    if MODE == 'r2v':
        required_nodes = {'LoadVideo', 'GetVideoComponents'} if REF_VIDEOS else set()
        if REF_AUDIOS:
            required_nodes.add('LoadAudio')
        missing_nodes = required_nodes - set(NODE_INFO)
        assert not missing_nodes, f'ComfyUI 缺少參考媒體節點：{sorted(missing_nodes)}，請更新 ComfyUI。'

    '''
    Validate that all graph dependency links target existing nodes.
    驗證所有 Workflow 連線的來源節點確實存在。
    '''
    for node_id, node in graph.items():
        '''
        Examine each declared input for possible node-reference links.
        檢查節點輸入中是否包含連到其它節點的參照。
        '''
        for value in node['inputs'].values():
            if isinstance(value, list) and len(value) == 2:
                assert str(value[0]) in graph, f'節點 {node_id} 連結到不存在的節點 {value[0]}。'
    for node_id, node in graph.items():
        schema = NODE_INFO[node['class_type']].get('input', {})
        accepted = set().union(*(set(schema.get(group, {})) for group in ('required', 'optional', 'hidden')))
        if node['class_type'] == 'MiniMaxH3ReferenceToVideo':
            assert 'ref_images' in accepted, 'MiniMaxH3ReferenceToVideo 不支援 ref_images；請更新 ComfyUI。'
            for family, paths in (('ref_videos', REF_VIDEOS), ('ref_audios', REF_AUDIOS), ('ref_video_audios', REF_VIDEOS if VIDEO_AUDIO else [])):
                assert not paths or family in accepted, f'目前 ComfyUI Ref2VA 不支援 {family}。'
            accepted.update(f'ref_images.ref_image_{index}' for index in range(len(REF_IMAGES)))
            accepted.update(f'ref_videos.ref_video_{index}' for index in range(len(REF_VIDEOS)))
            accepted.update(f'ref_video_audios.ref_video_audio_{index}' for index in range(len(REF_VIDEOS)) if VIDEO_AUDIO)
            accepted.update(f'ref_audios.ref_audio_{index}' for index in range(len(REF_AUDIOS)))
        unknown = set(node['inputs']) - accepted

        '''
        Fail early if a workflow input is no longer supported.
        當 Workflow 參數不再被節點支援時，立即停止並指出名稱。
        '''
        assert not unknown, f"節點 {node_id} ({node['class_type']}) 不支援的輸入：{sorted(unknown)}"
    print(f'{MODE.upper()} | {WIDTH}×{HEIGHT} | {FRAMES} frames | {STEPS} steps | GPU {MODEL_GPU_ID} {QUANT} | GPU {AUX_GPU_ID} {ENCODER_VARIANT} | HyperFlow', flush=True)
    client_id = uuid.uuid4().hex
    ws = None

    '''
    Connect before queueing to avoid missing early progress messages.
    在送出任務前先連線，避免漏掉初期進度事件。
    '''
    try:
        ws = websocket.create_connection(f'ws://127.0.0.1:{PORT}/ws?clientId={client_id}', timeout=5)
        ws.settimeout(2)

    except (OSError, websocket.WebSocketException):
        ws = None

        '''
        Explain why no step percentage will be shown.
        說明為什麼目前不會出現步數百分比。
        '''
        print('即時步數事件不可用；將顯示階段與經過時間。', flush=True)
    response = requests.post(f'{BASE}/prompt', json={'prompt': graph, 'client_id': client_id}, timeout=120)
    if not response.ok:
        if ws is not None:
            ws.close()
        raise RuntimeError(response.text[:5000])
    prompt_id = response.json()['prompt_id']
    start = time.monotonic()

    '''
    Create a single notebook progress bar with real step events only.
    建立單一 Notebook 進度條，僅使用真實採樣步數更新。
    '''
    pbar = tqdm(total=STEPS, desc='排隊／載入模型', unit='step', leave=True)
    item = None
    last_history_check = 0.0
    phases = {'1': '文字編碼器', '2': '載入 HyperFlow GGUF', '6': '建立影片條件',
              '11': '採樣中', '12': '影片解碼', '13': '音訊解碼',
              '14': '合成影片', '15': '儲存影片'}

    '''
    Wait until ComfyUI reports a final history record for this task.
    持續等待 ComfyUI 回傳此任務的最終執行紀錄。
    '''
    try:
        while True:
            '''
            Measure real wall-clock time instead of estimating model progress.
            計算真實經過時間，不將時間誤當模型百分比。
            '''
            elapsed = time.monotonic() - start
            message = None
            if ws is not None:
                try:
                    message = ws.recv()

                except websocket.WebSocketTimeoutException:
                    pass

                except (OSError, websocket.WebSocketException):
                    '''
                    Release the unusable client connection.
                    關閉失效的 WebSocket 連線。
                    '''
                    ws.close()
                    ws = None
            if isinstance(message, str):
                event = json.loads(message)
                data = event.get('data') or {}

                '''
                Reject progress from another client or generation.
                忽略非本次任務的進度，避免步數混亂。
                '''
                if data.get('prompt_id') == prompt_id:
                    if event.get('type') == 'executing' and data.get('node'):
                        pbar.set_description(phases.get(str(data['node']), '執行中'), refresh=False)
                    if event.get('type') in ('progress', 'progress_state'):
                        progress = data
                        if event.get('type') == 'progress_state':
                            progress = data.get('nodes', {}).get('11', {})

                        '''
                        Ignore progress emitted by loading or decoding nodes.
                        忽略非採樣節點的進度，以免誤顯示百分比。
                        '''
                        if str(progress.get('node', progress.get('node_id'))) == '11':
                            value, maximum = progress.get('value'), progress.get('max')
                            if isinstance(value, (int, float)) and isinstance(maximum, (int, float)) and maximum > 0:
                                pbar.total = int(maximum)
                                pbar.n = max(0, min(int(value), pbar.total))
                                pbar.set_description('採樣中', refresh=False)

            '''
            Check the persistent history regardless of WebSocket availability.
            不論 WebSocket 是否正常，都定期檢查 ComfyUI 任務紀錄。
            '''
            if elapsed - last_history_check >= 4:
                last_history_check = elapsed
                if not ready():
                    '''
                    Preserve recent server logs for actionable diagnostics.
                    保留最近服務紀錄供後續除錯。
                    '''
                    raise RuntimeError('ComfyUI 服務中斷：\n' + LOG.read_text(errors='replace')[-4500:])
                item = requests.get(f'{BASE}/history/{prompt_id}', timeout=60).json().get(prompt_id)

                '''
                Stop polling once the job has a final record.
                任務產生最終紀錄後停止輪詢。
                '''
                if item:
                    errors = [m for m in item.get('status', {}).get('messages', []) if m[0] == 'execution_error']
                    if errors:
                        raise RuntimeError(str(errors[-1][1])[:5000])
                    break
            pbar.set_postfix_str(f'{elapsed / 60:.1f} 分鐘', refresh=False)
            pbar.refresh()
            if ws is None:
                time.sleep(1)
        pbar.n = pbar.total

        '''
        Label the status as completed.
        將顯示狀態設為完成。
        '''
        pbar.set_description('完成', refresh=False)
        pbar.set_postfix_str(f'{(time.monotonic() - start) / 60:.1f} 分鐘', refresh=False)
        pbar.refresh()

    finally:
        pbar.close()

        '''
        Close the WebSocket when it exists.
        若 WebSocket 仍存在，關閉它以釋放資源。
        '''
        if ws is not None:
            ws.close()
    files = [p for p in (COMFY / 'output').rglob('*') if p.is_file() and p.name.startswith(run_name)]
    files = [p for p in files if p.suffix.lower() in ('.mp4', '.webm', '.mov')]
    assert files, f'工作流程完成但未找到影片：{item.get("outputs", {})}'
    source_video = max(files, key=lambda path: path.stat().st_mtime)

    '''
    Save the user-visible output in 設定的工作 storage.
    將成果儲存至 設定的工作 目錄。
    '''
    OUTPUT_DIR = ROOT / 'hyperflow_h3_outputs'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_base = OUTPUT_DIR / f'{Path(OUTPUT_NAME).stem}_{QUANT}{source_video.suffix}'
    output = output_base

    '''
    Number duplicate filenames deterministically.
    依序為重複檔名編號。
    '''
    for index in range(2, 1000):
        if not output.exists():
            '''
            Preserve the current unique output path.
            保留目前可用的輸出檔名。
            '''
            break
        output = output_base.with_name(f'{output_base.stem}_{index:03d}{output_base.suffix}')
    shutil.copy2(source_video, output)
    print(f'完成：{output}｜耗時 {(time.monotonic() - start) / 60:.1f} 分鐘')
    LAST_OUTPUT = output
    print('✓ 第 11 節完成：影片已生成；請執行第 12 節預覽與下載。')


12. 影片預覽與下載


In [ ]:
'''
Display the most recently generated video and its download link.
顯示最近一次成功生成影片的預覽與下載連結。
'''
from IPython.display import Video, FileLink, display
assert globals().get('LAST_OUTPUT') is not None and LAST_OUTPUT.is_file(), '請先執行第 9 節生成影片。'
display(Video(str(LAST_OUTPUT), embed=True, width=480))

'''
Provide a clickable file link for the generated video.
顯示本機影片的下載連結。
'''
display(FileLink(str(LAST_OUTPUT)))

'''
Keep the preview cell independent of GPU vendor-specific monitoring tools.
預覽 Cell 不再呼叫 NVIDIA 專用的 nvidia-smi。
'''
print(f'✓ 第 12 節完成：影片預覽與下載連結已顯示；{LAST_OUTPUT.name}')


13. 擷取最後影格（選用）


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import json
from IPython.display import FileLink, display
from PIL import Image
import shutil
assert shutil.which('ffmpeg') and shutil.which('ffprobe'), '找不到 ffmpeg 或 ffprobe；請先安裝 FFmpeg。'
assert globals().get('LAST_OUTPUT') is not None and LAST_OUTPUT.is_file(), '請先執行第 9 節生成影片，並確認第 10 節可正常預覽。'

'''
Build a PNG path next to the generated video.
在生成影片旁建立最後影格 PNG 檔案路徑。
'''
LAST_FRAME_PATH = LAST_OUTPUT.with_name(LAST_OUTPUT.stem + '_last_frame.png')
probe = subprocess.run(
    ['ffprobe', '-v', 'error', '-count_frames', '-select_streams', 'v:0',
     '-show_entries', 'stream=nb_read_frames', '-of', 'json', str(LAST_OUTPUT)],
    capture_output=True, text=True, check=True,
)
last_index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1

'''
Save the exact last frame as a PNG for the next I2V clip.
將最後一個影格儲存成 PNG，供下一段 I2V 使用。
'''
subprocess.run(
    ['ffmpeg', '-y', '-loglevel', 'error', '-i', str(LAST_OUTPUT),
     '-vf', rf'select=eq(n\,{last_index})', '-frames:v', '1', str(LAST_FRAME_PATH)],
    check=True,
)
display(Image.open(LAST_FRAME_PATH))
display(FileLink(str(LAST_FRAME_PATH)))
print(f"下一段：請在第 7 節 I2V 設定 I2V_FIRST_IMAGE = '{LAST_FRAME_PATH}'")

'''
Confirm completion of the extracted-frame export.
確認最後影格已成功輸出。
'''
print(f'✓ 第 13 節完成：已擷取最後影格；{LAST_FRAME_PATH.name}')


14. （選用）啟動 ngrok 與 FIFO 遠端 API


In [ ]:
'''
Require the first configuration cell before enabling the optional remote API.
選用遠端 API 啟動前，必須先執行第一格共用參數設定。
'''
assert all(name in globals() for name in ('REMOTE_PORT', 'REMOTE_SERVICE_ID', 'REMOTE_ROOT')), '請先執行第 1 節共用參數設定。'
import zlib
import math
import re

import queue
import copy
import secrets
import io
import json
import os
import shutil
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.request
import uuid
import zipfile
from email import policy
from email.parser import BytesParser
from getpass import getpass
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import urlsplit, parse_qs

REMOTE_KIND = 'video'
REMOTE_FIELDS = ['task', 'prompt', 'seed', 'width', 'height', 'seconds', 'fps', 'steps', 'ref_image_size', 'quant', 'video_audio']
REMOTE_IS_LTX = False
REMOTE_SCRIPT = zlib.decompress(b'x\xda\xd5}ks\x14W\x96\xe0w~EFGLTU[\xc8\x92\xc0/\xa6\xe9\r\x0c\x02k\xc6\x06\x16\x81{g\x19B\x91\xaaJI\xd9\xd4\xab+\xab\x10\x8cV\x11\x12 $\x1eB\xb2\xcd\xc3\x80x\x99\x87\xd5<$\xb0\r\x08\x01\xe2\xc3\xfc\x93me=>\xf9/\xecy\xdd\x9b7\xb3\xb2\x84h\xf7\xec\xc6\xf6\xc4`UV\xd6\xbd\xe7\x9e{\xeey\x9fs\x13\x89\xc4\x86\x9e\\\xb1P*[\xe5!\xc7\xca\xba\xfd%\xbb\xe4:\x9e\x95w\x9c\x8c\x93\xb1\x06\n%\xf8\xc2\xf5,\xcfI\x97\xddB\xbe}\x83\x7fn\xd1\x9f\xb8W\x9d{T]xV\x9b?\xeb\xbf\x9e\xa9\x9e\x1ek\xcc\x8d\xd5\xae\x9e\xf4\'\xdf\xc0G\x7f\xf9\xe1\xdf\xc6\x8eoH\xc0\xc8.\x8f\xfcg\xaf\x90W\x7f{C\x95\xb2\x9bU\x9f*\x157\xa3\xfe\xce\xd9\xe5!\xf5\xf7\xb0\xd3\xef\x15\xd2\x87\x9d\xf2\x86\x81R!g\x95\xff\x92\xc9\xb5\xdb\x95r\xc1\x92\xef\xf1\xc1\x06\x9a\xe2\x80\xe7\x10\xe4e\xdb;l9G\x8bY7\xed\x96\xb3\xc7\x00\xde,@\x0c+p\xf3\xf4}\xae\x90q\xb2\x1b3\x85\xe1|\xb6`g\x82\xe5T\x7fz[\xbb0_\x9d\xbf\xed\xdf8\xbb\xbat\xb6\xfe\xfat\xed\xe1\xad\xda\xe2X\xf5\xdc\xa4\xbfp\x15\x16e\xed\xdf\xd6\xfb\xaf\xbf\xbe>W\xfbay\xf5\xeduk\xd7\xde\x03V\x87\xc5oV\x1f\xdf\x81\xa7\xd5\x1b\xcf\xab?\x9e\x85%[{\x8a8\xa6g}h\xf93\x8b\x8d\xf1%\x7f\xec\xf5\x16+Q\xee:\x92h\xb3\x12.\xffg ;\xc0\x7f\x94\xe0?\nO8\x85\xb5\xd5\x1a\xcc\x16\xfa\xed\xac\x97L\xb5\x0f:\xe5d\x02\x9f\xe2\x9b8@\x8aW\xbb\xcf\x01\xd8q9zy\xb8\xee\x84g\x15K\xce\x11\xb7P\xf1`\xe5\xceQ\']\xc1\xaf\xd2\x85\xfc\x80;X)\xd9\x08\x95\x95v\xb2\xd9\xf6\r\xf5\x851\x7f\xe6\x12\xee\xd8\xf8\xd2\xea\xabW\xfe\xd9+\xfe\x8b\x9f\xfc\x9b/\xea\xb7\xcf5\xc6\xcf\xc3r\xeb\xf3\x8fa\xdd\xd6vx[Ag{\x9e\x038\' \x01\x9b1P\xf6m\xdf\xb3{g\xcf\xae^\x80vd4\xd5f\r$\xea\x0f\xcf\xfa\x13S<\xb05\x82\xef\xb4W\x8aE\xa7\x94L\x8dZH)3\'\xaa\x17\x97\xc2s%\xd4\x12\xbdB\xf6\x88\x13\xbbJ7\x9fq\x8a\x0e\xfc\x93\x07Br\xcae7?\xe8\x01E\xce\\\xf2W.\x9bk\x82\x19j\xd3\xf35\x80\x81\xe6Q\x0ba\x84\x00\xa2M\x98\x0f\xe2\x87C<w\xef\x90]\x82Y\xed\x1c\x90Q~\xb0\xcd\xca8\xe9B\x06\xfe\xb2\xec|\xc6:\xe2f\x9c\x82U\xa8\x94\x8b\x95\xb2\x95-\x0c\xbai\xcbN\x97\n\x9e\x07\x87\xa4Rb\x02D*C\x88\xae]\xab\xcd/\xd4.\xdc\xacN\xcd"i\xc1\xa1\x98x\x8adv\xfevu\xfe\xce\xdf\xc6\xc6\xeb?\x02\xe5\xbc\xf6g\xce\xf8o\x9e\xd6NO\xd6_/\xf9\x93\xcb\xd5g\xe3p\xa0\x14\xb0_\xed\xd9\xd1-\xa0\x86)\xbd\x08\'\xa2\x08\xeb/\x17J\x01y\x870%{\xbd\xfa\x06I\xbbvm\xc1?=\xcdx1\x10n\xd5\xef\xdcF\xea\xdeK\xa3\xa9I\xf7\xee\xdb\xf3\xd5\xde\xfd0-\xa3\xea`\x82\'K\x1c\n\x83\xa0gK\xdb\xf9#\xb6g\xee\xc5\xbd\xcb\xb8\xce\xf0^\xf0\xf2\xfc\'\xcb\xfe\xa2\xde\x8b?\xf5\xec\xd8\xffE\x9b\xf5Ew\xcf\xae/\xcc\t\x87\xddLy(q\xa8M?\x18r\xdc\xc1\xa1&\x08p\x99\x1b\xbd\xa2\x93v\x07`\x1f2\x8a\xc8q\x9f\xd4\xf6\xc1\x08\x95|\xb9\x05H\xb5\x1f\xbfE\x02\x9c\x9a\xe4-\xa9>\xbeg\xd0Io7\xd0\xc6\x8e\xde6\xabw\x7f\xf7\xde^\x038\xe0\x1a\x85|\xc63\xc1\xf3\xcaN\xd1k\x89\x9f\x9d\xf0sb\xa1\x0e\xf2-;ke\x9d\xfc`y\x88\xe0\xc4\x17\x85\x9a`\xb0\xb2\xed\xe6\x9d\x12l\xda\xca\xf5\xeawWy\xd3\xe8\xe7\xf5\xf9\xa9\xda\xc2e\xa6%\xa0\x96\xea\xad\xd7\x00)\xf0\xa3\xd5\xa5\xfb\xbc\x9d\x82]"$~\xa1v~R-eg\x08\xfe\x01\x02U\xce\xb3\xeb\xb9y\xafl\xe7\xd3N\x12\xdej\x03J*\xa7\x08\xb0|\xa1\xf9\xcb\xfeB!\xcb\xdfvZ\x7f\xd8J\x80\xc1\x7f>\xee\x00\xf6\x84\x7f\xfbo\'\x1a\xb7\xa6\xaa\xdf/Z\x9d\xbf\xbe\xb9\xf1q\x07\x9e\xf1\xea\xc5_\x18\xa7r\xb0\xb7\x15\x8b\xc0\x9d\x9a7\xcf\x03IC#\x0b2\x06\\@\x92\x9dsZ\xed\xdc\xfc\x82\xffx\x16vN\x0e\xcd\x83\x0b\xfe\xect\xb0s\xdd;\xda\xac=\x07\xf6\xef=\xb0\xbfo\xf7\xb6\xaf\xbaC\x9b\xe7d\xcc\x9d\xe3\xe9\xfap*\xb5\x7f\x9a\xf7\x84\x00,9\x03N\xc9\x01T\x80\xf4\xb1\x07A6\x0e\xbb\xe5!\xf8\xb5\x95s\x8f"\x9d\xc9\xa1\xe7\x9d3\xa1\xe5\x03\xe8?\x99\xaeN\x9e\x06NT\x1f;\xe1\xcf]\x82]\x82\xddk\x8c\xbf\xf5\'\xa6W\x97\xa6\xfd\xd9s\xcc!\xaa/^\xc0\xcbz\xdfz\xf6\xf5\xee\xef\xeb\xf9j\xdb\xae\xee6\xeb\xcbm\xeaos/\xdd\x92W\xee#\x88\xccUe\xed\xe0\xa9!3\ny\xc1}\xae\xe0\x95aEi`\xa1\xf0\x84~U\xca\xc1\x06\x14\xddt\xb9RR\xa4\x8bg\t\xe9v_\xd7\xd7\xed\x1b\xe0\x1f\x10h\x0fDt\xcc\x8d\xf9+@|c\xd5G\xb7\x91"\xcf\xdc\xf4\xef^EY\xb7r\x0e\xf9:\xad\xaf\xf6\xec\xa4?;\xa5\x96\xb2\xaf{\'\x03\x8f\x94\x98u\xbdr\x92ae\xf1\x01\xc8ep=\x10\x1e\x07\x0f\xa5D\xd0\xf5\x12\x18V\x81$*\x9c\x9b`\x0f\x88\r{|\xd2\x81@3v\xb6\x90w,\xbb\x92q\x0bJ\x14\x008\xc8\xf8H\xcc \xd2\x99\xc7NM\xb2Lh\xdc\xfc\xb9>\x7f\xc6\x84\xee\xeb\x9e\x1d\xdd{ZB\xc7\x13*\xe8\xf0\xfdm\x07v\xf4\xb4~\x9f@\xd1\xef\xd3\xd8\xfc\x0b\xf8\x01\x1e\xa3\xd0\x0fhp\xfe\t\xfcb\'\xc8UG\xa1`\xfbP\xa1\x00\xfc$Xy\xc9\xf1\xdc\xffP\x04g%A_J\x0fY\xb0I9\xfbh*\xba[\x01\xb11\x11Z\xfcvu\xea\x12\xben1/\xae_\xb9X\x9b=\xc5\xe4\xd7\xb4[}\xbd=\xff3\xa0\xb7\xc8f\xf5\x01$\x0e\xea&4lH;A\x05\xa9\x13d\xb5s\xd4\x86\r,;G\xcb\x16@\x0f\x10\x97\xac#\xa0Z\xda \xbaY\xa7C\xadR\xe4zHS\x81]\\\xb9\x81\xfa\xe5\xa3\xdb\xa6\xd4b\xf2\xe3\xe1\x89\xbd\\\x9a\xf4\x1f_\xae\xbd\x98GAze\xbevz\n~\xa3V\xd1\xbd{;\x88\xcf}}_o\xdb\xd7\xb3m\xb7)_\x04\x96>\x81E\x1d\x92\x1dN\xc9\x15\xa5CAk\x97\xd2Cn\xd9\xe1c!\xeap\xff\xb10\x7fG]\x12\xb96\xd3\x9d\xf5%}\xb4\xb4B\x1c\x81\x915E\xff\xee\x8f\xfe\x93\x99(\xa4\x82\xed\x08\xe0\xed\x1e\x882@|\x1f\xe0\xba3u\xb0\xe3P\x94\x95jH\xbe8\x06\xda\xd5\xcela\xd8\xda\xb5\xeb\xc0N\xeb/\x15X\x9c\xfb\x1f\xb6>\xc9\xa4\xff"\x16<\xa0YX"\x9c\xfd!\'}\xd8SjB\xf5\xf1]\xc6v\xa0\xf8F\x86lL\xce\xf8\xe7.\x81x\x01jA\xf9\xf3\xea\x1e\xab\xc4\xac;W\x1f\xfcP\xbdy\x0f\xf8B}\xf2\x17\xb5\xb6\xff~ \x8c{\x02*`K\x7f\xc6\x13^\xc9{\xa0\x18\x82N\x0fk\x08\x01\x8d|\xd9\xb3\xfa\x1d\x00\x1e\x8ew\x19%(\xea\x18\xd6\xa0\x03rR\x08E\xa4\xe2\xe9i\xd05\xab\x17.T\xa7\x1e\x00G\xad^X\xac\xce\xfc\x02\x0b0\x01\x06\x19Q\x9b\x7f\x1a\xd1h\x19>\xd0\xa0\xe8\x0f\xda\x82\xde\xbe]\x9f\x13o\x89R\x10\xbc%\x8fzQ\xbf5\xe7!\x9ad,0\xe7\xfb\xf5\xf5\xd5\x11\x1ar\xf4\xc3\x91\xc88\xa3\x89\xb0\xa6P\x18\x00\t\xe3\x02\x83\x0b\x90\xed\xb9\x839\xdb\xf2`{2\x95,\xa0\x00Y7P\xd8\x89\t\x11\'\x0b\xdfW/\xbd4\xdeG\x086\x83N\xf91\xfc\xff\xa7\x16h2V/\x8dP=\xff\xad\xa1N\xca\x92Y\xa3\x81\xc5$7\xb7Y\x1f\xb7Y\x9f\x82\xb6\x9e\x08\xc6\xe2\xafa.^\\0n \xc6\xbba\xc3`G\x9c\x8c\x8b[\xc6\xe6P\xce\xc9\x15J\xc7\xac\xfeJ\x06\xd8\x84g\r\xd9p\x94\xfa\x1d\x07\xf41 T\x17\xb5\xd2\x02\xad\xb6T\xc9\xe7q\x0f\x01\x94#n\x1a$<PP\xfd\xc1t\xfd\xc1[P\x16\xd8\xfe\x90\x11\xeb\xf3\xdfWO=o<\xbc\xd0\xb85\x0b\xea\x0f\x1a)\xa4\x0c\xf8SOX7\xaa\xceM\x03\xb1FV\x171M\xbe\xd8\xd4\xb7m\xfb\xfe\x9e\xaf\xbbE\xd7O\xa4\xac\xad[-x\xba\xa3\xbb\xb7g_\xf7\x0ey\x0c\x18\xe0I\xf5\\\xf5\x853\xd5k\xbf\x00\x91\x83\x19\xd3\x98\x9c\xae^z\xc2\x96L\xed\xd1#k\xb3\x05\xd6\xa1\x7f\xf1\xa6\x7f\xf6\xa2\xb5\xbd\x90\x1b8v\xa0\'@\xce^0\xc3\xd08\t\x88\x14\x08\x1c\xec]Z}z\x08\xf8y>L\xe5\xc3\xa0=+\xa3T\xd4\xa0\x923\x88G\x14\x94{\xd8uPPg\xae\x85\x08y\xe5\x1c\xc0\xc5\xfa\x16ZX\x0b\xe7\xe0\x04\xb09\n\x12NA$\xe4X\x9f\x7f\xe5\x9f:\xf3N$\xa1\xa9\xf1e\xdf\xf6/\xf6\xf4l\xeff\x1c%\xf9\xd1\xcem_\xf5|\xf9om|>\xda\xa2g\x82\xf5\xc1\x968\xa7!\xdeg\xb86+\xb9\xc1\x82\xff\r$x\x8bA\x8cW\xbf\x9b\x14\x82\x90\x03\xf5\xeb\xeb\x19\x96\x00M\x07\x0bY\xd2\x9bS\xfe\x93\xab\xd5\xb9\x07&V@\xe41\xad\xc0\xee\x02\x93\x02d\xf0\xa6\xd6_|\x83\xdb\xd9\x85\xdb\x89\xba\x81\xdaZ\xda\xccT\x88\xd6q\xf3\x86\x0b\xa5\xc3\x03xDP=\xa5\x85+9\xa1\xf4UOdl\x88-\x0f\x93\xf5\xe2)J\xb7\xfe\xa4\x86\xd1\xa4\x0e+\xea\xb4X\x8bm\x9cx\xc3\xca\xa2\xb9~\x14#\x8f\xce\x00\xc0\x91m4QJ8F\xe9\xdcu\xc4NX\xee\x00\xdb\xea\xf0\x90\xbc\x0c\x96\x03Z\x05z\x1e\xf0[<\xefb]0b\x85}\x93h\x02<\x88\xe6\xba4\xcd\xac\\\x9f\x01\x85.`\x06\x01\x9a\x08\x9e\xaf\xed\xac\x9b\xb1\xcb\x11\xcb\'\xe3\x0e\x0cT<\xb2\xc9L9\x8a\x88\x0b\x1d\x00{\xd0FK\x83~l\x1c\x04D*\x90?(\xc7W\x8e\xb3h\x89\xc0Y\xfbi\xc5_\xbe\xaf5i\xd8h\x7f\x02\xc4\xcf\x19\x00\xdb\x1f;\xab\xad\x05\x12]\x11\xcc\xed\xe8Q\xb6\xc1V\xa0\xb5\x9c\x9b\x07\x95\xe6h\xdf\xd0\xa6\xbe\x11\x13\xa7\xa3}C\xc8\x16q\xb3\xfa\x06@\xab\x1e\xea\xeb\xdc\xdc\xbfQ\xc8\xb0}p\xb02\x90\xd0\x8cC\x89@\xdc\xc4\xd6\xa8\x0b\xa3M\xe4\x9f\x8dZ,P\x14S\xb5\xa2\xa9\xf2\x90]\x06\x0e\n\xaa\xed\xa0\x87\xcc\xd3\xce[\x8e]\x02VZ\n\x14\xf4\xf6\r\xbct\xe6\x112\x00C\x82\x1a\xfa\xabi\x91\x17\xa7\xa7Y[g\x14!N\xbay\x96\x08Z\xf6\xda\xe5\xa1\xa4:U;{\xbe\xecN\xb5#Y#\x96\x94\xe4;\x189t\x87\x0ev\x1e\x12$t\xfe]H\xd8v\xa4\xe0\x82\x1e?T\x18&/\x8b\xe0\x02\x14\xddJV\xd4Ds\xe5\x95t\xda\xf1\xbc\x81J\x16\x05\x89e\x0f\x94QK\xb3\x06l7\x0b\xa4\xa5\xd0\xc1\x9a\xa3h\x06w\x9fV/^f\x96\x19\x18.\xb7^\x03r\xea\x0f\xee6n/\xd6\xee.\xd7\xa7\xc4\t\xa3\xd0A\xd6\x16\x9b\x90\xa0\xbb\xec\x06\xebB\xf4q\xb6\x95XG<\n,\x1aA\xce\x16\xd2 \xbcE\x9eY\xe8\'\x05n\x9fU\x82N\x1d{d\xd4\xa0\xa4\x90\xd0\xc0\xf3\x0c \xfe\xf5f\xf5\xfc|\xfd\xa1\xf0&\x00\xa8\xfa\xf8\x8e\xbf\xb4\xd4\x18?\xbb\xfaf.\xb23%\xd0\xab\x8f%\xf1\xe4*\xf6\x0e2ju\xe9q\xf5\xd2\x0b\xc6/\x8c\x1f\x15S\x9b\xd68\xa4\x86tB\xb5\xb0M\\Jm\x81\xeb\x84\xbc\x1amtZ\x8b\x05\xcf-\xa3n\x9c\xd1\xfa9\x1fH\xd3\xbd\x05\\A9\x92\xc6M\x8f\x8a?s\x86=-\xd5\xef\x17\xfd\xd9\xfb\xd5\xb9\xd3\xd5\x8bS1\x8c\x8c4\x92\x88\xa74\x1b\xe30e\x91\xc3^\xaav\xaf\\r\x8bI~\xc4\x8a\xcb\x1f\xe1,\xe2\'t%\xb7\xbb\xde\x00\x1c\xed\xb2\x93\x14w\x8e\xbc\xc8\x1f\xf0U%o\xd0Y\xc9\xfb\xd4jI<:\xcaW\xf5s\xb6I\xd8\x01\xfc\xeb\xebk\xea1\xcb\xe5\xda\xf1e\xf8F9?\xb4q\xf4\x97\x8a\x0b\x1c0\xc0\xa7\x9bs\xf2\x1e9\x8cas\x06\xf3\xac&m\xeaj\xd3\xfe\x05\x1b\x99g\xd9\x15\r\xb1\x84\xce\x89\n\xedZ\xd6\xcd\xb9\xe8\xd9Z|\xd4x\xf8=\x9a\x1aj\xc2\xd5\xa5\x0b\xb5G\xf7A\x05\x85q\xc8\xcf:6\xad\xbcE\xd3\xfe\xcc}\xff\xee\xa5\xc6\x95Y\x7f\xeay\xfd\xc7;\xd5\x1b\xb3\xc0@A:6.\xbe\x80\x81\xaa\x8b\x17"\xfbB\x1e:\x8dRq\xd4\xa9\x8f\xfc\xe5?\xe14\xc0\x1fB\xaf\xe8\x87\x1a\xbf\xfc2\xa2O^\x89\xa0\t\xa0F\xd7\x91\t\xb2\x89\xb9\xedv6]\xc9"\xf1\xda`B\x02\x1d[\xf9J\xae\x1f\xce~a\x00\x949`\x13$|\x93\x9d\x9f\xe4\xad\x0f\xac\x8fR\xed\x1b\xc4e\xc6\x88\x807\xea\xf7g\xfc\xd3(H\r\xff\xd9\x94\xbc\xfe\xeb\xeb\xd3\xda\xe7\xb2\x0f\x84\x02\x9a\xf5 \x0e\x92]\xb0\x0f\x9d\x9fX\xbfgZJ;n6\xa9\x08\t\x1e\xa2\xcfk#\xcce}\x08/\xa5hZ\xedA*"[\xc7c\x96\xb7i\x9f\xbf\x02*\xfc\nln\x00$\x0f\xa7\r\xcd\xddJ\xa9\x84Z\xe2\xa6\x8f;:6\x12\xf8\xa4\xee\x83\xce.\x1b\xdb\x18\x7f\xe6/L\x89\xa6o\xfc\x1e\x8et\xe3\xd5\r\xa5\x00\x8f\xf9w\xaf\xd2\x18\n\x05\xe8j\xbd?\x03+\x8c\xec\xa4,\xec\x0f[\xe9m4_4"\xac\x11\xfer\xd4\xaa?\x9f@7\xa2X\x17j.\x90\xa8@1\xf4;f5\xb5\xa5\x85\xda\xcd\xc7\xcc1\xf9h\x07Lf\x17\xf3\x15\xdc\xa7\x01\xa0\xd6!\xd8\xd1\x8d\xfdn\x99=|\xe4\x81\x1a\x1e\x02\x05xc\'2\xca\x12\x9c\x07\x07\xf8\x0cj\xb93\x0f\x803\x88s\x1a\xbeE\x99\x0f\xec\xf2\xd5r\xed\xe1Y`j(\xb2d\xa4\xc6\xd5y\xe0\x9c\xec\x02\x8c\x04\x99<\']\x02\xe3#\xec\xc8C\xd7\x10\xcd\x0e\x92\x9c\x84\x86:Z\x85#Ni\xb8\xe4\x12\x0f\xa7\xbd*\x94\x9d\xfeB\xe1\xb0re\x03\x93[\x9c\xd1r\xa4\xfa\x14\x10s\x95U\x0b\x9e\x9c\xb5M8U\xf5\xfb\xa7\xfc\xc5\x87 &\xe4\xf7He\xe7o\xfa\xa0\xa3\xd0r\xb4[\xe5\xc0\xee>tN\x02u\t\xa0\xed%83(\xe2\x87\x93]\xbf\xff\xfd\xa6\xae\x14\xeam\xfc\xcaVD\x02)mn\xbe\x9c\xc4g\x11\x1e\x02\x82R\x1b\xcd\xb4<1\x93\xfb+n6\xa3\x96\xe4\xe6\x95\xe3H\xe9\xae\x80\xe9\xb9yA+\t\x1f\xff\xc5=\x906\x1cw\x00\x92B\xc3\x02\x0ci\xe6\xee\xb4\xcc\xda\xe2I\x7f.\xaayv %\xe9\x05\xfd\xc1"\xf0\x81A\xd3G>\xda\xab\xaf`\xe3^\xa3\x1d7\xf9\xb3\xb5\xb9\xeb\xb3\xcd\x9f}\xfcI\xd7g\x1f\xa1\x18\x9e\xba\x04\x98\x81\xa3\x0fk\x0c(\xa7\x174\x00\x02\xda\x065\x91\x04*\xae\x0c\xf4\x1e\x0b\xb1\x04j\x00\xee]\x1a\xf8a?j\x07 \xae2\x954\xba_\x80\xd8Jp\\H\x90\xfb\x8b\xf7\x1bW\xafI\xd0C\xf9\x89q\xc6K/WW\xde\xae\xbeD\xad\x08\xa4dmfE\xa8\x88\xb6\xb6\xf6l\xb6zC\x0b\xdcb\t1>\x900\x95\x08\x02\x06Mz\xb5\xe4Q\xe5\xef\xfa\x1c\xd1\r0V\xf2.\xd0\xb2\xb6\xbd\xc4\x8f]\x04}\xdc=\xaa\x9c]\x8eU\xf1\x0cc\x01\xe8\xeb\xd6Kt\x17\x93\xb7\x85\x95&\xde\x19\xa0)T\x97\xee"\xbf\xd2C\x8a\x8f\xff\xf4t\xed\xb9\xf6\xae\x88\xff\x1a\x8eP\x0e\xa8\x8a\xb47\xc3\xf1\x9dj\xc7/\x18\xce\xaf\xec\xc3\xa0\xb2\xd8`\x9b \x1a\xcd \x1aZ\xab|(U\x08\xc7\xc2\x08X%\x9f\x1e\xb2\xf3\x83t6\xa7\x7f\x06\x9421\xd7\xae-\x81F\x8e\xa4\xff\xf2&2o\xe3\x84\xac\xbe\xbd^\xbbx\x85\x1d\xac\xb0\x9a\xeam-\xe5a\xca>\xd6#\x81\xef\x8c\x18@\x8f\xf6\x892\xdd7\x82a\xe0v\xfcg3\x98\x91C\xce\xd1\x83[>=\xa4<%\xdb\xc57\xe8\x98<uS\xd7\xe7\xe2{C\x07(\x1c\x87\x9c\x9d\xcd\xc2\x87\xedY\xb7\x08R\xfb\xcfJ\x91FY\x80\x90\xc3\x91\x048\xe9g\xa2\xfc2)\x82\xec\xac\xdf\x1f\xaf\x9e\xb9\x88\x0c\xed\xe6_\x1bW\xee\xe0\xf9}2\x03\x8alTKFOf\x9f\x9b\x07\xe8=X\xcaH"\rSq\xf0`K+\xd5\xb9\x8d\xc4\x9f\xf1\xbfD\x19L\nx_\x19\x1dd\xab\x85=\x80`\xb3\x01\x94\xcaf\xc3\x98w\xa2y\x98\x8c\x83\xea&\x0c\xb4\xbf\xfb\x7f\xec\xef\x03\xf5\x1b=\xb1([\xe4\t\xaaT\xa3\xa2ag8\xd2\x94\x05-\x1aU\x0c8;\x7f\x16\x07\x7f\xc0\x91mk\xf3\xe7\x1f~\xfa\xb96?(\xde\xcf\x86\x9c\xe2\xce\xe2F\xde\xfc9\xd8\xab\x9fj,\n\xab\xae\x9e\xbf\x07\xa6\x85\xffx\xa9\xf6\xf6[\x0ce\x18\xf8\xd4\\z\x00\xd5\xb6\x7f\xe9\xde\xbe\xbfg\xcfn\xc2\xd1\x96\r\xac\x1d\xc0\x97\xf8\xdf\x7fu\x9c"G(\xd0\x8a\x0eC*AN\x05\x1e\xef<\xc5\x83q;\xda\xe9\xf7\xfe\x93I\xa0O\x10\xfa\xa8`\x1b\x00\xf8gn\x01l\x1a`q\xd1\xea\x184\xc2g\x82al3E?\x05\x84\xc4!<c\xe1\x05\x98L\xa0\x85/\x8f\x1c\xa7\xdb\xf6\xf6X\x83%\xbb8D\xd2\'\x12\\b\xd5\x1b\xdd\xb0\xaf\xee5\xb9\xf6\xe8\xe7,\x88\x859p\xc8\xfe\xc9\x14\xee\x85\xf2%(\x14\xf3\x1c@\x9b\xbc\xa0N\xa0\x06$S\xe0\xda}BwD\x1c_\xee\xd9\x86\xe4\xb6\x1bh\x04\x95mZ*|g,|\x94).\xd1\xd5<B\xe0+dD&\xcc!F\x12\x95\xbcSV\x87B\x99\xd9\xf0\x86xA`xx\xfe\'\xd2\x01{\xfbzv\x8f\xaa\x896\xc7L\xf4\xf5\xb6\xee\xd8)\x8e\xd8\x8e\x9a\xc1\xb4\xdd9\x90\x82_\x0e\x14;?n\xf7\xec\x01\xa7\x0c\xeau\xa1\xe4%\xf44\x1f\xfd\xf6i(R#\xd3l\xea\x8a\x9f\xe6\xe3\xe6i\x92\t\xd1\xe2\xbe\xd8\xb4O\xc5s\xf6\x17\xbeF\x98\x89\x07p\xb0?\xec\xaf\xd1\xbf\xe8\xc1\x88\x8bz;\x15\x02Ss\x06bG\xf0\xe4 l{\x9b\xd5\x81V9\x00I\x0f6\xab\x07\x12\xcb\xdf"\xf6S\xc0V$\xe8\xbe\xc5\x92\xa0\xbc\x8a\xb9o\x11}\x1d\x8d1\x8a[\xc3\x13Q\x17\xd5Z?\x89A\xe9\xe7\xb6\xe7\xa6w\x017\x8f"5\x98\x90\xa2\x11\x04]\x97\x82\x0e#\xeb.\x1e5\x90B\xf4\xcd\xc7\xf4\x8d\x9e\xe9\xd3\x98\x99\xfe\xb5\x17\rV\xa7\xc4\xd1\xc2\xc8\x0ez\xfc\x9d\xdeE\xa7\x02\x9f\x82]\xfal-\xe2&\x0f\xba\x17\x190\x06h\x8e\xfeoa\xf3S\x0f\xdd\xd9\x113\xf6>Rdv\x17\\\xcf\x89\x8c\x9b\xc7g}\x14\x9e\xde\xa2\xf5\xaa`\xb0\x98s\x9c\x90uo\xafx\xe5Bn[\xe6\x08\x06\xea3\xad\x90M\xe33mt(\xc0\x07y{\xf0\xe1\'\xfc,x_\x10G_~\xaaW\xca\x18\xc1g\x9f\xa9g\xa8~\xe5U\xb4YmYg\xb0e\x9d]\xf1\x07n\x07\xa6\xd58\xb1\xdb\xc53t\xb6 \xe2`\xe4Mk\x8d\xbcM\xc2\xa9\xeb\x1f\xfe\xa3\xc8\xf0q\x0ci{\xc9\x81\xf5\xf2)l\x81i\td\xd3\x1c\x9aF8\xb8K\xcf6\xa9g\x03D5;M\x9a\xf9(v\x9b\x8f\xac=!q=\x1e[\x1fs\xa5b\xf6\xb1\x02\n\xdf*\x15\x0c\xbf,\x94@\xbe\x12S+n\xa6\xc3 zBoE\xc7\x143N\xc6M\xdbh[\x00\xaf\xea\xfaz\x9b\xa5\x8f&\x08N\x16a\xf0\x16!\xd9\x02\x9cK\xd0B\x19\x1bF\x8c\x9a\x83\xd1\xa29\xc8X,\xbb\xaa?\\_}\xf5\x9c%\x1b\xe7\xb2\xb0\na\x8cZ\x9f\x9a\xd4al\x0eW\x87\x03\xd5Q\xae\x19Q\'H\xe7\xd5\xde{\xd2\xbd\x8d\xec@`\x85\x88:4\x9c\xca\xc3\x18\xa8\xa2\x1c\x02\xcc}q\xf3\x92\xe6a\x15A\xa9\x13\xdd\x82\x03\x9d\xbc\x90\xfa\xd8D\xf5\xfb\xf3\x98\xf773\xbb\xbar\x8d\x12^>\xb3\xfc\xd7\xcf\xcc\xa0{\xfd\xc5\xa2\xbf\xf2MT\xb7hN\xbc\tr#\xda\xac$f\x13\xb4Y\xe5\n\xd0i\x8a\x9dUdi\xc1n\x1a\xef\xa5\xf0\xd1g\xb0\x93\x80\xda>#\xb5B\xdc\x01\x04\xc9\xaf\xaf\xaf\x81\x86\xee\xcf,\x82NQ\xbd\xf0\x12\x9d\xc0d\xaf\xa3\xc2\xab\x92\x1f\x12at\x897:\xe7z\x1e\xee^\x01\xbd\xad\xc0\x0b=\x8az\xeb\xcd\xd5\xf1X\xd4\xc08/\xee\x98`\x0b\xec\xa6\x822W\x18i\xecof|\xf8SO\xb4-\xc3\x86f\xf5\xec\xb7\xb5g\x17\xd1-\xf5\xf8{\xb0J\x014\x8c\xbb\xfd\xf0\x10\xec\xfd\xea\xe9\x97`W\xf83\x97V_\xdd\xd19%-p\tj\x7f\x92\xb4o\xdc\xadT\xbb\xeb\xf5!\xf9\'9\x1f\x02\x9f\xa1\xeeh`\x8f\x11g\xa1B\xab\xa6\xd6S\x00\xdat\xbc\x8fc/qxB\xaf\xa38x"\xa9\x19\x9cd\xaab\x9d\xe5\xe1B\xa0\x11\xa6\x87\n\xa0\xb0+M5L\xd60U\xe3\xd6\x04&\xd8\x9cx\xe3\xdf~\xe8/>d=0\x88/\x87r7Z "\x92\xb7A\xfeTN\xce\xa0,\x8d\xa3\x89T\x94f\xf8E\x9e\x17\xddo\xd1Y\x12\xd1\x03\x85_\xf3f\xb3\xff\r\t@4s\xc9I\x94\xa5wn\xec\xb7=\x93n\x80\x9e2\xf4>#\xa0zk\xc9\x7f2-\xa9\x87V\x906\xf4b\xbe~\xe5&{\x8a\xcd\xd3\x0f_5n\x9d\xf2\x97gTHa\xca_\xb9auZ\xf5g/j\x0b\x97\xc5\x81g\x82*\x1e\xa1\x92\x13\xdd;;\x83\x04\xa3\x19\x81\xb5W\x92\x9e\xb2v?\xe8\\j1\xc8,6\xa2\x83\x85B\xd7\xbc4\xb5s\x94\xe6!\x80\xaf.=\xf6\'\xe6\x1b\'\xe6\xad?\xa8\x81v\xff\xd1\xaa\xce_\xa9=\xbd\x1b\x05)\x07\x82\x12\x19h\x06\xf5s\xf410\n\x99N\xc5\x9d\t\xd3\x97\x9c\xf6\x01\x17s\x9a\xb2\xc9RB\x8d\xfa\xef\xde\x07\xc9\x7f\xcf|\x90\xfa#l\xa5\xa8n\xf8b\xcf\xae\xdd{\xf6uo\xdf\xd6\xdb\x9d\x1a\r/T\xa2*(\x07\x90\xd9\xa9m\xe0\x03+[H\x11&\x8c>c\xb6\x9fb\x87\xbcJ\x8e\xa3\xa8U\xa2\x87b\xf6\x9b\xd0i16\xa6\x055\x06\xcb\xfd\x03:\xbc\xca\xc9\x12\xba\x13\x92\x9dmM\xdc\xec\x03\xab3\x85\xa4\x19\x10C\x08\x9b\x12\xe8\xe6\x14\xba\x17?1\xcf\xd5SGHt{!\x9fW\xceWq$\xd9Z\x9eT\xbc \x7f\' LC\xb4i32\xe2H2dRc\xec\x0e\n)\xe2 \x86\x04\x8b"\x81\x0c1\xd4\x86\x0e\x1dTR\x1b\xfe\xd2\xa6\x03\xd9\x92J\xe7\x88\xec\\ \x84\x0b\xe9\nb\x11S|q\xc36*\xb9\xea\x94\x87\n\x9c\x97\xef\x1cqJ\xc1\xee:\n\xfe\xe9\xea\xe91\xe0o\x01\x97\xe1\xc4\t\xe2)\xe8\xc4Y\xf9+z\xc7\x96\x16\xaa\x17V\x02\x89\xfan\xf0#\xb9_\xb8\x860+\t\xafd/\xb0D\xa7\xa4R\xaa(\x17\x8c\xceU\x06t\x92\xfc\x1a\xbc\x81\x1dI\x81\xa8U\xd9H\xcd\xdc \n6\xa2\x04}[G\xdb\x18c}\x8a\xfd;p\xbe\xc8/m\x12\x9e\xa8\x0b\xe6\x00DB\x98\x14\xc5\\\xa0(\xa1h>/"\xf8\xd2\x85\xe21\xe2{\xe5\x18\x89GR\xef\xeed\xfd\xce\x9b\x88\xbc\xf3\'\xa6$\x10\xc8\xd2\x90N\x12F=t<,\x0e\x16\xafP)\xa5\x1d\xe5\xdb\x0b\xd6\x94\x8a\x07=,\xc2\xdb,L\x98:\x86\x92\xdcL\xb1R\xc2Z%\xdaz\x01\xe4\x11\x89\xfc\xb7\xb1\xf1\xda_\x97kW\xde`\x02\x05\xa5\x9e\x989O\xbc\x90\xc0\xd3\x17\x07\x90\xf0\x02^\x86!\x9a)u\x93\x1f\x82\x1aTN\xa2\xab\x92\x88JG\x97\xd4\xb7\x15\x90\x9fG\xdb\xc1\x1e\xc3\xc2\x01\x96i\xedE0\x0f\x81_\xb4\xff\xb9\xa8\xfe\xeb\xf0\x1f\xc3N\x7f1A\xc5\x07\xb5\x93\xb7\x01\xadxt\xa3\xc9\x90\xe8\xc6\xe5\xc1G\x13\xad\xb0\xc8d\xab\xbd\xba\x18\xb3\xb1zv\x04\x950\x91t`\x03\x81\x8b3\xa6\x1a(\x0c\x84\\\xa2x\xe28vC2\xae\x15\xc6p\xae>\x17e\x04h\x19\xc9\xae\x0e\xe0\x8eD\xd1-v\xbc\x17K\x00\xc4\xb1\xab\x81\xaa\xe4)y1\xea\x94\x0e\xb6\x9c\xdd\xcf\xeb\x00\x9a\xbd\xd2\xc0&\x90\xe7\x85\xc7\xa9][h\x9c;\xd9j\x19\x19\x07\x83\xe3\x1c`\xdejm\xdf\xf3\xd5\xce\x7f\xb3>\x14\x03&\x01\x7f\r\xa0\xc2\xde7BKC\x01\xd0\xca\x03<\x12K\x07\xadv\x8e\xed3\t} \x90\x19\xb7\x04\x07\x02\x13\xd3\xc0V\x80\x03\x9b)8,\xee(zo\x1c\xd9\xfb\xe2\x07f\x1f[d\xa9\xf5\x9f\'\xab\x97\xae\xf8wW\xd6\xb1\xda\xf6\xa2\x8d\x81\xbd\xf6\xdca\x98;\xc9\x1f\xbc\xad\xfbK\x150\xbdh\xd2\xbe\xc2a\xfa\xd8bC\xb7\x03w\x89\x08(>\xb0:\xe3\xbc\x90q\x07\x84\x01Q\xda,\x1a\xe0\xc1J\x98y\x06\x9c\x7f\xe2G\x7f\xe1%\x16\n(\xa6\x84q\x84\x89{\xef\\\x12\x17i\xb5#\xb3\xebJ\xf2&\xb4\x99\xebl\xc9\x818s\x8d\xcfI\x00y\xd8;\x8a\xee\xb6\x1eV#\xc9FT\xbfn\x8c\xcd4\xc6\xcf\x8b\x0f4x\x89\xcf\x8diL\xb4\x82\x9a\xe5\x96\x9c\xa1C\xe2\xa47\xcdi=h\xc47\xa0\x1c\x18\xe6N\xe2)\x19\x1d\x8d_\'&o\xfe\x87S*\x88\x9a\x8bN\xfa\xc1\x12fnR\xc9\x9a\xf7\xcf\x9c\xf5\x1cQ.A\xfa\xa1\x95)\xbf!\xda\xd7B\x8fD\xff\xde\x1e1\x95Q\xbb\xed\x10\xed\x16\x99\x98\x1e\xbf\xfa\xe8\xe4\xea\x1b0\x9e\xae)\xd5\x8cTM\x1f\x04\x8a\xa1\x0f\xaf\x8d\x9d\x88T\x1f0\xf2\xef\xdb\x03\t\xcfGs\x94\x15\x15A(++\xa1\\\x92\x80JsN\xc6\xb5\xb9\xe0\x86S\xf3Y\xa5\xd5\xea\xb3U\xcc\xdaig\xa8\x90\x05\xde\x04"\xa7\xf1\xd7\xcb\xf5\xc7\xa7\xd5n\x8e\xab\xfc\x9bq\xb6N\x81\x1fqm\x8ei!\x10M\x07\xdc3\xde\x03`H\x9d8\xab\x99 S_p\xd2?}\xb1)\xf4\x05g\xf7\xf3\x17\xda\\\x0c\x8ck\r\xf5&\xab\xba\xf0L\x83N\x9f"\x05\x06\xf1\x06Z/\xca\xe4\x00\x02\x99\xa2>\xf9\xb3\xff\xe4\x1b\xe6C\x18h\x0e\xfb\x13@\x06c\xe6\xd2\xc23\xb3\xa2\x01\x08A\x10\x06\x96\xfe\x9b\xb9\xda\xf1\xe5\xc6\x95\x93\xfe\x99[\xac\x9aF\xe7oV\xbc\xa3\x98\xf8 \x0e\x05\x9d]\x02\xa0\xec\xc0/\xb7\xaa\xd7g\xfd\xd9\xa9\xfa\xfc\x94`\xa5\xb3\xcb\xf2\xc7\x82\xa0\x9f\x84Q\xecA\x0e\x93\xc9)\x80\xd3\x15\x01\x00\x86\xfdZ\xbcW\x11@\xc2\xf1\xae\xc46\xf1\x9fE\x81\xfb\xc0JFW\x00\x14a\xd6[\x90\x1f\xbd\x03\xac#\xa5\x1e\xd2Qlc:%k\x0f\xa0lw\xcbN\xceK\xa6\xb6\x04R\x98h\x17\xc1?H\x96\x9a\x18i\xcd\xf6\xd9?y\x81if\xfd\x93\x1a\xdd4\xd1R\x87\xa2\xda\x10\xfe\x94J\xa8\xf2\xf8\x0fC\xa2\x87\x97\x99I\x89\xf9\xc3\x08\x8d7\x8aV\x10\xd3\xbddh\xb0u\xb2t\xc6?\xfe3\x1c\x93\xc6\xe4\x0c*5\xf2\xcb\xd18\x8f\x85\x9di.\x9f\xa9\x90\x9bG\xa2\xa9J\xe0q\x85#\x9ew/HB\x02VV\xb2\xad\xa2[\xb4\x8av\xfa01\nV\xd4\xd9\xad\xa7\xb5\\\x8a\xb0J^\x88\xc9\xb2\x15\xc1\x82\x0e\tG:(\xcd\xa17\x91~_\x9d[C\x93W\nK\xb3\x1e/{\x1el[\xa1\xe4\x0e\xbaX+$\n\xb3\xfai*\xba\x05\xea\xc5\x88J\xaa\x1f\xc7\xaa\x9d\xe8.E-3W8\xa2\xb4\xcd\x1ck\x9b\xe6\x914\xfdY\x81\xd2,YbWG\xd4\x14\xa3\xef\xa3+\rm\xea\xd3eHq\x8aRg\x17hJ-\x80\x1f\xfd\xadjJ\xacF\xa0&\x8b\xe8\x04\xf1Zlg\xa0\xc6Z\xbf\xb7\xba\x82\xd7`^\x0c\t\x06\xefu\xc6\xbf\xb7.\xb1\x1e\xe3%\'Gx\xbcT\x0f\x0f\xad\x00\x89\x1b{\x97S\xa6\xa1\x81\xcc\x8b \xbd\x01O\xd18\xa1\xf2\xc0\x1bb\xb2i\x86X\xa9\xcb\'\xb1\xdd\xd8[S\xea*\xa0H\xec\xaa\xd1\xc2\xfcmK\x88O\xbek&\xa9Kk\x8f<h5kg\xc43\xf1%U\xcdG\n\xee,\x8c:z\xa0\xde\x95\n\x95AV\xf3\xd2T\xaeC_\x06\xea],\xaf`\xd9e\xf2\nm+\xd5\xc7\x7f\x82o\xf9`\x85m\xcc\xf7b\x11")\xfeo\xb1\x88a\x9bYC\xae\xb8\x89\xfe;\x00:\x0f\xfdQ\x18d\x0b5\xb7\xd96Y\x06#\xe0\xbf\x90e\xc8\x06\xff\xff\xc726\x1b\x96\xef{\xb1\x81\xb8\xc8\x9f\x8a\xbf\xbd\x93\x0f\xc4\x9e\x1d\xe3\xd4\xc4\x9c\x97&\xddx[\xb9\x8c\xe6\xb8m\x1dqJ\xee\x80K%\x82\x98!.NW\xca\xe4\xffb\x93\xb8\xf4\xcc\x90\xb7\x18\x0f\xed\x1b0\x8f\x85}E\xca\xdf\xca\xe9\xe2\xa2\x08\x9e\xbf\x87\xbe%\x18B>\x93\xae\xc7f\x9d\xd2\x8bU\xb9\x9f\xe9\x07k\xb34\xa8\xb4\xc0>ub\x0e\x12\x12\x92\x92t\x1d*;Nt~L\t\xd8ThL)\xa6\t\xd1\xcd\x92:+\xfb\xbd\x7f\x10\xd42\xe3\xfb\x9fP\xa2\xb7\xdd\xfcz6\xe6\xe5O\x9b_>\x14\x89\xfa\xed\xc1\x94&\x9b\xf7@\x8a\xb5K\x9c3\xa9}\xbfv\x9at\x9e\x9cfN\x98\xdfD\xc1G\xc9\xab%\x9f(+\xe2-\xa3O\x86\xe9\x81\x03mie\x0fK\xbaf>(\x18`B\xa0""\xf6(\x1953\xb9\x90=\xccT\xc0\xf5\xe4\x0cT\xc4\xef\x06/\x802\xa8\xbcu\xff\x10\x1fbo\xb9PT\xaeN\xa3J\x00\x96k\x1b:$A\x0f\x16\xbe\xb8\x1a\rO\xcaY1\xd15S\xc3\x94R bU\xa7\x89-9\xa6\x7f\xf6\xc7\xe7\xaa\x8f\x7fxO\xb7!\xf2\xcd\xea\xe9\x15\x1cy\xea\xc9\xfb\xf9\xf4\xb4[\x85=:\x12\xa5\xd2n\x1e\xda\x11\xed,\nV\x83\xa1\x00\x8e\xe66\xf9t\xf9\xc0\xfdV/\xd8\x88\x9c\xc9\x7f\xb8\xff\xabl\x97\x06\x9d\xb2\xf2\xf9\x91\xd5\xcd\x89~\\\xc6\xdc\xd2\xf5%\xcb]\x9f\x93\xe8\x1f\xe8\xf7\n\x05\x0bZ\xf9\xbe(%\x95]_M\x1e/\xca\xb8\x16\xe7:9\xbd\xb8\xe4\xa11\xf6S\xfd\xf6\xb9\xd5\xb7\x0b\xd5\x0b/\xff\xeb|]\nPIH\x95(\xa2\xce\xfa\x8b#(Vw8\xb1\x8f\xf3\x13\xf1u3\xa1\xef\xff\xa9\x83\xcb\x0c\xa4\x85\xa2 pt@\xf4\x98r+\xbc\xba\x17?\x99\x8e:\x0c\x97\xb1\xb0\x8a\x8a\xa9u;\xa8\x02a\x15/o\xf79\x1bKl\xe0f\x9d#\xd8[ S*\x14\xb1\x8e\x10S\x15\xd26:\xdd\xb8\xf2\x1d\x93+<\x8a\x86`*\x08\x90(\x0cYF+?g\x1fc\xd2r\xa4\xa2\x0c\x94\xdcR\xb9R\xa4dy\x8c7.\xdf\xc5ZL\xca\n`\x96\xbc\xbat\xb6z\xf64\xf6\xbc\xb8\xb4\xa0\x0b\x10\xb9\xb8\xcb_9\xc7\xb5\xc3R\x0cG\xc9\xee\xfe\xc2\xd5\xc6\xd8-*H\x9d\xc6\x14\xea\x85\x97JRc\x0ef_\xcf\xee\x9d\xd8\x1aB\xea!<*\xa3\x05\xde\xf0\xf9\xb6\xde\xee\xd1\x0f\x0b\xfd\x18\xd4\xe9s\xf3\x03\xa8\xd6\x94\xdd\x9c\x03Ga\xebg\x1d\xa9vl\xaa\x95L\xc5Tw\xe1\x02\x13\x9e\xd4\x03r\xe2A\xa8\xcaR"\xa3*\x99\x96\xb5\xf4\xd5\x95\xeb\xb5\x89\x1fk\x17\x9f#\x93\xd6=\x158dq\xf5$\xa7(pa\x80\x0c\x1c.oP-\xa2$\x1d\x99\x14b\xc9yH\xc6$\x9dJ\xfer\n\xf7\xe3\xe0!t\x05E\xd2su\xa8\x89g{W\xb1\xa9ZKa`\x80\xba{\xb1 e/cPE\x1bn\xc6\xe0O/aup(\xa7\xc8\\\x86.K[c\x1d\x94s\xbd\x8eUp\x14\xd3\xda#\xe0\xe9I\xd5\xa2PW<\xc6!\xf9\x16\x89\xc4\xdd\x98\xbchIf\x1e\xca^\xfb\x88\xedf\xed\xfe\xac.\xa2oJ\x1e\xe6\x9fH\xa6 v$3\xda\xcb\xc8B%\'2\xb4\xcc\x98\xa4\xcaP&\xa5\xb1\xdc`.\xa9\xfa`3/4qS\xe3\x00J\x17\xa9\xf4\xc3y\x1c\n\xb5\xad\xe8\r\xb5=\xa05\x16Q,`|\xa3\xc5\n\xcd>\x07\xb8\xa3\xc0}\xe2\n\xde\x9b\xd2:q\xbd\xfa\xe8a=\xf4\xebe\xff\xc97M\x03\xeb\xe0~"\\\x00\xaa\x9b\xd2\x90\xa2\xcb\xbec\xd0J-\xa9\xe4\xe2 \x00\xcb\xbdf\x1d\x8f\xf9\xa8\xea\x1367/.\x7fN\x80QM\x9eP\xc5\xbbr\x9c\x9b\xceaY"\x9fCR\xf7\x98\xa0\x1a\xf7/\xf9OV\xc4\xab\x16R\xfe\x95fJ&i\\%ejK\x8c\x7f\x90\xf8\x82\xa2\xa8`\x81!\x03Ej\x1ft=\x8bj\xe6@\x01R\n\xeb\xbb\\\x1eJ\x8a\x05uk\xe1rT\xcc\x93\xe1z3#g\x82\xb9\x8a\x7f\xfb\xa1h\x1b\xban-d\xe5#F\x83\xa2\x0b\xbdg\x07[$G\x1f\xe2\x1e\x04\xacbQ/:~\xa0\x16\xc4\xcf\xa2\xebg%\x1d\x97\xc4\x1bJ\xb3\xb6Y\xc5l\xc5c\t\x18l.\x95L\x00jw~\xb9\xb3\xebk\xd3\x86\x80\xfdS\xdeO\xe9\x85\xc5\x91\xf2G\x8f\xb8\xe2\x18}\xf5\xc6\x9e\x89\x91q\xfc\xa4\x14\xa0\x18\xb5\xc9Q\x1c8G\x8bL9X\xe2o\x9aV\x11c(\x94F\xce[\xcd\x0e\xf0d\xd2|\xaf-\xfcb6x-<x*\x15\x9b \x88\xf9C\xc0\xe4r`C\xb8H*\xe4\xee!\np\xf1\xb8c\xc6\nk\xbcMF\x84 \xea\xfe\xf1\xea\x8d9L\x00{{BU\x18\n\x93l\x8c]Y]\xbe\xa6b\\\x81\xd9\x80\x95[\x94B\x116\x1bb\x12\x01\xc1\xde\xceRG\xbc\x10\xdd\xe0\x11\xd4\xdf(l\x929\x11KD\x96p\x83\x11\xf5\xea\xa8\xa5OX\xa2\xb9\x12\x9cR9\xb3Y\xd8\x1f\xedE\x07\xad\x021 I\x17\x1e\xa5\x9ci\xa5v#\x87\xcaDW\xcc3\x1f\x90c\xcem\x19\xb4\x97l\xe2\x85\x7f\xfcJ\xa4\x85\x15\xfb\x8c\x82\x8e\x9b\xe4I\x0b\xb2\x90Z\x87\xc5\x94\x19\xdcGS\x92\xee\x18\xf2\x9b\xc68:G\x91R\x8c\xfeXD&\x98\xcc\x95R6p\xe0d\x0b|l\xe1\x89\xda\xedL&i\xb8f\xf8\xa7b8jX"\xc0m\xa4Y\xf4iO\x99\xfb\x8c\xf1\xfb\xd0\xcfq#\x15\xcax\xeb\x04e\x0f\xbem<\xbc\xa0R\x8c\xd1D\xc4\x04\x98d\xe8\xb7\xa9Q.jE\x85\xed\xd2\x93\xe6v.\x86\xb6a\x13\x89I\x1d\x8f*\xa0K\x1f\x03\xbd2\x7f\xd8S\x06\x97\xb6\xf2e_Y\x03\xe1|\xac\xa0\xf5\x07\xa8\xc5\xb5\x17\x98\xd3\xb4\xbar\xaa\xba<kr\xf7\xc0\x98W\x8e\x1c\xad\xf1\xf2!\xcb\x8bM\xa1\x83V\xa1\xb3\xd0}\xd4\xcea\x023\xa5\x84d\x1c0\x0e\xb8E$\x9b\x81%\xac=\xf7\xe8\xd8\xe2`\x1b\x03C\x8b\x16!\xb9\x99\\\xfd)^\xd9%j\xaa\xf1\x98s2\xfds\x13\xfe\xecCV\xea\xfd\x89\xe7\xfe\xc2\t\xcd\xca\x01\xe5\xa0>\xc69i\x8f\xd8\xd9\n\xc1\x8dS\x06\x8a};=\x0fV\x105\x06\xc8\x91S\x1e.lt\x98\xa9\x88\xfe\x8e\xc5\x89j\xb3y+\xd2l\xaa`\x19\xbda\x8aP\xc9Z\xe3\xda\x15\x7f\x82"\x96K\x13\xa0\xaacI\xf1\xccb\xfd\xc4\x9b\xd5Ww\xea\xb7\x03o4/\x827\xa5\x95m\x82\t$A\xd66\xc1\xdeF \xa5t\xe8\x98\x1eR\xb7\x9b.cM\xd1\x81\xb4\x93\xc5\xcd\x11#(;\xd9c\xec^\xc1=pT\xea\x93\xecv\x8c\x8bE\xdc,\x08\xee\xb3Y\xd8\x0831\xd3\xa4\'T\x1e\xc8\xd9"\xc5\xb1\x13/\xb0\x99\xd2\xb9\xc5\xfa\x83\xbb\xe6"\xe3\xe0S\x0e\x98r\x89\x17u\xb0\xe3PJS\x1e\xe5W\xd1\x0c\x96v_X\xb1\xd0\xa8\xb7\xd4\x18\xa3&\x03\xcd\xa1\x93\x80\xe9Tw\xd6a:\x8d1U\x84\xf4)\xa7G\xeb\x19\x9e2Z\xb4\x92\x11\xec\xa8\x9a\x9c\xb5\n\xe9\x1f\xb38\xe3\x8f\x9d\rNa\xb85\xec{\x1f5Q\x9b\\/Z\xd4\xaf\x80\x90\x8a~$\x0f;\x9dv\x8ae}\x12\xa9Y\x90\xa9&1\xe3\x17w\xc0\xc4x}\xfe\xa94M\x9d\xb8\xc7\xe20\xe8^f\x82 3\x98z\x11\x1f2\xc3)p\xa8Y%\n\xaf\xa3\'\x9f\xceV2\x8ef\xc1m\x81\xf6G\x99x\xc7\xb0Tw\xa3\xae\x97\x18r3\xc0\xf7\xa2\x85\x97\xa0\xf2\x80&\x87\t\x1c\xac\xd2MM\xd6~~U{\xf6\xb4q\xf5i\xfd\xf2\x8c\xac\x81R\xa5k\xbfL\xe3\xaahG\x9a$\xb9B\x13\xa7\x0c\xa7\xda+y\x00$\xf9\xfb$~\xe2\xc5\xd2b\x06K\x85J\x91\x96\xc2\x91y\xfa\xcc\n\xafZ\x05J4C\xe9K0\xd8\x89TT\xab\xe9>Z\xd4\xadX\x95\xd1\x85\xc9\x0e:\xe5\x86\xd5u\xa5\xd3S\xca\'\xbc\xec\x96\xc8_\x93)\x94\x11\\c?\x11\x17l\xa5\x84F\x11\xaf-\xe3\xe1\xe9\xc5\xc6\xa5\xb3X\x00\x1fd\xfc\xa02\x0c\xd4z\xe5f\xfcF\x03\x7fh\xdeX\x92\xef\xad\xab\n\xb7\xb4\xf4\xech\xd5%\xa4\xc0\x0f\xd9^\xd8\xfb\xb6Q\xfc\xe7\x01*p\xcf\xa3\xeei&]}\x04\xf5\xc9c\x95\xcet\xd3\x06\x03\xad\xed\x06RF\x9b\xd1\x18\x14\xf7V\x11G\xdb\x1a\x8b6\xdc\x06\xc1\xaf\xb9\xaa#F\xc2\xab\tI?\x04\xb9\x99=\xd6\xc65?DJ\x91\xde\x9ffz\x8a\x15\xe9\xf3i\x84\x1a\xd5wf\xc45\xf4\xeb\xd8\x14\x15l\x14\xba%\x8e\x05Sn>\x81\xa4a\x0c\xe3B\xb5J\xd3\\G\xaa\xac\x02D\x8c\xf0\xcf\x8c\xc4\x90(\xc2\x15\x07\xd0-bu\xee\x96"{f\xc3\xf0m\xc5S\x1d\xa7\x83\x06\x1e!\xa9\xcb\xceL\xe9\x9e0w\xb3\xfa\xf8\x8e&\xf0P:\x93R\xf2\x11\xce\xf3\xf7\xfc\x99\xcb\xab\xaf^UO\xceT\xe7\x1e\xd4n\xde\x0b\x17f7\xd1\x87,\xbe\xbdRD\xd5,\xf9\x8e\x14\xb6 RL\xf9;T\x86\x10\xc9DJ\xa5\xde1x\xcbH\xfdZ\x83\x0b\xb5\xaco\xf0w\x05\xe7\xd73Q\x84\xb4\xde5o\xcb\xc0\xe6Zs\t\x99G\xd9\xe8N\x17\x1bE\x05>\xd5\x92\x93+\x1c\xc1\x9e/%\xf8\x13\xd9#E\xdfl\xee\xd7\xa1i\x15[\xd6{\xda2\xac\x9e^\xc1\xec&\xd2\xd0\xb8\x0e\xacvz*\x10\xe9?\xbej\\\xb9\x8b!\xf2\x0b/\x91I\xb6*\xdf\xaf\xe4\x0f\xe7\x0b\xc3y\x11!a\x9d3\x05\x86\x85\xc2E\x04~\xdb\xcdR\x9b1\xd1\xc5"*\t\ty\x0b\x9b\xb1Q\x172\xc9\xa5\x97\xd2\xc0\x8b\xcf\xa3:\x05*A\xa7\xa6a\r&+\x14\x7f\x8e\x11\xff\xc2\xa2\xc7s\x93\xb8\xe6X\xa6o\xb0\x00Y\x14\x1c\xf7\xdf5\xa9^\xc9\x91f\xd10\x9a\xb2\xcc\\}\xe6\xb7\x86\r$\xe3\xa5F\x7f\xa7\xbaV\x82`\x8bi#\x8f\xa2\x11QANK1\xe8\xd1*\xce{\x18AU\xcd\xc2\xc8\x8d\x05"\r{\xde\x8c]\x86u\xeah\x1f\xab\x9e\xa1V\xbeS\x93\xdcs\xa619\x13\xed\xb9B\xdd\xf7\x82[\x02\xfe\x975B\xe5\xe8\xa3\xffyy\x84\xcb\xd0\xe9\x99jW$-\x8f\xe0\t\xd7^s\xcb2\xf8\x1cj\x17)\x9fc\xdaE\xc27\xda\xdf\x07\x1cz [\xf1\x86T\x88\x8a\x02\xf9\xa0w\x0f\xe6\x83<\xfa?9\xfd\xbd\xc4\x0e1\xe7\x05\xf5<\x17\x1b\xa8\xb8\xe5c\x1cN\x04\x12\t5\xa6\xc5\xd6Vw\x83>t\x14]\xd3\xe9-\xc6`\xfe\xc2\x0f\xd5\xa9\xe7\xb5\x87\x8buP\x9b\xa7\xee\xd5~\xd0\xbd\x9bx\x1aN\x8a\x88F\x07\x19\xc4\xfd%;}\xd8\x18K\x1cxn\x16\xa1\xf2\x1c<\x8fd^\x90\xc7N\x93\xb4t\xd1\x83]\x1b;\x0b3\xd6\xdf\xbe\xa9\xbf\xbckBD\x8e\xe2\xea\xd8\x8f\xd25\xec\xcd\xd3\xc6\xcd\xb7\x9ct(\xcd\x90B\xd7\x14\x0c{\xd1&y\x149\x12j\x01\xccU\x1cjHT\x00\xf0\xb0*MU\x98\xf2i\x03\xbdr\xb0\x04\xd0X\xb0\x97\x1e\xe7\x16"1\x8d\x8d\xc3\xb1\x90^\x17\xe4\xfaa\xebL\x07[\xaa\xafg\xaa\xe7O\xfbS\xd7\xabs7\x1bc?\xf9\xcb\xf7W\x97\xcfr\xce+\xf7j)\x1d\xdb\xd2\xa24\xac\x10\xad\xaeR\x1cF\x83\xc2\x1d\\\xc1\xfaq\xec\x9c\x14\xc4\x8d\xdd\x01)\x158fX\xc3\x0bPf\x82\xb0\xba\xf4\x13 (z\x98\tK\xfaR\x93\xf64\x85j\xfb\x02\xdb\x15\xc8\x7f\xd8\xdb\xf2\xe1\x87\x9d]\x9f\xb4w\xc0\xffun\x19\xd9\xbbg\xdf\xfe\xd1\x0f\x87\xbd\xff\xc6\x84\xd0\x93\xd9:\xa2Ib\xd4\x88\x05}\x14\xe1\xc5_b\x0b1\xe4\xdf.\x08\x92\n\xf5\x8d\xb33\x1e6S\x92F\x89t\xac\xd1\x85\x97u\xb8\x03\'\x965a\xd9\x13Ue\rV\xb2\xb83\xb2pj\x16\x87\xb63P\xf2\xe3\xd3\xfe\xca\x04\x1c\xef\xc6\xa5\xef\xd0=\xb7p\x15\x90\xcf\x8a<\x9b\x95\xbay(w_\xad\x9d\x1d\xabN\x9cmFD;u\x1a"\xd8\x93]\x00\xbbs\x149\xb2\x95\xdc\xd3\xdb]*\x15Jm\x06\x9a4\x86\xbb\xe9\x1d\x0c\xc66ok\xd9\xcdW\xccfWAK\xa3,r)\xbd\xadpB\x8dB(Q\xd7\xa9\xb9\x11\xfc\x8b\xcb\xa2MdB\xe3\xbe\x91\xdc\xe8H\x9f\x01\t\xe9Q\xe3,sY\x91*\xd9\xd2a\xa1 \x96\xb6\xa0=\xe1\xc4\x81s\x9d\xab\xe5\xf1\xba\x07\xca\x0b\xc9\xdaEP\xaa6"F\xac\x8c\xeb\x15\xb3\xb6\xe0\x1e\x84\x04\x9cC&*\xd6^\xb1j\xfa\xc2Kd\'\x97\x16V\x97\xc6\x18`\x7f\xea2\xb3\xd9\xda\xf3\xef\x1a\xe3\xe7y\x7fb\xc9\x8f\x0fi\xc4\xfa\xc9"\x1c\xc3C\xc7P\xc0!\x07\xc5Fix7\x00\x07\xde\xb3Yl\xb8\x85\xed9\x95[\xff\xc1\x03,~?\x0e\x87s\xac\xb1\xfcZ\xee\x16\x01Y3w\x02Nlmf\x85\xdb>\xd6\xae\xac\xf8S\xa7\xa4\xa5\xa09#\xf3\xfa\x04c\x9c\xdfU\xc7f\x9a\x19\x0f\xa8\xeb`D\xf1\x92\x1aW\xcfW\x17\x9e\xa15\x19^[\x1c\xbf\xeeu\xc4\x8e\xd3\x8cN\x8e:Ly\x04]x\x9c\x87\xa5\xads\xd5\x81\x1a&c\xaa\x05\xa6\xb3\xba|\x97s\xad\x826\xca\xa1.\xd3@G\xc5B\xdes\xcchm\xb1\xe0\x19\xe1Zi\xb1\xd2FW\x1em\x1d\tZ\xae\x88\xef$\xa1\x8f0<\x0b\x8esp\x9a;\xbb:t\xaf7*bv\xf3\xdc\xf2P-K\x97\xe0\x13\x0f\xe1\xae\xa9\xc0S\xb3\x0e\x17\xe4kr\xef\xb7\xb1-\xbd\xd1\xdfMj.\x88\xb2\x01\x93\xe8O\xbfv\xc3\xbf\xf5T2%\xb8\x17\xea\x83\xbb\xfe\x93\xf3\xc0\xfb\xd1 %\xf2\xaa\xfep\xdd\xf0+\xa32\xa2\x90\xd0^8\x1c=\x8cY\xbc\xaf\xc0\xc6\x96\xd7d(\x04\x0cR@.\xd9\xae\xa7\xba\xd3Q\xbeGi\xa3\x07b\xd4r\xf0\xe0\x0b\xd5\x13L\xccP\x84\xf5_\x9ek\\:\xcd:UP\x93\xab\xc7\x8e\xb1\x95\x87=\xd6\xd8\xcaD\xf2-s\xb3\x00e\xd2\x84]$z\xc0\x8e\x8d\xa2\xa4\xc9\xb3\xd5\x0b+!^\xbf\x96\x9f\x10\xd8[\x1a\xb1\x90L\xb5\xa8\x86\x15\xc2Jx\xdc\xc9\x92c%\x19d\x07Y\xcf,\x80\r(\x90*\x9e\xb5\xbe+1mf\xb8?O\xcaE\x16\xe6T\x88d\xc7\xda\x87m\xc7s\x0e\xb1\xd4\xa4\xde2\xacK:\xb8\xe5\xa3\x8e\x8e\x8eC\x9a\xcar\x0e\x15\x9f\x1bU\xfc=;PZ4\x89f\xe9G\xac\xf5\x1d\xb9\xe0\x01\x14\xc5\xf6\rR\xb5K\x07\t~\x1fdE<\x99\x14O1\xebAD\x8f\x01q\xbdxaD.yvVx4\xc4\x9c\xe5\xa0\xee!\xc2s\xa3\xaej\xc2,\r<6\x8a\xa0\xc4WH\x1aG\xc6\xfas\xa1\xbf}\x83(\xa5\xd4\rT\x8a\xc7\x97\xefF a\x95U\xc1@\xb9\x1f0?\xe2\xae=W\x00\x1a*\xe4\xdd\xb4J\xb3\x90\x0c+\xdbB*\xce\x1aM#\xcdS\xc7\x15o\xf0f\x96\x19*\t\x03\xdd\xc5\x9f\xd3\xac\x88\x7f\x07=#\xf5QC\xb9\xaa\xfa\xfc\x83\xf9\x8c\xe1^\xa3\x99.K\x01\x8d.\x9ck+\xdd\x84\x96,\x17\xcavv+)\xc2\x94\xbb\x94\xde\x9a\xa8\x9e\xff\xb6q\xf5\xcc\xaf\xafg8!\x87\x85\x170&Pg\xcb[\xa9oQ\x02\x8b\xf1\xed#\x8e\xc9E5\x9d\xda\xd6\x00%J\xc3FS\x05e\xc9I\x17J\\vn\xb4a6\xb4\x08\xd2 \xc8\x1a`Z\xa8\xce\x8d\xd5\x9e\x89)P\xfbe\xacq\xee$\x9a?\xa0X^z-Y4\x86\xae\xa0Y\x0c\xf72\x0c\xd4I\xeedPr\x8a\x88\xf8\x0c*\x8c\x85,5(Vp\x91\xa0\xc7\xed\x0fNh\xc9\xe1\xb0\xdb\x00\xf1hir\x88\xce\xca\xe0\x15\x969\x8d[\xafj/\xc7Q\xaa\x10\xadb\xf1!\xec\xc2\xcd{\xf5\xbf\xfe`\xc2\xad\x80\xa3 \xaa\xcc\xdbG\xcb\x05PAYS\x17\x88`]\xbbU.U\xcaC\x88\x1a\x8a\xb0x$G\xb99<e\xe5\xaa\xa3\xa4r\xdaX\xe4\n\xf5p\xee/\x1bxp\x06\xa5\xe9\xc1\xd4\x13\xa6\x04\xb9\xa4\x8dD"j\x02?}\x8b\xd5\xe3L\x18W\x8e\xaf\xbe\xa2\xe0\xc6\xca<\xd8\x11\x81\xe6Et2\x04l\x8e\x03~\xd8I*\x11I\x83A\xef(\xb6iJ0\x9dD\x9a\xcf\xe1\xb7\xd8\xce,!\x84\xcb\xa9\t\x14\xc6\x0f\xb5%\xe4.U\t\xa6\xd6\xd5\xa5\xc7\xf8;\xea\xfe\x94\x90\xc8%\xddwFO\xb1sS\x82\xc3\x98\xeaih\xa0\xcd\xf4\xab\xd9)\x9d\tA\xbf\xfa\x88\x9e\x9e\xfc\t\xa3\x0b\xfcT\xfa\x16\xfd\xc9v\xcbr\xc7B\xe0W\x90\xb8kk\x1a\xd6\x97\xcf\xb4o\xa8\x9e\x1b\xaf-\x9fc\xed6P\xf1A\x0c\x1d\xffY\xdf\x89BI\xd8H\xcf\xd2_;D\x17\xcd\xc6\x06\xb52\x84]\xc5c\x82\xc4\xca\xe0\xc9\xb9i\xe3t`\xe9@\xe2\xb0\xce\x8b \xb1\x06 \xce\x10\x02\x8a\xef\xe7\xa9]\xfb\x05\x88\x80\xefZ\xc2\xf6\xda\xd4|\x88%\x006\xf4\x19[\x11\xfe\x15R\xf8\x860E\x17\x0fv\x0b\xf1\xf7\x15\x08?\xb4\x05\x88O\r\xdb\xd9\xecF\x90[\xd8\xf9\xc0\xcdqL\x9b.\x8d\x1a\xb00a0\xc7z\x06\xe7\xd1\x05\xc4\xaa\xa3Q\xdcm\x99\x88\xd4T\xd4\xc4~|2\xc9\x1f\x01\xdc\xda\xc5\xe7rMJT?\x8c\x82\'jq\x0c\'\xc6\xe002\xe9VR=\xed\xb8\xc4\xbe\x82\xc3.\x87\x0e5\x03\xda\xf6\x96qz3\xaf\x94\xf3\x14$0CW\x0b\x89?U\x8f\x8b\x91cNr1\xcc\x87\xc0\x0c\x8d\x83NAb\xea\xe2\xcd+\x00\xc4\x1b2C\xb62VMA?\x1b\x06\xcb\xac!\xd8\xc5\xf2\xd0\xb1h\x00\xd4@\x82t\x97\'\xe5\x85{v2\xf0k\x03\xbc\x962\x15\x179\xe4\x1e\x9eL\xc8\x96\xa3\x8c6\xc9\x19\x82CX,Fo\xf01\x7f\x1d@\xcb\xb01i\xa3u1y\x9a\xbb\xdd3[~W\xfc28\x90k\xbd\x15&\x97\x88\xf5_\xa04PLx-\xd9y\xcfm\x86\x95\x0e\xe9\xf9{\xd5\x0b\xcf\x99\xe7\x89\x08\x9f\xbad\xb2iN\x05\x8d\x82\xdb\n\x98\x80:\x86\xd1-\x9b>\x92L\x85W!\x96r\x8c}\xbc\x9fQ\xae\xcd\xe4u.\x7f\x1bhK\xaeS\x96\xbbQUO/\x12\x18\xbc\xe9\xa5\x9c\x84\x01q\xff\x8d\x9b\x16\x9a0\xc1\xb7\xcd\xb0\xa61w\x13N;\x8b\'\xa6.\xff\xe9#&\xbf\xea\x89\x89\xea\xcf\xe3\xcc\x178\x08\xcf\xe6H+\x0c\xadk\x11\xda\x05\xa04\x10\xd4\xfd4\xcfW\x97DP\x92p\x88\xc5\x86\x02\xea\x06\xbd\xe1Ev/_\x03\xf7\x95\x1b\xadX\x00h\x8e\xbf\x1e\x90\x8a\xb6\xe7\xc5n\xdd\xfb99\xd6A\xbd\xd8\xc6\x1cm\xcb\xb4\x03:G\xd6P\x87\x98\x8c1\xaf9\xef\xa0\x11\x0f\x9a\x1a(\x01\x19\xd7\x13\xeea\x16\x01\x04\r\x18\xd8\x03r\xb51\xfe\x1d\x9f7D\xc7\xb5\x1b\x8d\xb11\xd4B\x08\x05\xf5\xd7\x0f@=2]\x83\xf1g\xf2\xbd\xb6\xd0\xb4\xc5\xd0v$?\xc9\x1aF\x99\x06\x98LC\xa2\xa1\xa9\xb0[\xb5\xd9J[k\xfe\xa8\xd5\xf6\xae\xf7w\xb8\x0c\xe2@\xa5\x04 \x97\xc4\xc3\xa6\x13#M,\xeb\xb2\xae\xe8\x18\xec\xde\xd1\xf7xh\x9b\xe8}!_S\x98P\x16\xcd\xbf\xf4\xee\xd9\xcd\x1d)\xc4g.\xb7\x8d\x06<\x0f\xccP\xfb\x9f\xf9\nagX\xd5M\xa1[\xb0\x1f\xf4\xa7R\xb3X\xa1\x11Y\x8f\x94\x13\x8eiA\x08=\x17\x8d`\xb2\xe3\xad\xd9\xfa\xfd7\x92\xd6z|yu\xf9\x1c\xbc\xb0\xfaf\x9a_X_\x9a\x8d0\xc46\xf4\xc5\xa6\xde!y\xf6\xda%\xf4=h\xe1\x1e\x90\x02\x9d\x84H\xfe\x0c\xdd~\x11\xf3n\xb34\x8c\x9b\x8b\xcf\xd6V\xf2\xf1\xb4#\xd7\xf4\x14\xa8\xa9\xb5\x81\xd4\xa9\xb6<B\xd1>F\x91\x15}\xdd\x87\xe7U\xc8\x9c\xd5\x19\x18\x87\x9dc^\xb4[M\xc0\xaf\x18Z\x85rl\xa2)F\xf7\xf8d}\xfe1\xa7bp\xa4\x15\x1bJ,\\m\xae\x14\x8b\x03\x12\x89\x01\x16\xc7h\xa3\xdc\x11|\xc2i\xdf#\xa3\xefZ %\x89j\xc2\x92\x1b|\ntJ\xe4@c\xc2F\x0b\x05\xc0\x7f\xfb\xa6v\xf1^\xe3z\xe8\xf6G}2\x02\xef?\x1b=/^\xac.\x1f\x7f\xd7j\x80\xa2\x10|^H\xe0D\xa0<-\xfdq\x9d\xfc\x96\xe3-\xa0\x8f\xa5\x87\xa8x\x1d\xf5\xe3\x90\xc7\x91\x9c"x\xf9vS\xc6\x16m\x1aEZ\x9a\x13&\x82\xdf\x87\xbb\xc9\xbd\x0b\x1cX\x9a\xb1I\x14\xf9\xa3e%4\x0c\t\x92\xdd\xc1\xfa)S)N\xba\xb4\x9a\x82\xb2\xd5\xd4\xd5\x0c\xe2\xd2@\x15\x056o\x88o\xa9\xc2T \x04\x82\x1d\x91\x81\xf3\xb8=v,\xf3\xa2\x06I\xbbS\xf6,^\xa5q\xe5\x1a^\x8f\xc9\xd1\x838\x0ba=\xe0\xa2[\x04C\x0c}\xe8\x06)\xb9t\x92\x92l\x03\x13\x120\xab\r1r\x90\xb1Au\x0cz7\xb0:\xbb\xe4\xd0\xdd%[\xf9.\xd6\xf5Q\x066\x04\xd2\xf9Cd\xd3+\xe7\xbf\x16\xc8\xaatC\x19\xa7\xf0,\x17\xa7R\xa22\x89\xb5\x0c\xe4\x07\xd5\xe6=]7\x12\xdc\xfeh\x9a\x1c\xbf\x89\\(iJ\x1d\xd7\x04\xf7\x08\xa7\xbf\xfb\xb03\xdb\xfb\x13\xcbv\x8cB\xbbie\xc0\xd0\x1d\xe5\x8c\x10\xe9\r\xc4e\x96\x9c\xc6B.G\xe6\x81\xf1\xe4\x02&8&\x91\x19\x96\x15\xd8\xc2\x98/27\xcf\\\x0fe\'\xf9G\x83\xfb\xbb\xdf\x8bV\x14\x9f\xdaJ\x87\xe4\xfd\x96\xda}\xb4\x8c9\xc0\xdaX\xe3\x8d\xc5#\xb0\x91\x12\xaa\xf4\xe0\xc8\xde1\xd8\xc3I\xc1\x16Wt\xc5/\x98\x93\x1e$"\xc2\xc54\xfc#\x90\xb4\xfe\xc4\x92t\xbf\x9cU9V\x7f\xd7\xa2[\xb2\x8d\xc8\xd6\xb7\xd8\xf9\xb5\x86\xd62NR\x89\x80\x1f*\xde\x96\xf0\x82\xcb\xc5\xc2m\xbf\x9a6]\xdf4\xfd\xf8\xae\xc1\x1a\xaf\x9e\xe4\xb3\xa1\xb2u[\x91\xfez`\x8c\xec{\xc0\x1c=\xb3h\x84\xba\x84\xeb|\xc9\xf5\x8e\xdf3\x98G\x9a\x0f\xec\xc9\x9cKI\x82\xfd\xc7t\xd3d\x90\x80\x19\xecSn$\x8a\xc7"BKCc\xe1\xa64\xc4\x1b#&\x82\x8b\xec\xfen\x96\t\x14\x81\\Q{v\x02Y\xd1f5?$\xf1\x99b\x92A\xaf\xdfo \x13\xc3m/\x9ed\xb9\x02\x95\xfc\xd9\x91\xdb\xe8\xd6\xa2\x18\xea\xcak\xfa\xeena\x06-^\x1aK\xc2\x04#\x88KoLW\xfa\xdfK9\x92\xf9\x9d\xb3\x8f\xba\xb9\n\xba\xaa\xc3\xf8\xa1\xafQ\x88\x84\x1fS\xfb\xe5\xbf\x0fO\xba\x02\x81\x1a\x9b\x00g%>\x8a9V\x12S\x0b:\x02\x8a\xe9\x11D\x90cq\xa5:\x91\x8a\xc3\x82\xd4C\xccV\xa1N\xa4xc\xdf\xd8k\x8cB\x9c\x9a\xe6hC\xed\xe2\xc3\xc6\xf5\x1f\xfe^t\xc5\xa6\xcd\'\xdd|\x19\x03\xb8\x05\xbb,m\xceM\x9d\x9f1\x1b\xf7\x96B\xfa\x1f\xad\x8e5\x88\xee] Q\xbaW!\x9b\x15\xa5F\xddu\xa0\x92\x1c\x85\xf4\xa4a\xa4;0\x80\xa8\xd6\x96\x9d\xbe\x1cl\xad\xd1\xa5\xc4),\xc9\xd1\\\x9e=\xc7\xc4\x87\xf8=\xf9\x00vBD:\xabC@\xa0k\xd3\xe5zVF\xda\x0f/a+]\x1e&HK\xfd6\x84\x1d(\xea\x9a[\x8c:\xa9\xdb\x9cu\x8c\x8b\x88\xb2\xcd\xcacGb\x10t\xe8g\x91>\x87\xe2?v\xd6F\xd9\xea\xca\xf5\xeawW%\xe2e\xc4\xba\x02V\xb74\xadZ\x02c\xb8\xaez~~\xf5\xf5\x93\x7f\x08\xb2\xf2r\xdf_\x07\x9cj7\x8fT\'\x95\x1cm\x062S\xbf\x11\x7f=\x94+6\xc0\xe9\xb5\xec_\xb3\xd3\xe9\n\xe7g\xad\x8d\x1a\xee\x04\x84\x9d\x1e\x81\xc3\xc39eo#i\xcd\xff\x90\xf5GU\xe5P\x0c\xa7\x95.\x1c\xdf1\x19\xef\x0b\xa0\xcb\xe5\xcbF\xc8e\xd0.e\xb2(\x08\xc1X\x8aOU3\x9a9\x80\r\xf0\xf0\x92\xe9\xb5\xe6[C\xc9y\x88\xe6\xdf\x897f\xda\x91\xe6\xf8\xad|t\x11\xd7\x82\n*l\xb4bBy\x7f\xdcjm\xde\xf2.\xeb\x96\xe2GtGU\xc1\xc3@c\xda\\\xeb_*\xd8\x8e[\'\xbae*\x18\xe2\xe16\x99\x92+\x12\x1a\x8d)|u\xe9\x0cX\xba\x1ct\xd4a\x13\x89GRx\x92\xef\x82\xe3\x8c8\xccc\x7f\xfaN\x9376N)k\x7fGa\x12\x9az\xb2\xaala\xd0\xe3\x90\x89\xa3\xaf\xb7\xcd`?#\xee/,7\xbec\x85\xb6\xd7\xe4\x12\x16\x0eH\xb93\xfe\xb5[\xfa.w\xff\xcew\xfe\x99k\xfa\xe2Z\x95\x1e:V\x7f\xfb\r\xea5\x14\xd2\xb2\xbe,\x0c\xae\xc3\xa8\xe7L\x14\xba\x16w\x9d\xd6\xbb\x0em\xcb\x029\x01\x85\xd7\x89!A\x9b\x9c\x8c\xe4\xd7\xcb\xb86hq\xc0\xb9\xd21Z\x87\x0eo\x03\xd4\xbcD&<\xd4\xccW\xce\xd5\x96\xcf5\xae\xdc\x85\xa5\xac\xd7,\x8bI\xd7H\x84\xb3\x8f\x14\xbe\xae\xfe{>a}`}\xb9gW;\xae\xbc\x0f]zI\x8a\x1fz[\x13`}`\x87\xd5D\xea\xe0\xc6\xcd\x1futl9\xb4>W\x14\x15\xd9S\x8a\x92\x17\x1b,\x8ds;\x85.XVi\x14\xf1\xe1\xd157\x91c\xfd\xf1m/\x04\x8a\x0fG\xb4\x8f\xc6Ly\xfcX\xb7\xbf\xa0_\xe9wR\xeb\xa8\xbbS\t\x04\x05\xcc\x15D\xe2\xa6h\x81\x1dD\x8b\xe3\x16.\x0c\xe6\x02\xe5\x05pb\x03\xad\x116\x9cs\xb9\xd9-\xbe\x0e\xba\xc5U\xaf\x93`\xbb\xb1\xf1f\xba,\x99\x04\xea\xbe8q\xee\x14\xf2\x1c:\x0e\xd2\xc1\xb26\xe7\xc2\xe8\xfc\xde&\xbddn\xde_\xb8\xe3\x7fs\x86C\xc9\x18\xd8\xe4\xb6\n\xda\x9eTw4K\x0c+\xb6X\xb0\x15\xb4\x02\xcdV\xeb`\x8eS\x1d\xd0\xc2\xc7\xd5\xb2\n\x8c6e%d^\xa9\\\x08xv\xf0\x10\x15/\xe4\x0ev\x1c2\x1dX\x85|\x1f\x8d\x9a8\xb4\xde0\x02Y\xe0\x9c\xc9\x04\xe8\xf2\xa4\x8cR\xe5\xbd\x92/\x06{\x17\x9a\xad%Zho7g\xf1\xaaW\xae\xc8S\xc9\xa0\xac\xc9Q>\xd3\xb9\xea\xc5_\xe2\x92\xb0\xd6\xe3\x84!D\xbd\xaf\x13\xaeR\x1a\xb0\x85`i\x80@\'\xe5\xee\x04\xb0D\xacDkiYk^\xcb}\x060\xf7\x9b\x96\x15X\x95-w{-\xb8b\xb8\x17\x9a\x92\xbc\xc6\x83\x1b;\xf1\x82\xf8T\x90q\xb6\xbex\x1e[\x82\xecD\xe0\xf8\x9db\x88\x8a9\xa5\xb9\x8a\xcd[\x83\xd89R\xa1\xcbG\xe8\xb8\xaa6"h\x13V\x81\xfc\x9f\xcdV\xaf\xdf\xe4\xcc\x8f\xf5n`?\xb0\xde\xc3-\xba\x82\xb3\x92LW\x0e\x97I\x8eh\'\x00V\xbdQZ\x98RA(\xcd\x82\x03\xab(\x81\xd0\xce!\x7f\x80\x11j\x81\xd3:K-!~\xb8\xae3}\xc5&\x0b\'Y\x88S\x9c2F\xaa\xbf\\\xae\x9dny!\x83V\xf80iu\xc0=\xda\x87{\x05<WA\xf5\xa1\xf5q\xc7\x96\xf6\xce\x81Q\xcb\x9f:\xd5\x98\xfd~\x9d\x1a\xe0>~Gn\xfa\x90\nx\xae\x18\xdcHR)\\"6\xf5\x02/\xaa?\x7fS\xafJ\xea\xc0\xe0\xe1\x0fp\xd8~lL\xce`\x8b\x83\xf1E\xe0Ak\xaeD`K\xb6\x00\x8bs\xc7\xcax\x93\xa1\xe6\xfa\xa2\xd0\x18\xc5\x1cA2\x85\x1b\xca\xd3\x0e`6\xc2C*I\x19\x8b\\\xe6\xa7Q\xf2\x11\xf1\x04*\xdb\xf8y\x7f\xf1T\xb30\x88\xcd\xa9XG>\xc5\xbfQ\xdb\x8a\xed{\x0f0\xc1p*\x88\xd1\x97\xafT\xc9\xeb[\x8bX\x9f\x89\n\xec\xefV_/\xa3\x1aqe\x9e\x13\xd79:\x8f\xe8&zY}s\x01\x9d\x8c:\x13\x11\xa6zgv\x05&\xe4xY\xc7)&;S1\xc9\xef\xa6\xb3[\xdc\x11(Y\xb9\xdd\x8fq\xa6\xa9\xdfJ%\x9b%\x01L\xe78hM\xc7\xa7\x17x,\xf2)#\x87Y$\xd7\xca9\xac\xe5\x7f2\xa9\xf3&\xb5y\x14\xa4\x13\x86\xb2\xce\x95\xa1\x17Xt\x91\x9a\t\xec\xb4\xcd\x90\x93\x9c\xc2\x9c}\xed\x91\xd2\x95\xbd\xe2f\xa3\xec:\xbe\xc3z\x8d\xe9\x9a\xec*~\xb7\xc5\x91\n\xaa\xca\x83\xa3\xc4\xeaH\xa8@@\x9c?\x92#\xc6\x06\x84\x91F\x89d\x89W\xfb\xbc\t\x92Zc\xe1\x8a\x1c\xffd\xab\x1c\xab\xd4z9\x82\x9aA\xc7\x8a\x18v\xcd\xff\xc8\xa3,%%J\x10\xa1\xba\xb8F\xa1H\xf4\x84\xd3\x88\xd9h\xa6\xdd\xb6\xec\xb0}\x8cR=Y6\x86K>\x86\xdd\xcc\xa0#\x17\x12\x18\x07]\xd1\x1f\x9cAJ\x13\xa62\xbe\x93\xb7\xe1t02\x99\xca0\x97GI~0<9\x11\\\xf3,\xbc\x9d\xa0U\xbaASQ\x08)\x9e\x14\x11j\xban>\x02\xab.\xfd\x00\x9dRJ\x96\xcd\x9b\xe5%E\x80P\x18\x8b\xb0\xd8\xe4s\xce\xcb\x0f\xf3=\xd2\x85\\\xe9X\xa2\x9a\x11\x9c\xbdg2\xbbWA\x0bOI\xb4X8\xb1\xfa\xea\x1e#\x02\xe3/\xcb\xb3\xff\x88\xe4{\xae\xcb\x90\xa2\xabJ\xbf>1\xd1\x1c|\xae\xcc0\x83A\xebO\xc5\xc7\xe7_\x16\xd2\xca\x87%\x1c\x14\xf8\x11w\xbc\x07\x0e\x1a\xb9\xbb]u\x03u\xa9"m\xe5F\xe4"v\xdd\x1b\x93\x8bVC\x16\x91\\\x08!m%\xf0\x8a%j\xac\xcd\xfdW(\xe0\xa6[\x81\xf2d\x89T{i0[\xe8O&~\xcf]\x9c\x8a\x91\xae\xd3E\xea\xd5\xd8NG\xd2C5"\xa9.\xf4LI2<\xe5\xdd\x11oEvK\xba\x07\xaf\x8c\xaf\xfb\xf4\xe8\x12ou9\xccb\xf5\xdb\xc7\x80O\xdd\x83\x9f\x9bN\xaf\x051?#\xc8\xd6\xec\x8dO=\xf1U\x93\xfc\x94\xee\xd08\x80]\x18\xcb\x056Q\x94\xca&\xfdH\x0b\x02(\xdeD\xe7\xd9G\x90\xdbr\xeaXs/4\xf4\x0e=\xb8\xebO\xdd\r1^\xa3\xcf\x1b\x81I\xed\xb5\r\x91\xc1o\xae\xbe9U\x9d{\x80\xbb5\xf5D]\x9equD\x9b\'\xbf\xe3\x9d\xf0~G\xf6\x89\xbay\xfe\x80P\xa8\xe1\xec\x01\x0c\x93\xc3\x1d\x1d!t_"\x1ehA\xb5\x8bJ\x03f]\xcc\\\xaa=\xba\x8f\x19\xccD\x16L1\\\xe3\xcb\x862j:!*\xe1\x9e/\\\xf8-\xeeHY\xcba\xe7\xd8\xd6\xac\x9d\xeb\xcf\xd8\xd4\x07`\x0b\xfdk\xdc\x9a\x95C\xd6\xad*\xa0\xec#A\x03\xc8\x8dXZM\xbd\xdc\x98\xa0a\xa3Hi\x1e\xac`a?\x86\xd58\xb0R(Q\xb0\x1e\xb3tA~\xdc\x98\xe3\x14\xeb\xfa\xe4\xcfq\xaf\x07\xfdnq\xbe=\x07\xf6\xef=\xb0\xbfoG\xcf>\xbc\nn\xcf\x9e\xfdH\xd2C\x98>\x8e\x11;\xbc,[\xb0\xda\xdc\xff/r\xca\x8ei6\xc4\x17\xadF\x0eY\xb8)R0\xed\xba\xda\xcd\x86\xaa\x1a\x8c\x80\x13#[R\xa1\xe8rX\xbc\xc7\x9b\xf8\x0e\x1f\x14\x93\x00#7\x02\xfa\xa7\x17\xf9n,\x05\x13\xaf\xa7\x0f]\xde\x80\x0b\x031\xd4\xe5\x97Z/\xcbC\xbc\x10\x1dw\xce\xc9\x8d\xf6I\x8d\xf2\x88\xb9\xfdr\xc0\x14\t\xee\x12NeTU\xc9\xadv:C\x1bS\x00\x87Kn\x99\xab\xbf\\8\xb3\x82v\xc0\xa68,\x1eh\xc8\x19l\xad\xaf\xd6\xef\x9f\xaa\x7fw\x16K\xc1NO\x87IRvh\xabe,\x8dA\xda\xcd\x97U\x06\xeeM\xd5D\xdf\xc3:&\xa7\x84\x97\xa5\xa3\xeb\x0c\xa55\xb5\xe8\xf1\x97g\xf0j\x1a\xf2c\xf2\xfc\xe1k|b\xda\rt\xb5Y\x9d`2F\xab=I\x8e\x92\xe3\x06\xd0\x11TT\x06\x17\x00\xa0\'\xaf\x92\xcf\x04\r\x05\xb0S\xf4bd\xf5\xdai\x13\xdbf\xa5,+ng\x92l\xd9\xa0*DS\xaa\xf7\x8f\xd4\x8a\x0b\xf2\xe8\xacF:\x15K\xfa\x0fY\x11R\x9a\x0f\xf2C\x13S\xdcL\x86\xc9\x19\xa8=\xd9@\x9a\xe51\xb5O\xf7\xec\x08\xdf\xf0\xc6\x8dGW\x97\xce\xb2\xcd/7\xac\x86\xa6Sc\xc6\xedx;J\x1b\x924\xa8\'\x9a_\x08\x05\xd3\xbem\xe9\xd8\x94\x19\r\x7f+D\xac\xea\xa6T\xff\xed \xca\xcb\xc7O\x02I\xd2\xb16\xca\x15\x88-I\xfd\x88\xea\xc0\xad/\x86\xd4\xc8\x0b3\xa5\x98n\xce|\xb0\xdad\xf8pu%\x87\xb1\xbcJ\xb6\xdc\x16\xb2\xcd\xdb8.\xa9\x8b\x8a\xd3v\x1ac_\xce\x90}\xc4\xc5RE\xd1b\xe5j\x8aq\xd6\xb7A\'\xe4Dh\xff\xedC\x14\x02\xe12\x1c\xe9r\xc0\xf2\x88nv xF\x7f}3\xc7?\xb7\xdeK\x11\x8f2}l\xad\xe1\x95\xcd\xe2,^\x97\xee@\xaaTrJ\xf2\xfc\x10[$S\xf2a\xda\xc9f\x99\xff\x93\xb4b\xddW\xca\x9f\xb4\x86\xc05\x88\xcf\xee\x9b}*9\xc3\x13\x8b\xbe\x970\xf3[\xad\x93Z\xf63\xb3\xd3\xb4\x14\xed\x9a(mnE\xf9\xf2\xd8\x91\x1f\\8\x04\x94\x9c\xc9\xe8K6M\x8d\x8d\n\xcd\x08`q\xe50w\xc6\xae\xd3H\nTa\x86\x90[\xdb\xe1%\x0bVT_~\xa8\x8b[\x19\xe0\xf0~$\xfe\xf7\xdcwV\xed\xd1#\xab\xb3\x13\xb3\xef\xf4\xe6\xe8\x81y<\xee!$\xdem|\xbb\x8br\xf5\x9a0\x00\xbb\xf2\x7f\x00\xd8\x95oe').decode('utf-8')
REMOTE_ALLOWED_TASKS = ['t2v', 'i2v', 'l2v', 'flf2v', 'r2v']
REMOTE_MAX_IMAGES = 9

'''
Prepare remote-only defaults for tasks not selected manually in the Notebook.
遠端 API 為尚未手動執行的 Task 建立預設參數，不影響 Notebook 既有設定。
'''
TASK_CONFIGS = globals().setdefault('TASK_CONFIGS', {})
for task_name in REMOTE_ALLOWED_TASKS:
    if task_name not in TASK_CONFIGS:
        defaults = dict(prompt='A cinematic landscape with ambient sound.',
                        width=384, height=224, seconds=10, fps=24, steps=6 if REMOTE_SERVICE_ID.startswith('06_HyperFlow_') else 20,
                        seed=42, quant=QUANT, output_name=f'{task_name}.mp4',
                        first_image='', last_image='', ref_images=[], ref_videos=[], ref_audios=[],
                        video_audio=False, ref_image_size='match')
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_'):
            defaults['audio_shift'] = 6.0
        else:
            defaults['encoder_variant'] = ENCODER_VARIANT
        TASK_CONFIGS[task_name] = defaults

REMOTE_ROOT.mkdir(parents=True, exist_ok=True)
REMOTE_JOBS = globals().setdefault('REMOTE_JOBS', {})
REMOTE_JOB_LOCK = globals().setdefault('REMOTE_JOB_LOCK', threading.Lock())
REMOTE_ENGINE_LOCK = globals().setdefault('GENERATION_LOCK', threading.Lock())
REMOTE_QUEUE = globals().setdefault('REMOTE_QUEUE', queue.Queue(maxsize=REMOTE_QUEUE_CAPACITY))
assert REMOTE_QUEUE.maxsize == REMOTE_QUEUE_CAPACITY, 'FIFO 容量已變更，請先重新啟動 Kernel 及遠端服務。'
REMOTE_UPLOAD_SLOTS = globals().setdefault('REMOTE_UPLOAD_SLOTS', threading.BoundedSemaphore(REMOTE_UPLOAD_CONCURRENCY))
REMOTE_RESULT_TTL = REMOTE_RESULT_TTL_HOURS * 3600
REMOTE_RESULT_BYTES = REMOTE_RESULT_LIMIT_GIB * 1024**3
REMOTE_MIN_FREE_BYTES = REMOTE_MIN_FREE_GIB * 1024**3

'''
Do not start a second GPU job when a long-running backend appears stalled.
當底層生成疑似卡住時，停止接收新工作，絕不啟動第二筆 GPU 推論。
'''
REMOTE_STALL_SECONDS = REMOTE_STALL_HOURS * 3600

'''
Verify the CUDA backend and existing model without loading another model.
確認 CUDA 與已啟動的模型後端，不另外載入第二份模型。
'''
def remote_gpu_status():
    try:
        yue = REMOTE_SERVICE_ID.startswith('06_YuE2_')
        if yue:
            gpu_id = globals().get('GPU_ID', 0)
            process = globals().get('SERVER_PROCS', {}).get(gpu_id)
            selected = globals().get('YUe_DEVICE_NAME', globals().get('YUE_DEVICE_NAME'))
            if process is None or process.poll() is not None:
                return False, f'YuE2 Vulkan{gpu_id} 常駐服務尚未啟動。', []
            if not selected or selected not in [str(part) for part in process.args]:
                return False, 'YuE2 Vulkan 啟動裝置與目前設定不一致。', []
            return True, f'YuE2 Vulkan 後端已就緒（{selected}）。', [selected]

        import torch
        if not torch.cuda.is_available():
            return False, 'PyTorch ROCm／CUDA GPU 不可用，請先執行模型初始化 Cell。', []
        selected_gpus = sorted(set(globals().get('MODEL_GPU_IDS', [0]) + [globals().get('AUX_GPU_ID', 0)]))
        available = torch.cuda.device_count()
        if any(type(i) is not int or i < 0 or i >= available for i in selected_gpus):
            return False, f'指定的 GPU {selected_gpus} 超出 PyTorch 可見範圍（{available} 張）。', []
        devices = [f'cuda:{i} {torch.cuda.get_device_name(i)}' for i in selected_gpus]
        if REMOTE_KIND == 'image_multi':
            state = globals().get('QWEN21_STATE', {})
            if state.get('pipeline') is None or state.get('manager') is None:
                return False, 'Qwen Pipeline／記憶體管理器尚未初始化。', devices
            if state.get('device') != f"cuda:{globals().get('MODEL_GPU_ID', 0)}":
                return False, 'Qwen Transformer 的 GPU 配置不一致，請重新載入。', devices
        elif REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
            pipeline = globals().get('pipe')
            if pipeline is None or pipeline.lazy_load or pipeline._mula is None or pipeline._codec is None:
                return False, 'HeartMuLa Pipeline 尚未完整載入。', devices
            for name, module, gpu_id in (
                ('HeartMuLa', pipeline.mula, globals().get('MODEL_GPU_ID', 0)),
                ('HeartCodec', pipeline.codec, globals().get('AUX_GPU_ID', 0)),
            ):
                parameter = next(module.parameters(), None)
                if parameter is None or parameter.device != torch.device(f'cuda:{gpu_id}'):
                    return False, f'{name} 並未載入指定的 cuda:{gpu_id}。', devices
        else:
            base = globals().get('BASE')
            if not base and REMOTE_IS_LTX:
                base = f"http://127.0.0.1:{globals().get('COMFY_PORT', 8188)}"
            if not isinstance(base, str) or not base:
                return False, 'ComfyUI 尚未啟動。', devices
            with urllib.request.urlopen(base.rstrip('/') + '/system_stats', timeout=4) as response:
                stats = json.load(response)
            comfy_devices = stats.get('devices', [])
            if not any('cuda' in str(item.get('type', '')).lower() or
                       'cuda' in str(item.get('name', '')).lower() or
                       'rocm' in str(item.get('name', '')).lower()
                       for item in comfy_devices if isinstance(item, dict)):
                return False, 'ComfyUI 未回報 GPU 後端，已拒絕 CPU-only 推論。', devices
        return True, 'GPU 後端已就緒（不表示所有權重常駐 GPU）。', devices
    except Exception as error:
        return False, f'GPU 檢查失敗：{type(error).__name__}: {error}', []

'''
Verify every global required by the selected Notebook generation engine.
驗證所選模型的生成程式所需變數與設定已完成初始化。
'''
def remote_inference_ready(task):
    namespace = globals()
    common = {
        '06_HeartMuLa_': ('ROOT', 'pipe'),
        '06_YuE2_': ('ROOT', 'GPU_ID', 'PORTS', 'SERVER_PROCS'),
        '06_Z-Image-Turbo_': ('ROOT', 'BASE', 'LOG_PATH'),
        '06_Qwen_Image_': ('MODEL_MODE', 'GPU0_MODEL_GIB', 'GPU_ID', 'QWEN21_STATE'),
        '06_LTX-2.5_': ('ROOT', 'ENGINE_PATH', 'COMFY_PORT', 'COMFY_LOG', 'COMFY_PROC',
                          'ltx_available_ram_gib', 'LTX_MIN_AVAILABLE_RAM_GIB', 'importlib'),
        '06_MiniMax_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                              'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                              'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                              'QUANT_SIZES_GIB', 'ready', 'choices', 'check_choice'),
        '06_HyperFlow_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                                'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                                'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                                'QUANT_SIZES_GB', 'ready', 'choices', 'ENCODERS',
                                'ENCODER_FILE', 'MODEL_FAMILY', 'WEIGHTS_IN',
                                'TEXT_LOADER_NODE', 'PROJECTION_FILE'),
    }
    required = next((names for prefix, names in common.items()
                     if REMOTE_SERVICE_ID.startswith(prefix)), ())
    missing = [name for name in required if name not in namespace or namespace[name] is None]
    if missing:
        return False, 'Notebook 尚未完成初始化，請先執行對應的模型／服務 Cells；缺少：' + ', '.join(missing)
    if task not in REMOTE_ALLOWED_TASKS:
        return False, '此模型不支援生成模式：' + task
    if REMOTE_SERVICE_ID.startswith(('06_MiniMax_H3_', '06_HyperFlow_H3_')):
        config = namespace['TASK_CONFIGS'].get(task)
        if not isinstance(config, dict):
            return False, f'{task.upper()} 尚未設定；請先執行 Notebook 對應的模式設定 Cell。'
        if namespace['H3_ACTIVE_CONFIG'] != namespace['H3_DESIRED_CONFIG']:
            return False, 'GPU／Offload 設定與已啟動 ComfyUI 不一致，請重新啟動模型服務 Cell。'
        quant = config.get('quant')
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            variant = 'ref2va_pruned' if task == 'r2v' else 'fl2va_pruned'
            expected = (variant, quant)
            expected_file = f'minimax_h3_{variant}-{quant}.gguf'
        else:
            variant = 'ref2va' if task == 'r2v' else 'fl2va'
            encoder = config.get('encoder_variant')
            expected = (variant, quant, encoder)
            expected_file = f'minimax_h3_{variant}_hyperflow_flash_14b-{quant}.gguf'
            if encoder not in namespace['ENCODERS']:
                return False, f'{task.upper()} 的 Encoder 設定無效，請重新執行模式設定 Cell。'
            if Path(namespace['ENCODER_FILE']).name != namespace['ENCODERS'][encoder][1]:
                return False, 'GPU 1 Encoder 與此模式不一致，請重新執行模型下載／啟動 Cells。'
        if namespace['H3_MODEL_CHOICE'] != expected or namespace['H3_ACTIVE_MODEL'] != expected:
            return False, f'{task.upper()} 尚未載入相應模型權重；請設定模式並重新執行模型下載／啟動 Cells。'
        if namespace['DIT_NAME'] != expected_file:
            return False, 'ComfyUI 使用的模型檔名與任務不一致，請重新載入正確模型。'
    elif REMOTE_SERVICE_ID.startswith('06_Qwen_Image_'):
        state = namespace['QWEN21_STATE']
        if state.get('mode') != namespace['MODEL_MODE']:
            return False, 'Qwen MODEL_MODE 與已初始化的模型模式不同，請重新執行模型初始化 Cell。'
        if state.get('gpu0_model_gib') != namespace['GPU0_MODEL_GIB']:
            return False, 'Qwen GPU0_MODEL_GIB 與目前載入設定不同，請重新初始化 GPU 記憶體管理器。'
    elif REMOTE_IS_LTX:
        if not Path(namespace['ENGINE_PATH']).is_file():
            return False, 'LTX 模型引擎檔案不存在；請先執行模型與 Workflow Cell。'
        if namespace['COMFY_PROC'].poll() is not None:
            return False, 'LTX ComfyUI 行程已停止，請重新執行啟動 ComfyUI Cell。'
    return True, '模型初始化與所選模式已就緒。'

'''
Validate client parameters before they occupy a FIFO slot.
在進入 FIFO 前檢查 client 參數，避免排隊後才因設定錯誤而失敗。
'''
def remote_validate_fields(task, fields):
    required = {'task'}
    if REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
        required.update(('lyrics', 'tags', 'duration', 'seed'))
    elif REMOTE_SERVICE_ID.startswith('06_YuE2_'):
        required.update(('style', 'lyrics', 'duration', 'steps', 'lm_seed', 'seed', 'cot'))
    elif REMOTE_KIND == 'image_text':
        required.update(('prompt', 'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_KIND == 'image_multi':
        required.update(('prompt', 'negative_prompt', 'sampler', 'reference_size',
                         'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_IS_LTX:
        required.update(('prompt', 'width', 'height', 'seconds', 'seed', 'image_strength'))
    else:
        required.update(('prompt', 'width', 'height', 'seconds', 'fps', 'steps', 'seed', 'ref_image_size'))
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            required.add('audio_shift')
    missing = required - fields.keys()
    if missing:
        raise ValueError('請求缺少必要參數：' + ', '.join(sorted(missing)))
    integer_fields = ('seed', 'lm_seed', 'width', 'height', 'fps', 'steps', 'duration', 'reference_size')
    float_fields = ('seconds', 'cfg', 'image_strength', 'audio_shift')
    numbers = {}
    for key in integer_fields:
        if key in fields:
            raw = fields[key]
            if isinstance(raw, bool) or str(raw).strip() != str(int(raw)):
                raise ValueError(f'{key} 必須是整數')
            numbers[key] = int(raw)
    for key in float_fields:
        if key in fields:
            numbers[key] = float(fields[key])
            if not math.isfinite(numbers[key]):
                raise ValueError(f'{key} 必須是有限數值')
    for key in ('seed', 'lm_seed'):
        if key in numbers and not -1 <= numbers[key] < 2**32:
            raise ValueError(f'{key} 必須介於 -1 至 4294967295')
    for key in ('width', 'height'):
        if key in numbers:
            size = numbers[key]
            divisor = 64 if REMOTE_KIND == 'image_text' else 32
            if size < 1 or size > 4096 or size % divisor:
                raise ValueError(f'{key} 必須介於 1～4096 且為 {divisor} 的倍數')
    for key in ('seconds', 'duration'):
        if key in numbers and not 0 < numbers[key] <= 3600:
            raise ValueError(f'{key} 必須大於 0 且不超過 3600')
    if 'fps' in numbers and not 1 <= numbers['fps'] <= 60:
        raise ValueError('fps 必須介於 1～60')
    if 'steps' in numbers and not 1 <= numbers['steps'] <= 200:
        raise ValueError('steps 必須介於 1～200')
    if REMOTE_SERVICE_ID.startswith('06_HyperFlow_H3_') and numbers.get('steps') not in (4, 6, 8):
        raise ValueError('HyperFlow steps 只支援 4、6、8')
    if 'image_strength' in numbers and not 0 <= numbers['image_strength'] <= 1:
        raise ValueError('image_strength 必須介於 0～1')
    if 'reference_size' in numbers and not 64 <= numbers['reference_size'] <= 2048:
        raise ValueError('reference_size 必須介於 64～2048')
    if 'audio_shift' in numbers and not 0.01 <= numbers['audio_shift'] <= 100:
        raise ValueError('audio_shift 必須介於 0.01～100')
    if fields.get('ref_image_size', 'match') not in ('match', 'max'):
        raise ValueError('ref_image_size 只支援 match 或 max')
    if REMOTE_KIND in ('video', 'image_text', 'image_multi') and not str(fields.get('prompt', '')).strip():
        raise ValueError('PROMPT 不可為空白')
    if task == 'r2v':
        image_count = int(fields.get('_image_count', 0))
        video_count = int(fields.get('_video_count', 0))
        audio_count = int(fields.get('_audio_count', 0))
        if not (image_count or video_count):
            raise ValueError('R2V 至少需要參考圖片或影片，不能只提供音訊。')
        if image_count > 9 or video_count > 3 or audio_count > 3 or image_count + video_count + audio_count > 12:
            raise ValueError('R2V 最多 9 張圖片、3 段影片、3 段音訊，合計最多 12 個素材。')
        count_by_type = {'Picture': image_count, 'Video': video_count,
                         'Audio': audio_count + (video_count if str(fields.get('video_audio', 'false')).lower() == 'true' else 0)}
        for media, count in count_by_type.items():
            matches = [int(number) for number in re.findall(r'<%s\s+(\d+)>' % media, str(fields['prompt']), re.I)]
            if any(number < 1 or number > count for number in matches):
                raise ValueError(f'R2V Prompt 的 <{media} N> 編號超出已上傳的參考素材數量。')

'''
Report whether the model is ready for CUDA inference.
回報模型是否已具備 CUDA 推論條件。
'''
def remote_backend_ready():
    return remote_gpu_status()[0]

'''
A Python thread cannot safely terminate an already running CUDA inference.
Python thread 無法安全強制終止正在執行的 CUDA 推論。
'''
def remote_stalled():
    with REMOTE_JOB_LOCK:
        return any(record.get('status') == 'running' and
                   time.monotonic() - record.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
                   for record in REMOTE_JOBS.values())

'''
Remove obsolete input files once the model no longer needs them.
模型不再需要輸入檔案時立即移除，避免參考圖片長期佔用空間。
'''
def remote_remove_inputs(job_id):
    folder = REMOTE_ROOT / job_id
    for file in folder.glob('*'):
        if file.name.startswith('input.') or (file.stem.isdecimal() and file.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}):
            file.unlink(missing_ok=True)

'''
Keep only finished results; never delete active jobs or in-progress downloads.
僅清理已結束工作，絕不刪除排隊、推論或正在下載的檔案。
'''
def remote_cleanup():
    now = time.time()
    with REMOTE_JOB_LOCK:
        finished = [(job_id, record) for job_id, record in REMOTE_JOBS.items()
                    if record.get('status') in ('done', 'error', 'expired') and
                    not record.get('_downloads')]
        finished.sort(key=lambda item: item[1].get('finished_at', now))
        sizes = {}
        for job_id, record in finished:
            folder = REMOTE_ROOT / job_id
            sizes[job_id] = sum(path.stat().st_size for path in folder.glob('**/*') if path.is_file()) if folder.exists() else 0
        total = sum(sizes.values())
        for job_id, record in finished:
            old = now - record.get('finished_at', now) > REMOTE_RESULT_TTL
            if old or total > REMOTE_RESULT_BYTES:
                shutil.rmtree(REMOTE_ROOT / job_id, ignore_errors=True)
                total -= sizes[job_id]
                record.update(status='expired', error='結果已過期或因儲存空間不足而清除，請重新提交。')
                record.pop('_path', None)
                record.pop('_last_frame', None)
            if record['status'] == 'expired' and now - record.get('finished_at', now) > 2 * REMOTE_RESULT_TTL:
                REMOTE_JOBS.pop(job_id, None)

'''
Sweep completed records periodically without interfering with inference.
定期整理已完成資料，不干擾單一 GPU worker 的推論。
'''
def remote_maintenance():
    while True:
        try:
            remote_cleanup()
        except Exception as error:
            print('API cleanup warning:', type(error).__name__, str(error)[:200])
        time.sleep(300)

'''
Serve only the generation API without exposing notebook controls.
只提供生成 API，不公開 Notebook 的管理功能。
'''
class RemoteHandler(BaseHTTPRequestHandler):
    def log_message(self, fmt, *args):
        return

    def send_json(self, code, payload, retry_after=None):
        data = json.dumps(payload, ensure_ascii=False).encode('utf-8')
        self.send_response(code)
        self.send_header('Content-Type', 'application/json; charset=utf-8')
        self.send_header('Content-Length', str(len(data)))
        if retry_after is not None:
            self.send_header('Retry-After', str(retry_after))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        path = urlsplit(self.path).path
        if path == '/health':
            with REMOTE_JOB_LOCK:
                running = sum(record.get('status') == 'running' for record in REMOTE_JOBS.values())
            gpu_ready, gpu_message, gpu_devices = remote_gpu_status()
            requested_task = parse_qs(urlsplit(self.path).query).get('task', [REMOTE_ALLOWED_TASKS[0]])[-1]
            inference_ready, inference_message = remote_inference_ready(requested_task)
            if gpu_ready and not inference_ready:
                gpu_message = inference_message
            self.send_json(200, {'ok': True, 'api': 'notebook-generation-bridge-v3',
                                 'service_id': REMOTE_SERVICE_ID, 'task': requested_task,
                                 'backend_ready': gpu_ready and inference_ready, 'gpu_ready': gpu_ready,
                                 'gpu_message': gpu_message, 'gpu_devices': gpu_devices,
                                 'stalled': remote_stalled(), 'running': running,
                                 'waiting': REMOTE_QUEUE.qsize(), 'queue_capacity': REMOTE_QUEUE.maxsize})
            return
        if path == '/schema':
            self.send_json(200, {'kind': REMOTE_KIND, 'tasks': REMOTE_ALLOWED_TASKS,
                                 'fields': REMOTE_FIELDS, 'image_limit': REMOTE_MAX_IMAGES, 'video_limit': 3, 'audio_limit': 3,
                                 'upload_field': 'images' if REMOTE_MAX_IMAGES else None,
                                 'queue_capacity': REMOTE_QUEUE.maxsize,
                                 'model_switching': 'configure in notebook first'})
            return
        pieces = path.strip('/').split('/')
        if pieces[0] != 'jobs' or len(pieces) not in (2, 3):
            self.send_json(404, {'error': 'Not found'})
            return
        job_id = pieces[1]
        with REMOTE_JOB_LOCK:
            record = REMOTE_JOBS.get(job_id)
            item = dict(record) if record else None
            if item and item['status'] == 'running':
                item['stalled'] = time.monotonic() - item.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
            if item and item['status'] == 'queued':
                waiting = [key for key, value in REMOTE_JOBS.items() if value.get('status') == 'queued']
                if job_id in waiting:
                    item['queue_position'] = waiting.index(job_id) + 1
            if item and len(pieces) == 3 and pieces[2] in ('file', 'last-frame') and item['status'] == 'done':
                record['_downloads'] = record.get('_downloads', 0) + 1
                downloading = True
            else:
                downloading = False
        if item is None:
            self.send_json(404, {'error': 'Unknown job'})
            return
        if len(pieces) == 2:
            for field in tuple(item):
                if field.startswith('_'):
                    item.pop(field)
            self.send_json(200, item)
            return
        if not downloading:
            self.send_json(404, {'error': 'Result not available for this job'})
            return
        try:
            file_path = item.get('_path') if pieces[2] == 'file' else item.get('_last_frame')
            if not file_path or not Path(file_path).is_file():
                self.send_json(404, {'error': 'Output is no longer available'})
                return
            result = Path(file_path)
            self.send_response(200)
            self.send_header('Content-Type', 'application/octet-stream')
            self.send_header('Content-Disposition', f'attachment; filename="{result.name}"')
            self.send_header('Content-Length', str(result.stat().st_size))
            self.end_headers()
            with result.open('rb') as stream:
                shutil.copyfileobj(stream, self.wfile, length=1024 * 1024)
        except (BrokenPipeError, ConnectionResetError):
            pass
        finally:
            with REMOTE_JOB_LOCK:
                record['_downloads'] -= 1

    def do_POST(self):
        if urlsplit(self.path).path != '/generate':
            self.send_json(404, {'error': 'Not found'})
            return
        supplied_id = self.headers.get('X-Request-ID', '').strip()
        try:
            job_id = uuid.UUID(supplied_id).hex if supplied_id else uuid.uuid4().hex
        except ValueError:
            self.send_json(400, {'error': 'X-Request-ID must be a UUID'})
            return
        with REMOTE_JOB_LOCK:
            existing = REMOTE_JOBS.get(job_id)
        if existing:
            self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                 'download_url': f'/jobs/{job_id}/file', 'reused': True})
            return
        if remote_stalled():
            self.send_json(503, {'error': 'GPU worker appears stalled; inspect the Notebook before accepting more jobs'}, 60)
            return
        gpu_ready, gpu_message, _ = remote_gpu_status()
        if not gpu_ready:
            self.send_json(409, {'error': gpu_message})
            return
        if shutil.disk_usage(REMOTE_ROOT).free < REMOTE_MIN_FREE_BYTES:
            self.send_json(507, {'error': 'Host disk space is low; clean old outputs before retrying'})
            return
        if not REMOTE_UPLOAD_SLOTS.acquire(timeout=1):
            self.send_json(503, {'error': 'Upload slots busy; retry later'}, 5)
            return
        staging = None
        try:
            self.connection.settimeout(120)
            length = int(self.headers.get('Content-Length', '0'))
            if not 0 < length <= 120 * 1024 * 1024:
                raise ValueError('Request must be between 1 byte and 120 MiB')
            body = self.rfile.read(length)
            if len(body) != length:
                raise ValueError('Incomplete request body')
            content_type = self.headers.get('Content-Type', '')
            fields, files, videos, audios = {}, [], [], []
            if content_type.startswith('application/json'):
                fields = json.loads(body)
            elif content_type.startswith('application/x-www-form-urlencoded'):
                fields = {key: values[-1] for key, values in parse_qs(body.decode('utf-8')).items()}
            elif content_type.startswith('multipart/form-data'):
                message = BytesParser(policy=policy.default).parsebytes(
                    b'Content-Type: ' + content_type.encode() + b'\r\nMIME-Version: 1.0\r\n\r\n' + body)
                for part in message.iter_parts():
                    name = part.get_param('name', header='content-disposition')
                    data = part.get_payload(decode=True)
                    if name in ('images', 'videos', 'audios'):
                        collection = {'images': files, 'videos': videos, 'audios': audios}[name]
                        collection.append((part.get_filename() or 'upload', data))
                    elif name:
                        fields[name] = data.decode('utf-8')
            else:
                raise ValueError('Use application/json, form data, or multipart/form-data')
            if not isinstance(fields, dict):
                raise ValueError('Parameters must be a JSON object')
            if 'abc' in fields and len(str(fields['abc']).encode('utf-8')) > 1024 * 1024:
                raise ValueError('ABC score must be 1 MiB or smaller')
            task = str(fields.get('task', REMOTE_ALLOWED_TASKS[0])).lower()
            if task not in REMOTE_ALLOWED_TASKS:
                raise ValueError('Unsupported task: ' + task)
            if len(files) > REMOTE_MAX_IMAGES or len(videos) > 3 or len(audios) > 3:
                raise ValueError(f'Only {REMOTE_MAX_IMAGES} input images are supported')
            if REMOTE_KIND in ('music', 'image_text') and (files or videos or audios):
                raise ValueError('This model does not accept reference images')
            if REMOTE_KIND == 'video':
                minimum, maximum = {'t2v': (0, 0), 'i2v': (1, 1), 'l2v': (1, 1), 'flf2v': (2, 2),
                                    'r2v': (0, REMOTE_MAX_IMAGES)}[task] if task == 'r2v' else {
                                    't2v': (0, 0), 'i2v': (1, 1), 'l2v': (1, 1), 'flf2v': (2, 2)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError('Input image count does not match the selected video task')
            if task != 'r2v' and (videos or audios):
                raise ValueError('Reference videos/audio are supported only for R2V')
            if task == 'r2v' and not (files or videos):
                raise ValueError('R2V requires an image or a video; audio alone is not supported')
            if REMOTE_KIND == 'image_multi':
                minimum, maximum = {'text': (0, 0), 'edit': (1, 1), 'multi': (2, REMOTE_MAX_IMAGES)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError(f'Task {task} requires {minimum}–{maximum} images')
            for key in fields:
                if key not in REMOTE_FIELDS:
                    raise ValueError('Unsupported parameter: ' + key)
            remote_validate_fields(task, dict(fields, _image_count=len(files), _video_count=len(videos), _audio_count=len(audios)))
            if 'quant' in fields and fields['quant']:
                active_model = globals().get('H3_ACTIVE_MODEL', ())
                if fields['quant'] not in active_model:
                    self.send_json(409, {'error': '指定的量化尚未載入；請先在 Notebook 下載並預檢該 GGUF。'})
                    return
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                self.send_json(409, {'error': inference_message})
                return
            for filename, data in files:
                if not isinstance(data, bytes) or len(data) > 20 * 1024 * 1024:
                    raise ValueError('Each image must be 20 MiB or smaller')
                if Path(filename).suffix.lower() not in {'.png', '.jpg', '.jpeg', '.webp'}:
                    raise ValueError('Only PNG/JPEG/WebP images are supported')
                from PIL import Image
                with Image.open(io.BytesIO(data)) as photo:
                    photo.verify()
            if str(fields.get('video_audio', 'false')).lower() not in ('true', 'false'):
                raise ValueError('video_audio must be true or false')
            for filename, data in videos:
                if not data or len(data) > 80 * 1024 * 1024 or Path(filename).suffix.lower() not in ('.mp4', '.mov', '.webm'):
                    raise ValueError('Reference video must be MP4/MOV/WebM and at most 80 MiB')
            for filename, data in audios:
                if not data or len(data) > 30 * 1024 * 1024 or Path(filename).suffix.lower() not in ('.mp3', '.wav', '.flac', '.ogg', '.m4a'):
                    raise ValueError('Reference audio must be an audio file at most 30 MiB')
            staging = Path(tempfile.mkdtemp(prefix='pending_', dir=REMOTE_ROOT))
            image_names, video_names, audio_names = [], [], []
            for index, (filename, data) in enumerate(files, 1):
                target = staging / f'{index:03d}{Path(filename).suffix.lower()}'
                target.write_bytes(data)
                image_names.append(target.name)
            for kind, uploaded, names in (('video', videos, video_names), ('audio', audios, audio_names)):
                for index, (filename, data) in enumerate(uploaded, 1):
                    target = staging / f'{kind}_{index:03d}{Path(filename).suffix.lower()}'
                    target.write_bytes(data)
                    names.append(target.name)

            '''
            Commit the validated upload and queue entry in one critical section.
            在同一個鎖定區段確認請求、保存資料並加入工作佇列。
            '''
            with REMOTE_JOB_LOCK:
                existing = REMOTE_JOBS.get(job_id)
                full = REMOTE_QUEUE.full()
                if not existing and not full:
                    folder = REMOTE_ROOT / job_id
                    staging.rename(folder)
                    staging = None
                    image_paths = [str(folder / name) for name in image_names]
                    fields['_ref_videos'] = [str(folder / name) for name in video_names]
                    fields['_ref_audios'] = [str(folder / name) for name in audio_names]
                    REMOTE_JOBS[job_id] = {'id': job_id, 'task': task, 'status': 'queued',
                                           'created_at': time.time()}
                    REMOTE_QUEUE.put_nowait((job_id, fields, image_paths))
                    position = REMOTE_QUEUE.qsize()
            if existing:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'reused': True})
            elif full:
                self.send_json(429, {'error': 'Queue is full; retry later'}, 15)
            else:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'queue_position': position})
        except (ValueError, KeyError, TypeError, json.JSONDecodeError) as error:
            self.send_json(400, {'error': str(error)})
        except (TimeoutError, ConnectionError, OSError) as error:
            self.send_json(503, {'error': f'Upload temporarily failed: {type(error).__name__}'}, 10)
        except Exception as error:
            self.send_json(500, {'error': type(error).__name__ + ': ' + str(error)[:250]})
        finally:
            if staging is not None:
                shutil.rmtree(staging, ignore_errors=True)
            REMOTE_UPLOAD_SLOTS.release()

'''
Convert numeric HTTP form fields into explicitly supported generator settings.
只轉換白名單中的 HTTP 參數，不執行客戶端提供的任何程式碼。
'''
def remote_generate(job_id, fields, image_paths):
    with REMOTE_ENGINE_LOCK:
        with REMOTE_JOB_LOCK:
            REMOTE_JOBS[job_id].update(status='running', _start_clock=time.monotonic())
        namespace = globals()
        keys = ['TASK', 'PROMPT', 'WIDTH', 'HEIGHT', 'SEED', 'LM_SEED', 'SECONDS', 'STEPS', 'CFG',
                'LYRICS', 'TAGS', 'STYLE', 'DURATION', 'GPU_ID', 'COT', 'ABC_PATH',
                'OUTPUT_NAME', 'REFERENCE_PATHS', 'ACTIVE_JOB', 'TASK_CONFIGS',
                'R2V_REF_DIR', 'FPS', 'NEGATIVE_PROMPT', 'SAMPLER', 'REFERENCE_SIZE',
                'OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT', 'RUN_SEED',
                'QUANT', 'ENCODER_VARIANT']
        before = {key: copy.deepcopy(namespace.get(key)) for key in keys if key in namespace}
        missing = set(keys) - set(before)
        try:
            task = fields.get('task', REMOTE_ALLOWED_TASKS[0]).lower()
            numerics = {'seed': int, 'lm_seed': int, 'width': int, 'height': int, 'steps': int,
                        'seconds': float, 'fps': int, 'cfg': float, 'duration': int,
                        'image_strength': float, 'reference_size': int, 'audio_shift': float}
            inputs = {}
            for key, value in fields.items():
                inputs[key] = numerics[key](value) if key in numerics else value
            for seed_key in ('seed', 'lm_seed'):
                if seed_key in inputs:
                    if not -1 <= inputs[seed_key] < 2**32:
                        raise ValueError(f'{seed_key} must be -1 or between 0 and 4294967295')
                    if inputs[seed_key] == -1:
                        inputs[seed_key] = secrets.randbelow(2**32)
            with REMOTE_JOB_LOCK:
                for seed_key in ('seed', 'lm_seed'):
                    if seed_key in inputs:
                        REMOTE_JOBS[job_id][seed_key + '_used'] = inputs[seed_key]
            for dimension in ('width', 'height'):
                if dimension in inputs and not 0 < inputs[dimension] <= 4096:
                    raise ValueError('Width and height must be between 1 and 4096')
            for key in ('seconds', 'duration'):
                if key in inputs and not 0 < inputs[key] <= 3600:
                    raise ValueError('Duration must be between 0 and 3600 seconds')
            if 'reference_size' in inputs and not 64 <= inputs['reference_size'] <= 2048:
                raise ValueError('Reference size must be between 64 and 2048')
            if 'ref_image_size' in inputs and inputs['ref_image_size'] not in ('match', 'max'):
                raise ValueError('Reference image mode must be match or max')
            if 'image_strength' in inputs and not 0 <= inputs['image_strength'] <= 1:
                raise ValueError('Image strength must be between 0 and 1')
            if 'fps' in inputs and not 1 <= inputs['fps'] <= 60:
                raise ValueError('FPS must be between 1 and 60')
            if 'steps' in inputs and not 1 <= inputs['steps'] <= 200:
                raise ValueError('Steps must be between 1 and 200')
            if REMOTE_KIND == 'video':
                if 'TASK_CONFIGS' in namespace:
                    if task not in namespace['TASK_CONFIGS']:
                        raise ValueError('Run this task configuration cell before API generation')
                    updated = copy.deepcopy(namespace['TASK_CONFIGS'][task])
                    for field in ('prompt', 'width', 'height', 'seconds', 'fps', 'steps', 'seed',
                                  'audio_shift', 'ref_image_size', 'quant'):
                        if field in inputs:
                            updated[field] = inputs[field]
                    updated['output_name'] = f'remote_{job_id[:12]}.mp4'
                    updated['quant'] = inputs.get('quant') or namespace.get('QUANT', updated.get('quant'))
                    if task == 'r2v':
                        updated['video_audio'] = str(inputs.get('video_audio', 'false')).lower() == 'true'
                        updated['ref_videos'] = inputs.get('_ref_videos', [])
                        updated['ref_audios'] = inputs.get('_ref_audios', [])
                    if task == 'i2v':
                        updated['first_image'] = image_paths[0]
                    elif task == 'l2v':
                        updated['last_image'] = image_paths[0]
                    elif task == 'flf2v':
                        updated['first_image'], updated['last_image'] = image_paths
                    elif task == 'r2v':
                        updated['ref_images'] = image_paths
                        namespace['R2V_REF_DIR'] = REMOTE_ROOT / job_id
                    namespace['TASK_CONFIGS'][task] = updated
                    namespace['TASK'] = task
                else:
                    namespace['ACTIVE_JOB'] = {'mode': task, 'prompt': inputs.get('prompt', namespace.get('PROMPT', '')),
                                               'run_name': f'remote_{job_id[:12]}',
                                               'first_path': image_paths[0] if image_paths else None,
                                               'last_path': image_paths[1] if len(image_paths) > 1 else None}
                    for key in ('width', 'height', 'seconds', 'seed', 'image_strength'):
                        if key in inputs:
                            namespace[key.upper()] = inputs[key]
            else:
                for key in ('prompt', 'width', 'height', 'seed', 'lm_seed', 'steps', 'cfg',
                            'lyrics', 'tags', 'style', 'duration', 'negative_prompt',
                            'sampler', 'reference_size', 'cot'):
                    if key in inputs:
                        namespace[key.upper()] = inputs[key]
                if REMOTE_KIND == 'image_multi':
                    namespace['TASK'] = task
                    namespace['REFERENCE_PATHS'] = image_paths
                if REMOTE_KIND == 'music':
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.mp3'
                    if 'ABC_PATH' in namespace:
                        namespace['ABC_PATH'] = ''
                    if 'abc' in inputs:
                        score = (REMOTE_ROOT / job_id / 'input.abc')
                        score.write_text(str(inputs['abc']), encoding='utf-8')
                        namespace['ABC_PATH'] = str(score)
                if REMOTE_KIND in ('image_multi', 'image_text'):
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.png'
            for output_key in ('OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT'):
                namespace.pop(output_key, None)
            '''
            Recheck the CUDA backend after queuing to prevent CPU-only generation.
            工作排入佇列後再次確認 CUDA，避免後端異常時改走純 CPU 生成。
            '''
            gpu_ready, gpu_message, _ = remote_gpu_status()
            if not gpu_ready:
                raise RuntimeError(gpu_message)
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                raise RuntimeError(inference_message)
            exec(compile(REMOTE_SCRIPT, '<notebook-generation-cell>', 'exec'), namespace)
            paths = []
            if REMOTE_KIND == 'video':
                paths = [namespace.get('LAST_OUTPUT') or namespace.get('OUTPUT_MP4')]
            elif REMOTE_KIND == 'image_text':
                paths = [namespace.get('OUTPUT_PATH')]
            elif REMOTE_KIND == 'image_multi':
                paths = [namespace.get('output')]
            elif REMOTE_KIND == 'music':
                paths = namespace.get('OUTPUTS', []) if 'OUTPUTS' in namespace else [namespace.get('output')]
            results = [Path(path) for path in paths if path is not None]
            if REMOTE_KIND == 'music' and REMOTE_SERVICE_ID.startswith('06_YuE2_'):
                results += [sidecar for path in list(results)
                            for ext in ('.json', '.abc')
                            if (sidecar := path.with_suffix(ext)).is_file()]
            if not results or any(not path.is_file() for path in results):
                raise FileNotFoundError('Generation finished without a saved result file')
            folder = REMOTE_ROOT / job_id
            if len(results) == 1:
                target = folder / f'result{results[0].suffix.lower()}'
                shutil.copy2(results[0], target)
            else:
                target = folder / 'results.zip'
                with zipfile.ZipFile(target, 'w', compression=zipfile.ZIP_STORED) as archive:
                    for result in results:
                        archive.write(result, arcname=result.name)
            last_frame = None
            if REMOTE_KIND == 'video' and shutil.which('ffprobe') and shutil.which('ffmpeg'):
                try:
                    probe = subprocess.run(['ffprobe', '-v', 'error', '-count_frames',
                                            '-select_streams', 'v:0', '-show_entries',
                                            'stream=nb_read_frames', '-of', 'json', str(target)],
                                           check=True, text=True, capture_output=True, timeout=120)
                    index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1
                    if index >= 0:
                        last_frame = folder / 'last_frame.png'
                        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(target),
                                        '-vf', rf'select=eq(n\,{index})', '-frames:v', '1',
                                        str(last_frame)], check=True)
                except (OSError, ValueError, KeyError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
                    last_frame = None
            for original in results:
                if original.name.startswith(f'remote_{job_id[:12]}') and original != target:
                    try:
                        original.unlink(missing_ok=True)
                    except OSError:
                        pass
            comfy = globals().get('COMFY')
            if comfy is not None and (Path(comfy) / 'output').exists():
                for artifact in (Path(comfy) / 'output').rglob(f'remote_{job_id[:12]}*'):
                    if artifact.is_file():
                        try:
                            artifact.unlink()
                        except OSError:
                            pass
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='done', filename=target.name,
                                           bytes=target.stat().st_size, _path=str(target),
                                           last_frame_available=bool(last_frame),
                                           _last_frame=str(last_frame) if last_frame else None)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            for key in missing:
                namespace.pop(key, None)
            namespace.update(before)

'''
Use one GPU worker, regardless of the number of HTTP request threads.
不論有多少 HTTP 連線，都只有一個 GPU worker 消費佇列。
'''
def remote_worker():
    while True:
        job_id, fields, image_paths = REMOTE_QUEUE.get()
        try:
            remote_generate(job_id, fields, image_paths)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id]['finished_at'] = time.time()
            try:
                remote_remove_inputs(job_id)
            except OSError as error:
                print('API input cleanup warning:', str(error)[:180])
            finally:
                REMOTE_QUEUE.task_done()

if not (globals().get('REMOTE_WORKER') and REMOTE_WORKER.is_alive()):
    REMOTE_WORKER = threading.Thread(target=remote_worker, daemon=True)
    REMOTE_WORKER.start()

if not (globals().get('REMOTE_MAINTENANCE') and REMOTE_MAINTENANCE.is_alive()):
    REMOTE_MAINTENANCE = threading.Thread(target=remote_maintenance, daemon=True)
    REMOTE_MAINTENANCE.start()

old_tunnel = globals().get('REMOTE_NGROK_PROC')
if old_tunnel is not None and old_tunnel.poll() is None:
    old_tunnel.terminate()
    try:
        old_tunnel.wait(timeout=5)
    except subprocess.TimeoutExpired:
        old_tunnel.kill()
        old_tunnel.wait(timeout=5)

NGROK_URL = ''

'''
Install the official ngrok agent only if no usable binary exists.
系統尚無 ngrok 才下載官方執行檔，不重複安裝。
'''
ngrok_bin = shutil.which('ngrok') or str(ROOT / 'ngrok')
if not Path(ngrok_bin).is_file():
    url = 'https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz'
    with urllib.request.urlopen(url, timeout=90) as response:
        blob = response.read()
    with tarfile.open(fileobj=io.BytesIO(blob), mode='r:gz') as archive:
        entry = next((member for member in archive.getmembers() if Path(member.name).name == 'ngrok' and member.isfile()), None)
        if entry is None:
            raise RuntimeError('ngrok 官方下載檔案缺少執行檔')
        Path(ngrok_bin).write_bytes(archive.extractfile(entry).read())
    Path(ngrok_bin).chmod(0o755)

ngrok_token = getpass('輸入 ngrok Authtoken：').strip()
if not ngrok_token:
    raise ValueError('未輸入 ngrok Authtoken')

result = subprocess.run([ngrok_bin, 'config', 'add-authtoken', ngrok_token], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok token 設定失敗（請確認 Token 是否有效）')

del ngrok_token
result = subprocess.run([ngrok_bin, 'config', 'check'], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok 設定格式檢查失敗：' + result.stderr[-500:])

server = globals().get('REMOTE_SERVER')
server_thread = globals().get('REMOTE_SERVER_THREAD')
server_is_current = (
    server is not None
    and server_thread is not None
    and server_thread.is_alive()
    and getattr(server, 'notebook_id', None) == REMOTE_SERVICE_ID
)
if server is not None and not server_is_current:
    if server_thread is not None and server_thread.is_alive():
        server.shutdown()
    server.server_close()

if not server_is_current:
    try:
        REMOTE_SERVER = ThreadingHTTPServer(('127.0.0.1', REMOTE_PORT), RemoteHandler)
    except OSError as error:
        raise RuntimeError(f'本機 API Port {REMOTE_PORT} 已被其他服務占用') from error
    REMOTE_SERVER.notebook_id = REMOTE_SERVICE_ID
    REMOTE_SERVER.daemon_threads = True
    REMOTE_SERVER_THREAD = threading.Thread(target=REMOTE_SERVER.serve_forever, daemon=True)
    REMOTE_SERVER_THREAD.start()
    print('已啟動新的本機 API Server。')
else:
    REMOTE_SERVER.RequestHandlerClass = RemoteHandler
    print('已沿用本機 API Server；現有生成工作與結果保持不變。')

with urllib.request.urlopen(f'http://127.0.0.1:{REMOTE_PORT}/health', timeout=4) as response:
    if json.load(response).get('ok') is not True:
        raise RuntimeError('本機 API Server 尚未就緒')

'''
Use the agent inspection API to obtain the actual public URL.
透過 ngrok 本機檢查 API 取得實際對外 HTTPS 網址。
'''
NGROK_LOG = REMOTE_ROOT / 'ngrok.log'
with NGROK_LOG.open('w') as log:
    REMOTE_NGROK_PROC = subprocess.Popen([ngrok_bin, 'http', str(REMOTE_PORT)], stdout=log,
                                         stderr=subprocess.STDOUT, start_new_session=True)

NGROK_URL = ''
for attempt in range(25):
    if REMOTE_NGROK_PROC.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:4040/api/tunnels', timeout=2) as response:
            tunnels = json.load(response).get('tunnels', [])
        NGROK_URL = next((item['public_url'] for item in tunnels
                          if item.get('public_url', '').startswith('https://') and
                          str(REMOTE_PORT) in item.get('config', {}).get('addr', '')), '')
        if NGROK_URL:
            break
    except (OSError, ValueError, KeyError):
        pass
    time.sleep(1)

if not NGROK_URL:
    raise RuntimeError('ngrok URL 建立失敗：\n' + NGROK_LOG.read_text(errors='replace')[-2500:])

print('ngrok HTTPS URL:', NGROK_URL)
print('提醒：公開的 ngrok API URL 不需要額外驗證，請只分享給授課學生。')
print('GET /health | GET /schema | POST /generate | GET /jobs/{id} | GET /jobs/{id}/file')
print('影片輸出另提供：GET /jobs/{id}/last-frame')
print('✓ 第 14 節完成：選用 FIFO／ngrok API 已啟動。')


15. （選用）遠端 API 與 client 使用說明


In [ ]:
'''
Query the active bridge for supported parameters and task names.
從正在執行的橋接服務讀取實際支援的參數與模式。
'''
import json
import urllib.request
request = urllib.request.Request(f'http://127.0.0.1:{REMOTE_PORT}/schema')
with urllib.request.urlopen(request, timeout=5) as response:
    schema = json.load(response)

print(json.dumps(schema, ensure_ascii=False, indent=2))
print('用法：POST NGROK_URL + "/generate"（JSON 或 multipart/images），取得 job_id；')
print('輪詢 GET /jobs/{job_id}；完成後下載 GET /jobs/{job_id}/file。')
print('本機 requests 範例：06_HyperFlow_H3_Flash_影片生成_client.py（與本 Notebook 同名對應）。')
print('使用前只需在對應的 client.py 填入 API_URL。')
print('本服務使用單一 FIFO worker，遠端請求不會同時啟動多筆 GPU 推論。')
print('✓ 第 15 節完成：API Schema 和 client 說明已顯示。')
